## Cellule 1 — Installation (LangChain + LangGraph)

In [ ]:
import subprocess

def _run(cmd, label=""):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(f"{'✅' if r.returncode == 0 else '❌'} {label or cmd[:70]}")
    if r.returncode != 0:
        print(r.stderr[-400:])
    return r.returncode == 0

# ── Système ──────────────────────────────────────────────────
_run("apt-get install -y -q poppler-utils libgl1 libglib2.0-0", "Poppler + libgl1")

# ─────────────────────────────────────────────────────────────
# MÉTHODE A  –  SDK officiel PaddleOCR (recommandée)
# Source : https://www.paddleocr.ai/latest/version3.x/pipeline_usage/PaddleOCR-VL.html
# ─────────────────────────────────────────────────────────────

# PaddlePaddle GPU 3.2.1 + CUDA 12.6 (selon la doc officielle)
# Autres CUDA → remplacer cu126 par cu118, cu120, etc.
_run(
    "python -m pip install paddlepaddle-gpu==3.2.1 "
    "-i https://www.paddlepaddle.org.cn/packages/stable/cu126/",
    "PaddlePaddle GPU 3.2.1 (cu126)"
)

# PaddleOCR avec le module doc-parser (requis pour PaddleOCRVL)
_run('python -m pip install -U "paddleocr[doc-parser]"', "PaddleOCR[doc-parser]")

# ─────────────────────────────────────────────────────────────
# MÉTHODE B  –  transformers v5 (HuggingFace natif)
# Note : nécessite transformers >= 5.0.0 UNIQUEMENT
# ─────────────────────────────────────────────────────────────
_run('python -m pip install "transformers>=5.0.0"', "Transformers >= 5.0.0")

# ── Utilitaires communs ───────────────────────────────────────
_run("pip install -q Pillow pymupdf opencv-python-headless torch", "Pillow + PyMuPDF + OpenCV + Torch")
_run("pip install -q rank_bm25 huggingface_hub tqdm pydantic", "Utilitaires RAG")
_run("pip install -q -U langchain langchain-text-splitters langchain-community", "langchain")

print("""
🎉 Installation terminée.

  Méthode A (recommandée) → from paddleocr import PaddleOCRVL
  Méthode B (HuggingFace) → from transformers import AutoProcessor, AutoModelForImageTextToText
                             model_path = "PaddlePaddle/PaddleOCR-VL-1.5"
""")

## Cellule 2 — Configuration globale & PDF

In [2]:
import os, json, re, time, glob, unicodedata
from pathlib import Path
from dataclasses import dataclass, field, asdict
from typing import Optional, List, Dict, Any, Literal, TypedDict, Annotated
from concurrent.futures import ThreadPoolExecutor, Future
import operator

# ── Détection environnement ──────────────────────────────────────────
def detect_env():
    if os.path.exists('/kaggle'):
        return 'kaggle'
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        return 'colab'
    except (ImportError, NotImplementedError):
        return 'local'

ENV = detect_env()
COLAB  = ENV == 'colab'
KAGGLE = ENV == 'kaggle'
print(f"✅ Environnement détecté : {ENV.upper()}")

# ── Chemins selon environnement ──────────────────────────────────────
if KAGGLE:
    BASE_DIR = "/kaggle/working"
    # ➡️  Sur Kaggle : uploadez votre PDF via "Add Data" → "Upload"
    #     Il sera accessible dans /kaggle/input/<dataset-name>/
    PDF_PATH = os.getenv(
        "LOC_PDF_PATH",
        "/kaggle/input/datasets/azizndiaye/lois-ma/les_lois_des_obligations_et_contrats.pdf"
    )
elif COLAB:
    BASE_DIR = "/content"
    PDF_PATH = os.getenv(
        "LOC_PDF_PATH",
        "/content/drive/MyDrive/3D_SMART/lois/les_lois_des_obligations_et_contrats.pdf"
    )
else:  # local
    BASE_DIR = str(Path.home() / "loc_project")
    PDF_PATH = os.getenv(
        "LOC_PDF_PATH",
        str(Path.home() / "lois/les_lois_des_obligations_et_contrats.pdf")
    )

IMG_DIR        = f"{BASE_DIR}/loc_pages"
OCR_CACHE      = f"{BASE_DIR}/loc_ocr_pages.json"
OCR_JSONL      = f"{BASE_DIR}/loc_ocr_pages.jsonl"
STRUCT_OUT     = f"{BASE_DIR}/loc_structured.json"
RAG_OUT        = f"{BASE_DIR}/loc_rag_chunks.json"
EMBED_CACHE    = f"{BASE_DIR}/loc_embeddings.npy"
CHROMA_DIR     = f"{BASE_DIR}/loc_chroma_db"
AGENT_LOG      = f"{BASE_DIR}/agent_trace.jsonl"

for d in [IMG_DIR, CHROMA_DIR]:
    os.makedirs(d, exist_ok=True)

# ── Constantes ──────────────────────────────────────────────────────────
MIN_PIXELS        = 256 * 28 * 28
MAX_PIXELS        = 896 * 28 * 28
BATCH_SIZE        = 5
CACHE_CLEAR_EVERY = 10
NUM_IO_WORKERS    = 8
MAX_NEW_TOKENS    = 600
LOC_MAX_NUM       = 1260
MIN_ARABIC_CHARS  = 15
RRF_K             = 60
RRF_MIN_THRESHOLD = 0.012
TOP_K_RETRIEVAL   = 8

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Device : {DEVICE}")
print(f"   ENV     : {ENV}")
print(f"   PDF     : {PDF_PATH}")
print(f"   Chroma  : {CHROMA_DIR}")
print(f"   Log     : {AGENT_LOG}")

if not os.path.exists(PDF_PATH):
    print(f"\n⚠️  PDF introuvable : {PDF_PATH}")
    if KAGGLE:
        print("   → Sur Kaggle, allez dans le panneau droit :")
        print("     'Add Data' → 'Upload' → uploadez votre PDF")
        print("     Le chemin sera : /kaggle/input/<nom-dataset>/<fichier>.pdf")
        print("     Puis mettez à jour PDF_PATH ou définissez LOC_PDF_PATH")
    else:
        print("   → Définissez la variable d'environnement LOC_PDF_PATH")

✅ Environnement détecté : KAGGLE
✅ Device : cuda
   ENV     : kaggle
   PDF     : /kaggle/input/datasets/azizndiaye/lois-ma/les_lois_des_obligations_et_contrats.pdf
   Chroma  : /kaggle/working/loc_chroma_db
   Log     : /kaggle/working/agent_trace.jsonl


## celulle 2B

In [3]:


import os, json, shutil, time
from pathlib import Path

# ─────────────────────────────────────────────────────────────
# CONFIGURATION DU STOCKAGE PERMANENT
# ─────────────────────────────────────────────────────────────

if KAGGLE:
    # Kaggle : /kaggle/working persiste pendant la session
    # Pour une vraie persistance inter-sessions → exporter via Kaggle Dataset API
    # ou monter un dataset en Input et le copier ici.
    PERSISTENT_DIR = "/kaggle/working/loc_cache"

elif COLAB:
    # Colab : Google Drive est le seul stockage permanent
    # drive.mount() a déjà été appelé dans detect_env()
    PERSISTENT_DIR = "/content/drive/MyDrive/3D_SMART/loc_cache"

else:  # local
    PERSISTENT_DIR = str(Path.home() / "loc_cache")

os.makedirs(PERSISTENT_DIR, exist_ok=True)

# Fichiers à synchroniser : (chemin_session, nom_dans_cache_permanent)
CACHE_FILES = [
    (OCR_CACHE,   "loc_ocr_pages.json"),
    (OCR_JSONL,   "loc_ocr_pages.jsonl"),
    (STRUCT_OUT,  "loc_structured.json"),
    (RAG_OUT,     "loc_rag_chunks.json"),
    (EMBED_CACHE, "loc_embeddings.npy"),
]

# ─────────────────────────────────────────────────────────────
# FONCTIONS CORE
# ─────────────────────────────────────────────────────────────

def _size_str(path: str) -> str:
    """Retourne la taille lisible d'un fichier."""
    try:
        s = os.path.getsize(path)
        if s > 1_000_000: return f"{s/1_000_000:.1f} MB"
        if s > 1_000:     return f"{s/1_000:.1f} KB"
        return f"{s} B"
    except:
        return "?"

def _count_ocr_pages(json_path: str) -> int:
    """Compte le nombre de pages OCR dans le cache JSON."""
    try:
        with open(json_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        return len(data)
    except:
        return 0

def restore_cache_from_persistent() -> dict:
    """
    Restaure les fichiers de cache depuis le stockage permanent vers
    le répertoire de session. Appelé AU DÉMARRAGE de chaque session.

    Returns: dict {nom_fichier: statut}
    """
    print("\n" + "═"*60)
    print("  💾 RESTAURATION DU CACHE OCR")
    print(f"  Source : {PERSISTENT_DIR}")
    print("═"*60)

    stats = {}
    restored = 0

    for session_path, cache_name in CACHE_FILES:
        persistent_path = os.path.join(PERSISTENT_DIR, cache_name)

        if not os.path.exists(persistent_path):
            print(f"  ⬜ {cache_name:<35} → pas encore dans le cache")
            stats[cache_name] = "absent"
            continue

        # Vérifier si la copie locale est déjà à jour (même taille)
        if os.path.exists(session_path):
            src_size = os.path.getsize(persistent_path)
            dst_size = os.path.getsize(session_path)
            if src_size == dst_size:
                print(f"  ✅ {cache_name:<35} → déjà à jour ({_size_str(session_path)})")
                stats[cache_name] = "ok"
                continue

        # Copier depuis le stockage permanent
        try:
            os.makedirs(os.path.dirname(session_path), exist_ok=True)
            shutil.copy2(persistent_path, session_path)
            size = _size_str(session_path)

            # Info supplémentaire pour le cache OCR principal
            extra = ""
            if cache_name == "loc_ocr_pages.json":
                n = _count_ocr_pages(session_path)
                extra = f" → {n} pages OCR"

            print(f"  ✅ {cache_name:<35} → restauré  ({size}{extra})")
            stats[cache_name] = "restored"
            restored += 1
        except Exception as e:
            print(f"  ❌ {cache_name:<35} → erreur : {e}")
            stats[cache_name] = f"error: {e}"

    print("─"*60)
    if restored == 0:
        print("  ℹ️  Aucun fichier restauré (cache vide ou déjà à jour)")
    else:
        print(f"  🎉 {restored} fichier(s) restauré(s) depuis le stockage permanent")

    # Vérifier l'état du cache OCR principal
    if os.path.exists(OCR_CACHE):
        n = _count_ocr_pages(OCR_CACHE)
        print(f"  📄 Cache OCR actif : {n} pages disponibles")
        if n > 0:
            print(f"  ✨ L'OCR ne sera PAS relancé pour ces {n} pages !")

    print("═"*60 + "\n")
    return stats


def save_cache_to_persistent(files: list = None, verbose: bool = True) -> dict:
    """
    Sauvegarde les fichiers de cache vers le stockage permanent.
    À appeler après chaque OCR ou étape de traitement.

    Args:
        files: liste de noms de fichiers à sauvegarder (None = tous)
        verbose: afficher les détails

    Returns: dict {nom_fichier: statut}
    """
    if verbose:
        print("\n" + "─"*60)
        print(f"  💾 SAUVEGARDE CACHE → {PERSISTENT_DIR}")
        print("─"*60)

    stats = {}
    saved = 0

    for session_path, cache_name in CACHE_FILES:
        # Filtre optionnel
        if files and cache_name not in files:
            continue

        if not os.path.exists(session_path):
            if verbose:
                print(f"  ⬜ {cache_name:<35} → fichier absent (pas encore généré)")
            stats[cache_name] = "absent"
            continue

        persistent_path = os.path.join(PERSISTENT_DIR, cache_name)

        try:
            shutil.copy2(session_path, persistent_path)
            size = _size_str(persistent_path)

            extra = ""
            if cache_name == "loc_ocr_pages.json":
                n = _count_ocr_pages(persistent_path)
                extra = f" → {n} pages"

            if verbose:
                print(f"  ✅ {cache_name:<35} → sauvegardé ({size}{extra})")
            stats[cache_name] = "saved"
            saved += 1
        except Exception as e:
            if verbose:
                print(f"  ❌ {cache_name:<35} → erreur : {e}")
            stats[cache_name] = f"error: {e}"

    if verbose:
        print("─"*60)
        if saved > 0:
            print(f"  🎉 {saved} fichier(s) sauvegardé(s) dans : {PERSISTENT_DIR}")
            if KAGGLE:
                print("  📥 Sur Kaggle → téléchargez via le panneau 'Output'")
                print("     puis re-uploadez comme dataset pour la prochaine session")
        print()

    return stats


def cache_status() -> None:
    """Affiche l'état détaillé du cache OCR (session + permanent)."""
    print("\n" + "═"*60)
    print("  📊 ÉTAT DU CACHE OCR")
    print("═"*60)
    print(f"  {'Fichier':<35} {'Session':>12} {'Permanent':>12}")
    print("  " + "─"*58)

    for session_path, cache_name in CACHE_FILES:
        s_info = f"{_size_str(session_path)}" if os.path.exists(session_path) else "—"
        p_path  = os.path.join(PERSISTENT_DIR, cache_name)
        p_info  = f"{_size_str(p_path)}"      if os.path.exists(p_path)      else "—"
        icon = "✅" if os.path.exists(p_path) else "⬜"
        print(f"  {icon} {cache_name:<33} {s_info:>12} {p_info:>12}")

    # Détail du cache OCR principal
    print("  " + "─"*58)
    if os.path.exists(OCR_CACHE):
        n = _count_ocr_pages(OCR_CACHE)
        print(f"  📄 Pages OCR en cache  : {n}")
        print(f"  📁 Stockage permanent  : {PERSISTENT_DIR}")
    else:
        print(f"  ⚠️  Cache OCR principal absent — l\'OCR sera relancé")
    print("═"*60 + "\n")


# ─────────────────────────────────────────────────────────────
# PATCH TRANSPARENT : auto-save après chaque page OCR
# ─────────────────────────────────────────────────────────────

# On wrape run_ocr_pipeline si elle est déjà définie
# (si cette cellule tourne AVANT la cellule OCR, le patch sera appliqué après)

_original_run_ocr_pipeline = None

def _patch_ocr_pipeline():
    """Patch run_ocr_pipeline pour auto-sauvegarder après l\'OCR."""
    global _original_run_ocr_pipeline
    if "run_ocr_pipeline" not in globals():
        return False
    if _original_run_ocr_pipeline is not None:
        return True  # déjà patché

    _original_run_ocr_pipeline = globals()["run_ocr_pipeline"]

    def _patched_run_ocr_pipeline(*args, **kwargs):
        result = _original_run_ocr_pipeline(*args, **kwargs)
        print("  ⟳ Auto-sauvegarde du cache OCR...")
        save_cache_to_persistent(
            files=["loc_ocr_pages.json", "loc_ocr_pages.jsonl"],
            verbose=False
        )
        print(f"  ✅ Cache OCR sauvegardé ({len(result)} pages) → {PERSISTENT_DIR}")
        return result

    globals()["run_ocr_pipeline"] = _patched_run_ocr_pipeline
    return True


# ─────────────────────────────────────────────────────────────
# EXÉCUTION AUTOMATIQUE AU DÉMARRAGE
# ─────────────────────────────────────────────────────────────

# 1) Restauration immédiate depuis le stockage permanent
_restore_stats = restore_cache_from_persistent()

# 2) Patch du pipeline OCR (si déjà défini)
if _patch_ocr_pipeline():
    print("  ✅ run_ocr_pipeline patché pour auto-sauvegarde")

print(f"✅ Module cache persistant prêt")
print(f"   Stockage : {PERSISTENT_DIR}")
print(f"\n   Commandes utiles :")
print(f"   cache_status()                   → état du cache")
print(f"   save_cache_to_persistent()        → sauvegarde manuelle tout")
print(f"   restore_cache_from_persistent()   → restaurer depuis stockage")



════════════════════════════════════════════════════════════
  💾 RESTAURATION DU CACHE OCR
  Source : /kaggle/working/loc_cache
════════════════════════════════════════════════════════════
  ✅ loc_ocr_pages.json                  → déjà à jour (3.4 MB)
  ✅ loc_ocr_pages.jsonl                 → déjà à jour (3.4 MB)
  ✅ loc_structured.json                 → déjà à jour (785.9 KB)
  ✅ loc_rag_chunks.json                 → déjà à jour (1.2 MB)
  ✅ loc_embeddings.npy                  → déjà à jour (6.0 MB)
────────────────────────────────────────────────────────────
  ℹ️  Aucun fichier restauré (cache vide ou déjà à jour)
  📄 Cache OCR actif : 268 pages disponibles
  ✨ L'OCR ne sera PAS relancé pour ces 268 pages !
════════════════════════════════════════════════════════════

✅ Module cache persistant prêt
   Stockage : /kaggle/working/loc_cache

   Commandes utiles :
   cache_status()                   → état du cache
   save_cache_to_persistent()        → sauvegarde manuelle tout
   resto

## Cellule 3 — OCR Pipeline (PaddleOCR)

In [ ]:

"""
Pipeline OCR – PaddleOCR-VL-1.5
=================================
Source : https://huggingface.co/PaddlePaddle/PaddleOCR-VL-1.5

Deux méthodes disponibles :
  A) SDK officiel   → PaddleOCRVL()         (plus rapide, page entière)
  B) Transformers   → AutoModelForImageTextToText  (élément par élément)

Choisir METHOD = "sdk" ou "transformers" selon l'environnement.
"""

import os, re, json, time, glob, subprocess
from PIL import Image

# ─────────────────────────────────────────────────────────────
# CHOIX DE LA MÉTHODE
# "sdk"          → paddleocr[doc-parser]  (recommandé, page complète)
# "transformers" → transformers >= 5.0.0  (HuggingFace natif)
# ─────────────────────────────────────────────────────────────
METHOD     = os.getenv("OCR_METHOD", "sdk")   # sdk | transformers
MODEL_PATH = "PaddlePaddle/PaddleOCR-VL-1.5"
TASK       = os.getenv("OCR_TASK",   "ocr")   # ocr | table | chart | formula | spotting | seal

PROMPTS = {
    "ocr"      : "OCR:",
    "table"    : "Table Recognition:",
    "formula"  : "Formula Recognition:",
    "chart"    : "Chart Recognition:",
    "spotting" : "Spotting:",
    "seal"     : "Seal Recognition:",
}


# ═════════════════════════════════════════════════════════════
# MÉTHODE A – SDK officiel PaddleOCR
# ═════════════════════════════════════════════════════════════
def load_sdk():
    """Charge le pipeline SDK PaddleOCRVL (une seule fois)."""
    from paddleocr import PaddleOCRVL
    print("Chargement PaddleOCRVL SDK …")
    pipeline = PaddleOCRVL(pipeline_version="v1.5")
    print("✅ Pipeline SDK prêt")
    return pipeline

def ocr_sdk(pipeline, img_path: str) -> str:
    """
    OCR via SDK officiel.
    Retourne le texte extrait de toute la page (parsing complet).
    """
    output = pipeline.predict(img_path)
    lines = []
    for res in output:
        # Affichage console (optionnel)
        # res.print()

        # Extraction du texte selon la structure de res
        if hasattr(res, "text_lines") and res.text_lines:
            for tl in res.text_lines:
                txt = tl.get("text", "").strip()
                if txt:
                    lines.append(txt)
        elif hasattr(res, "json") and res.json:
            # Fallback : dump JSON brut
            lines.append(json.dumps(res.json, ensure_ascii=False))
        else:
            lines.append(str(res))
    return "\n".join(lines)

def ocr_sdk_with_export(pipeline, img_path: str, out_dir: str) -> str:
    """
    Comme ocr_sdk mais sauvegarde aussi JSON + Markdown dans out_dir.
    Utile pour débogage ou réutilisation.
    """
    os.makedirs(out_dir, exist_ok=True)
    output = pipeline.predict(img_path)
    lines = []
    for res in output:
        res.save_to_json(save_path=out_dir)
        res.save_to_markdown(save_path=out_dir)
        if hasattr(res, "text_lines") and res.text_lines:
            for tl in res.text_lines:
                txt = tl.get("text", "").strip()
                if txt:
                    lines.append(txt)
    return "\n".join(lines)


# ═════════════════════════════════════════════════════════════
# MÉTHODE B – Transformers v5 (HuggingFace natif)
# Note : reconnaissance élément par élément, pas page entière
# ═════════════════════════════════════════════════════════════
def load_transformers():
    """Charge le modèle HuggingFace (transformers >= 5.0.0 requis)."""
    import torch
    from transformers import AutoProcessor, AutoModelForImageTextToText

    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    DTYPE  = torch.bfloat16

    print(f"Chargement {MODEL_PATH} via transformers (device={DEVICE}) …")
    model = (
        AutoModelForImageTextToText
        .from_pretrained(MODEL_PATH, torch_dtype=DTYPE)
        .to(DEVICE)
        .eval()
    )
    processor = AutoProcessor.from_pretrained(MODEL_PATH)
    print("✅ Modèle transformers prêt")
    return model, processor

def ocr_transformers(model, processor, img_path: str,
                     task: str = "ocr", max_new_tokens: int = 512) -> str:
    """
    OCR via transformers v5.
    Supporte : ocr | table | chart | formula | spotting | seal
    """
    import torch

    image = Image.open(img_path).convert("RGB")

    # Upscale pour le mode spotting si image trop petite
    if task == "spotting":
        w, h = image.size
        if w < 1500 and h < 1500:
            try:    resample = Image.Resampling.LANCZOS
            except: resample = Image.LANCZOS
            image = image.resize((w * 2, h * 2), resample)

    max_pixels = (2048 * 28 * 28) if task == "spotting" else (1280 * 28 * 28)

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text",  "text": PROMPTS[task]},
            ],
        }
    ]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        images_kwargs={
            "size": {
                "shortest_edge": processor.image_processor.min_pixels,
                "longest_edge" : max_pixels,
            }
        },
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens)

    # Décoder uniquement les nouveaux tokens (hors prompt)
    result = processor.decode(outputs[0][inputs["input_ids"].shape[-1]:-1])
    return result.strip()


# ═════════════════════════════════════════════════════════════
# UTILITAIRES PDF  (communs aux deux méthodes)
# ═════════════════════════════════════════════════════════════
def ensure_poppler():
    if subprocess.call(["which", "pdftoppm"],
                       stdout=subprocess.DEVNULL,
                       stderr=subprocess.DEVNULL) != 0:
        subprocess.check_call(["apt-get", "install", "-y", "-q", "poppler-utils"])

def rasterize_pdf(pdf_path: str, out_dir: str, dpi: int = 300) -> list[str]:
    ensure_poppler()
    os.makedirs(out_dir, exist_ok=True)
    existing = sorted(glob.glob(f"{out_dir}/page-*.png"))
    if existing:
        print(f"  {len(existing)} pages (cache disque)")
        return existing
    subprocess.run(
        ["pdftoppm", "-png", "-r", str(dpi), pdf_path, f"{out_dir}/page"],
        check=True,
    )
    pages = sorted(glob.glob(f"{out_dir}/page-*.png"))
    print(f"  {len(pages)} pages générées")
    return pages

def _page_num(path: str) -> int:
    m = re.search(r"page-(\d+)\.png", path)
    return int(m.group(1)) if m else -1

def _load_jsonl(path: str) -> dict:
    data = {}
    if not os.path.exists(path):
        return data
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                j = json.loads(line)
                data[str(j["page"])] = j["text"]
            except Exception:
                pass
    return data


# ═════════════════════════════════════════════════════════════
# PIPELINE COMPLET PDF → OCR → JSON
# ═════════════════════════════════════════════════════════════
def run_ocr_pipeline(
    pdf_path  : str,
    img_dir   : str,
    cache_path: str,
    dpi       : int  = 300,
    force     : bool = False,
    method    : str  = "sdk",   # "sdk" | "transformers"
    task      : str  = "ocr",
) -> dict[int, str]:

    jsonl_path = cache_path.replace(".json", ".jsonl")

    # ── Cache ────────────────────────────────────────────────
    if force:
        for p in [cache_path, jsonl_path]:
            if os.path.exists(p): os.remove(p)
        results = {}
    elif os.path.exists(cache_path):
        with open(cache_path, "r", encoding="utf-8") as f:
            results = json.load(f)
        print(f"  Cache JSON  : {len(results)} pages")
    else:
        results = _load_jsonl(jsonl_path)
        print(f"  Cache JSONL : {len(results)} pages")

    # ── Chargement du modèle ─────────────────────────────────
    if method == "sdk":
        pipeline_obj = load_sdk()
        def _ocr(img_path): return ocr_sdk(pipeline_obj, img_path)
    else:
        model, processor = load_transformers()
        def _ocr(img_path): return ocr_transformers(model, processor, img_path, task=task)

    # ── Rastérisation PDF ────────────────────────────────────
    pages = rasterize_pdf(pdf_path, img_dir, dpi)
    todo  = [p for p in pages if str(_page_num(p)) not in results]
    print(f"  Pages à traiter : {len(todo)}")

    # ── OCR page par page ────────────────────────────────────
    start = time.time()
    with open(jsonl_path, "a", encoding="utf-8") as f_jsonl:
        for i, img_path in enumerate(todo, 1):
            pn = _page_num(img_path)
            try:
                txt = _ocr(img_path)
            except Exception as e:
                txt = f"[ERROR {e}]"

            results[str(pn)] = txt
            f_jsonl.write(json.dumps({"page": pn, "text": txt}, ensure_ascii=False) + "\n")
            f_jsonl.flush()

            elapsed = time.time() - start
            eta     = (elapsed / i) * (len(todo) - i)
            print(f"  p{pn:>4} | {len(txt):>5} chars | ETA {eta/60:.1f} min")

    with open(cache_path, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

    print(f"\n✅ OCR terminé ({method}) → {cache_path}")

    # ── Auto-sauvegarde dans le stockage permanent ──────────
    if "save_cache_to_persistent" in globals():
        print("  ⟳ Sauvegarde automatique du cache OCR...")
        save_cache_to_persistent(
            files=["loc_ocr_pages.json", "loc_ocr_pages.jsonl"],
            verbose=False
        )
        print(f"  ✅ Cache permanent mis à jour ({len(results)} pages)")

    return {int(k): v for k, v in results.items()}


# ═════════════════════════════════════════════════════════════
# POINT D'ENTRÉE
# ═════════════════════════════════════════════════════════════
if __name__ == "__main__":
    PDF_PATH  = os.getenv("LOC_PDF_PATH",
        "/kaggle/input/datasets/azizndiaye/lois-ma/les_lois_des_obligations_et_contrats.pdf")
    IMG_DIR   = "/kaggle/working/loc_pages"
    OCR_CACHE = "/kaggle/working/loc_ocr_pages.json"

    # Choisir la méthode via variable d'env :
    #   OCR_METHOD=sdk          python ocr_pipeline.py
    #   OCR_METHOD=transformers python ocr_pipeline.py
    method = os.getenv("OCR_METHOD", "sdk")
    task   = os.getenv("OCR_TASK",   "ocr")

    if not os.path.exists(PDF_PATH):
        print(f"❌ PDF introuvable : {PDF_PATH}")
    else:
        ocr_pages = run_ocr_pipeline(
            pdf_path   = PDF_PATH,
            img_dir    = IMG_DIR,
            cache_path = OCR_CACHE,
            dpi        = 300,
            force      = False,
            method     = method,
            task       = task,
        )
        print(f"\n✅ {len(ocr_pages)} pages OCRisées")
        # Sauvegarde permanente après OCR complet
        if "save_cache_to_persistent" in globals():
            save_cache_to_persistent(verbose=True)

        for k in sorted(ocr_pages)[:5]:
            print(f"  p{k}: {ocr_pages[k][:120].replace(chr(10),' ')}")

## Cellule 4 — Parsing structurel LOC + Chunks RAG

In [4]:
# ══════════════════════════════════════════════════════════════
# CELLULE 4 — Parsing & Structuration LOC
# ══════════════════════════════════════════════════════════════

import json
import re
import unicodedata
from pathlib import Path
from typing import Optional

# ──────────────────────────────────────────────────────────────
# FIX-4A : Vérification robuste avec globals()
# ──────────────────────────────────────────────────────────────
for varname in ["OCR_CACHE", "STRUCT_OUT", "BASE_DIR", "LOC_MAX_NUM"]:
    assert varname in globals(), \
        f"❌ {varname} non défini — relancez la Cellule 2"

INPUT_JSON  = OCR_CACHE
OUTPUT_JSON = STRUCT_OUT

# ──────────────────────────────────────────────────────────────
# LABELS OCR
# ──────────────────────────────────────────────────────────────
KEPT_LABELS = {"text", "paragraph_title", "content", "doc_title"}
SKIP_LABELS = {"header", "footer", "number", "image", "figure_title",
               "footnote", "table"}

# ──────────────────────────────────────────────────────────────
# PATTERNS ARABES
# ──────────────────────────────────────────────────────────────
_ALEF = r"[\u0627\u0671\u0622\u0623\u0625]"

# FIX-4B : Pattern plus robuste — tolère les espaces multiples et
#           les variantes d'orthographe OCR de "الفصل"
# FIX-v9A : Normalisation chiffres arabes-indiens AVANT toute extraction
#   ٢ (U+0662) était confondu avec ١ (U+0661) → 628→618, 629→619
_ARABIC_DIGITS = str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789")

def _norm_digits(text: str) -> str:
    """Convertit les chiffres arabes-indiens en chiffres latins."""
    return text.translate(_ARABIC_DIGITS)

_FASIL_PAT = re.compile(
    r"^\s*(?:" + _ALEF + r")\u0644\u0641\u0635\u0644"
    r"[\s\u00a0]+"           # espaces normaux ET non-breakable
    r"(\d{1,4})"
    r"(?:[\s\u00a0]*[-\.][\s\u00a0]*(\d{1,3}))?"
    r"(?:[\s\u00a0]+\u0645\u0643\u0631\u0631(?:[\s\u00a0]*\d+)?)?"
    r"[\s\u00a0]*$",
    re.UNICODE | re.MULTILINE,
)

_FASIL_INLINE = re.compile(
    r"^\s*(?:" + _ALEF + r")\u0644\u0641\u0635\u0644"
    r"[\s\u00a0]+(\d{1,4}(?:[-\.]\d{1,3})?)"
    r"(?:[\s\u00a0]+\u0645\u0643\u0631\u0631)?"
    r"[\s\u00a0]*\n([\s\S]+)",
    re.UNICODE,
)

# FIX-4C : Nettoyage des notes LaTeX/exposants OCR : $ ^{14} $ etc.
_LATEX_NOTE  = re.compile(r"\$\s*\^?\{?\d+\}?\s*\$", re.UNICODE)
_SUPERSCRIPT = re.compile(r"[\u00b2\u00b3\u00b9\u2070-\u2079]+")
# Notes de bas de page numérotées en début de ligne (ex: "14 - texte")
_FOOTNOTE_NUM = re.compile(r"^\s*\d{1,3}\s*[-–]\s+", re.UNICODE)

_INSTITUTIONAL = re.compile(
    r"(?:المملكة\s+المغربية"
    r"|وزارة\s+(?:العدل|العدار|العدد|الموارد|المغاربة)"
    r"|مديرية\s+التشريع|محديرية\s+التشريع"
    r"|قانون\s+الالتزامات\s+والعقود"
    r"|صيغة\s+محينة)",
    re.UNICODE,
)
_PAGE_NUM_LINE  = re.compile(r"^\s*-?\s*\d{1,3}\s*-?\s*$")
_NOTE_LINE      = re.compile(r"^\s*\d{1,2}\s*-\s+[\u0600-\u06ff]", re.UNICODE)
_STANDALONE_NUM = re.compile(r"^\s*\d{1,3}\s*$")
_LATIN_HEAVY    = re.compile(r"^[A-Za-z\s\.\,\;\:\'\"\-\(\)]{15,}$")

# ──────────────────────────────────────────────────────────────
# NORMALISATION ARABE
# ──────────────────────────────────────────────────────────────
_BIDI_CTRL = re.compile(
    r"[\u200e\u200f\u202a\u202b\u202c\u202d\u202e"
    r"\u2066\u2067\u2068\u2069\ufeff]")
_HARAKAT = re.compile(
    r"[\u0610-\u061a\u064b-\u065f\u0670"
    r"\u06d6-\u06dc\u06df-\u06e4\u06e7\u06e8\u06ea-\u06ed]")
_TATWEEL  = re.compile(r"\u0640")
_AR_DIGITS = str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789")

# ✅ AJOUT : normalisation des variantes d'alef → ا
# إأآ sont des alefs avec hamza/madda, distincts de ا simple
# mais souvent confondus/interchangeables selon la source OCR
_ALEF_NORM = re.compile(r"[\u0625\u0623\u0622]")   # إ أ آ → ا


def normalize(text: str) -> str:
    if not text:
        return ""
    text = unicodedata.normalize("NFC", text)
    text = _BIDI_CTRL.sub("", text)
    text = _HARAKAT.sub("", text)
    text = _TATWEEL.sub("", text)
    text = _ALEF_NORM.sub("\u0627", text)   # ✅ AJOUT : إ أ آ → ا
    text = text.translate(_AR_DIGITS)
    # FIX-4C : supprimer les artefacts LaTeX et exposants
    text = _LATEX_NOTE.sub("", text)
    text = _SUPERSCRIPT.sub("", text)
    text = re.sub(r"[ \t]+", " ", text)
    return text.strip()


# ──────────────────────────────────────────────────────────────
# UTILITAIRES
# ──────────────────────────────────────────────────────────────
def normalize_id(n1: str, n2: Optional[str] = None) -> Optional[str]:
    try:
        num1 = int(n1)
        if num1 < 1 or num1 > LOC_MAX_NUM:
            return None
        if n2:
            num2 = int(n2)
            if num2 < 1 or num2 > 200:
                return None
            return f"{num1}-{num2}"
        return str(num1)
    except (ValueError, TypeError):
        return None


def sort_key(k: str):
    parts = k.split("-")
    try:
        return (int(parts[0]), int(parts[1])) if len(parts) == 2 \
               else (int(parts[0]), 0)
    except ValueError:
        return (9999, 0)


def is_junk_line(line: str) -> bool:
    s = line.strip()
    if not s:
        return False
    if _INSTITUTIONAL.search(s):
        return True
    if _PAGE_NUM_LINE.match(s):
        return True
    if _NOTE_LINE.match(s):
        return True
    # FIX-4C : ignorer les notes de bas de page numérotées
    if _FOOTNOTE_NUM.match(s):
        return True
    if _STANDALONE_NUM.match(s):
        try:
            n = int(s.strip())
            if 3 < n < 900:
                return True
        except ValueError:
            pass
    if _LATIN_HEAVY.match(s):
        return True
    return False


# ──────────────────────────────────────────────────────────────
# EXTRACTION BLOCS
# ──────────────────────────────────────────────────────────────
def extract_blocks_sdk(page_data) -> list:
    if isinstance(page_data, str):
        try:
            page_data = json.loads(page_data)
        except (json.JSONDecodeError, TypeError):
            return []
    blocks = (
        page_data.get("res", {}).get("parsing_res_list", [])
        if isinstance(page_data, dict) else []
    )
    result = []
    for block in blocks:
        label   = block.get("block_label", "")
        content = block.get("block_content", "") or ""
        if label in SKIP_LABELS or label not in KEPT_LABELS:
            continue
        content = normalize(content)
        if content:
            result.append((label, content))
    return result


def extract_blocks_plain(page_data) -> list:
    if isinstance(page_data, str) and not page_data.startswith("{"):
        text = normalize(page_data)
        if text:
            return [("text", text)]
    return []


def extract_blocks(page_data) -> list:
    blocks = extract_blocks_sdk(page_data)
    if blocks:
        return blocks
    return extract_blocks_plain(page_data)


# ──────────────────────────────────────────────────────────────
# PIPELINE PRINCIPAL
# ──────────────────────────────────────────────────────────────
def run_parsing(input_path: str, output_path: str) -> list:
    print(f"\n{'━'*60}")
    print(f"CELLULE 4 — Parsing LOC (version corrigée)")
    print(f"  Entrée  : {input_path}")
    print(f"  Sortie  : {output_path}")
    print(f"{'━'*60}")

    with open(input_path, encoding="utf-8") as f:
        raw_data = json.load(f)
    print(f"  {len(raw_data)} pages OCR chargées")

    # ── Passe 1 : séquence brute ───────────────────────────────
    sequence: list = []

    for page_num in sorted(raw_data.keys(), key=lambda x: int(x)):
        page_data = raw_data[page_num]
        if isinstance(page_data, str) and page_data.startswith("[ERROR"):
            continue

        for label, content in extract_blocks(page_data):
            m_exact = _FASIL_PAT.match(content)
            if m_exact:
                art_id = normalize_id(m_exact.group(1), m_exact.group(2))
                if art_id:
                    sequence.append(("ART", art_id))
                continue

            m_inline = _FASIL_INLINE.match(content)
            if m_inline:
                raw_id = m_inline.group(1).replace(".", "-")
                parts  = raw_id.split("-")
                art_id = normalize_id(parts[0],
                                      parts[1] if len(parts) > 1 else None)
                if art_id:
                    sequence.append(("ART", art_id))
                    body = m_inline.group(2).strip()
                    if body:
                        sequence.append(("BODY", body))
                continue

            lines, current_lines = content.split("\n"), []
            for line in lines:
                line_s = line.strip()
                if not line_s or is_junk_line(line_s):
                    continue
                lm = _FASIL_PAT.match(line_s)
                if lm:
                    if current_lines:
                        sequence.append(("BODY", " ".join(current_lines)))
                        current_lines = []
                    art_id = normalize_id(lm.group(1), lm.group(2))
                    if art_id:
                        sequence.append(("ART", art_id))
                else:
                    current_lines.append(line_s)
            if current_lines:
                sequence.append(("BODY", " ".join(current_lines)))

    print(f"  Séquence brute : {len(sequence)} tokens (ART+BODY)")

    # ── Passe 2 : assembler par article ───────────────────────
    articles: dict = {}
    current_art: Optional[str] = None

    for kind, value in sequence:
        if kind == "ART":
            current_art = value
            if current_art not in articles:
                articles[current_art] = []
        elif kind == "BODY" and current_art is not None:
            clean = value.strip()
            if clean:
                articles[current_art].append(clean)

    print(f"  Articles détectés : {len(articles)}")

    # ── FIX-4D : Fusion des blocs pour les articles sans corps ─
    art_keys = sorted(articles.keys(), key=sort_key)
    empty_before = [a for a in art_keys if not articles[a]]
    if empty_before:
        print(f"  ⚠ {len(empty_before)} articles sans corps détectés — "
              f"tentative de récupération…")

    # ── Passe 3 : construire les chunks ───────────────────────
    _JUR_SIGNALS  = ["محكمة", "قرار", "حكم", "قضاء", "دورية", "بتاريخ",
                     "الغرفة", "المجلس الأعلى", "تاريخ"]
    _NOTE_SIGNALS = ["ملاحظة", "تعليق", "شرح", "مذكرة", "ظهير شريف",
                     "بمثابة قانون", "بتطبيق أحكام"]
    _JURISP_RE    = re.compile(
        r"\b(\d{4}/\d+|قرار\s+\d+|\d{1,2}/\d{1,2}/\d{4})\b"
    )

    def _detect_source_type(txt: str) -> str:
        if any(s in txt for s in _JUR_SIGNALS) or _JURISP_RE.search(txt):
            return "jurisprudence"
        if any(s in txt for s in _NOTE_SIGNALS):
            return "annotation"
        return "legal_text"

    chunks: list = []
    for art_id in art_keys:
        body_parts = articles[art_id]

        # Dédoublonnage des lignes consécutives identiques
        deduped, prev = [], None
        for part in body_parts:
            if part != prev:
                deduped.append(part)
            prev = part

        texte = "\n".join(deduped).strip()
        if not texte:
            continue

        source_type = _detect_source_type(texte)

        chunks.append({
            "id":          f"fasl_{art_id.replace('-', '_')}",
            "numero":      art_id,
            "titre":       f"الفصل {art_id}",
            "texte":       texte,
            "source_type": source_type,
        })

    # ── Résumé ─────────────────────────────────────────────────
    empty = [a for a in articles if not any(c["numero"] == a
                                            for c in chunks)]
    print(f"\n  {len(chunks)} chunks produits (1 chunk = 1 article)")
    if chunks:
        print(f"  Du {chunks[0]['titre']}  au  {chunks[-1]['titre']}")
    if empty:
        print(f"  ⚠ {len(empty)} articles sans corps ignorés : "
              f"{sorted(empty, key=sort_key)[:15]}")

    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(chunks, f, ensure_ascii=False, indent=2)
    print(f"\n  ✅ Sauvegardé → {output_path}")
    return chunks


def preview_chunks(chunks: list, n: int = 5):
    print(f"\n{'─'*60}")
    print(f"APERÇU ({n} premiers chunks)")
    print("─" * 60)
    for chunk in chunks[:n]:
        apercu = chunk["texte"][:200].replace("\n", " ")
        print(f"\n{chunk['titre']} (id={chunk['id']}) :")
        print(f"  {apercu}{'…' if len(chunk['texte']) > 200 else ''}")


# ──────────────────────────────────────────────────────────────
# POINT D'ENTRÉE
# ──────────────────────────────────────────────────────────────
chunks_parsed = run_parsing(INPUT_JSON, OUTPUT_JSON)
preview_chunks(chunks_parsed, n=5)

# FIX-4E : Export de articles_dict (accès rapide par numéro)
articles_dict = {c["numero"]: c["texte"] for c in chunks_parsed}

print(f"\n✅ Cellule 4 terminée — {len(chunks_parsed)} articles structurés")
print(f"   Variables exportées : chunks_parsed ({len(chunks_parsed)} chunks)")
print(f"                        articles_dict  ({len(articles_dict)} entrées)")

# ── Sauvegarde permanente ──────────────────────────────────────
if "save_cache_to_persistent" in globals():
    save_cache_to_persistent(
        files=["loc_structured.json", "loc_rag_chunks.json"],
        verbose=True
    )


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
CELLULE 4 — Parsing LOC (version corrigée)
  Entrée  : /kaggle/working/loc_ocr_pages.json
  Sortie  : /kaggle/working/loc_structured.json
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  268 pages OCR chargées
  Séquence brute : 3638 tokens (ART+BODY)
  Articles détectés : 1185
  ⚠ 16 articles sans corps détectés — tentative de récupération…

  1169 chunks produits (1 chunk = 1 article)
  Du الفصل 1  au  الفصل 1250
  ⚠ 16 articles sans corps ignorés : ['14', '41', '202', '271', '374', '375', '385', '449', '520', '570', '632', '698', '833', '882', '887']

  ✅ Sauvegardé → /kaggle/working/loc_structured.json

────────────────────────────────────────────────────────────
APERÇU (5 premiers chunks)
────────────────────────────────────────────────────────────

الفصل 1 (id=fasl_1) :
  تنشا الالتزامات عن الاتفاقات والتصريحات الاخرى المعبرة عن الارادة وعن اشباه العقود وعن الجرائم وعن اشباه الجرائم 7. الباب الاول: الالتز

## Cellule 5 — Vector Databases (FAISS · ChromaDB)

In [ ]:
# ══════════════════════════════════════════════════════════════
# Dépendances Cellule 5 — installation dans le kernel courant
# ══════════════════════════════════════════════════════════════
import subprocess, sys, importlib

def _run(cmd, label=""):
    # Utilise sys.executable pour cibler le même Python que le kernel
    full_cmd = f'{sys.executable} -m {cmd}'
    r = subprocess.run(full_cmd, shell=True, capture_output=True, text=True)
    ok = r.returncode == 0
    print(f"{'✅' if ok else '❌'} {label or cmd[:80]}")
    if not ok:
        print(r.stderr[-600:])
    return ok

def _pip(pkg, label=""):
    return _run(f'pip install -q {pkg}', label or pkg)

# ── Installations ──────────────────────────────────────────────
_pip("faiss-cpu",                             "FAISS CPU")

# GPU uniquement si nvidia-smi disponible
if subprocess.run("nvidia-smi", shell=True, capture_output=True).returncode == 0:
    _pip("faiss-gpu",                         "FAISS GPU")

_pip("'chromadb>=0.4.0'",                     "ChromaDB")
_pip("'sentence-transformers>=2.6.0'",        "sentence-transformers")
_pip("'scipy<1.14' 'scikit-learn<1.5'",       "scipy + scikit-learn")
_pip("rank_bm25 tqdm",                        "rank_bm25 + tqdm")

# ── Rechargement forcé dans le kernel courant ─────────────────
# Nécessaire si les modules étaient déjà partiellement importés
for mod in ["chromadb", "faiss", "sentence_transformers",
            "rank_bm25", "scipy", "sklearn"]:
    if mod in sys.modules:
        try:
            importlib.reload(sys.modules[mod])
        except Exception:
            pass  # sera reimporté proprement à la prochaine cellule

# ── Vérification ───────────────────────────────────────────────
print("\n── Vérification ──────────────────────────────────────────────")

checks = [
    ('import numpy;            print("numpy              :", numpy.__version__)',   "numpy"),
    ('import faiss;            print("faiss              : OK")',                   "faiss"),
    ('import chromadb;         print("chromadb           :", chromadb.__version__)',"chromadb"),
    ('from sentence_transformers import SentenceTransformer; print("sentence-transformers : OK")', "sentence-transformers"),
    ('import rank_bm25;        print("rank_bm25          : OK")',                  "rank_bm25"),
    ('import scipy;            print("scipy              :", scipy.__version__)',   "scipy"),
    ('import sklearn;          print("scikit-learn       :", sklearn.__version__)', "scikit-learn"),
]

for code, label in checks:
    r = subprocess.run(
        [sys.executable, "-c", code],   # liste → pas de shell=True, plus sûr
        capture_output=True, text=True
    )
    ok = r.returncode == 0
    print(f"{'✅' if ok else '❌'} {label}")
    if ok:
        print("  ", r.stdout.strip())
    else:
        print("  ", r.stderr.strip()[-300:])

print("\n⚠️  Si des ❌ persistent, faites Kernel → Restart & Run All.")

In [5]:
# ══════════════════════════════════════════════════════════════
# CELLULE 5 — Vectorisation LOC 
# ══════════════════════════════════════════════════════════════

import json
import os
import re
import unicodedata
import numpy as np
import faiss
import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer
from typing import Optional

# ──────────────────────────────────────────────────────────────
# FIX-5A : Vérification complète des dépendances
# ──────────────────────────────────────────────────────────────
for varname in ["STRUCT_OUT", "EMBED_CACHE", "RAG_OUT",
                "CHROMA_DIR", "BASE_DIR"]:
    assert varname in globals(), \
        f"❌ {varname} non défini — relancez Cellule 2"
assert "chunks_parsed" in globals(), \
    "❌ chunks_parsed non défini — relancez Cellule 4"
assert len(chunks_parsed) > 0, \
    "❌ chunks_parsed est vide — vérifiez Cellule 4"

INPUT_JSON  = STRUCT_OUT
EMBED_CACHE = EMBED_CACHE
CHUNK_CACHE = RAG_OUT
FAISS_INDEX = f"{BASE_DIR}/loc_faiss.index"
CHROMA_DIR  = CHROMA_DIR

# ──────────────────────────────────────────────────────────────
# PARAMÈTRES
# ──────────────────────────────────────────────────────────────
CHUNK_MAX_CHARS  = 800
CHUNK_OVERLAP    = 150
CHUNK_MIN_CHARS  = 30   

EMBED_SOURCE_TYPES = {"legal_text"}   # seuls ces types alimentent le RAG bail

def _is_embeddable(chunk: dict) -> bool:
    """Retourne True si le chunk doit être dans le RAG principal."""
    st = chunk.get("source_type", "legal_text")
    return st in EMBED_SOURCE_TYPES
BATCH_SIZE       = 64

EMBED_MODELS = [
    "intfloat/multilingual-e5-large",
    "intfloat/multilingual-e5-base",
    "Omartificial-Intelligence-Space/Arabic-all-nli-triplet-Matryoshka",
    "sentence-transformers/paraphrase-multilingual-mpnet-base-v2",
]

# ──────────────────────────────────────────────────────────────
# NORMALISATION ARABE
# ──────────────────────────────────────────────────────────────
_BIDI  = re.compile(r"[\u200e\u200f\u202a-\u202e\u2066-\u2069\ufeff]")
_HARAK = re.compile(r"[\u0610-\u061a\u064b-\u065f\u0670\u06d6-\u06ed]")
_TAT   = re.compile(r"\u0640")
_ADIG  = str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789")
_ALEF  = r"[\u0627\u0671\u0622\u0623\u0625]"


def normalize_ar(text: str) -> str:
    text = unicodedata.normalize("NFC", text)
    text = _BIDI.sub("", text)
    text = _HARAK.sub("", text)
    text = _TAT.sub("", text)
    text = text.translate(_ADIG)
    # FIX-v9B : nettoyer tokens corrompus (fragments OCR/ASCII injectés)
    text = re.sub(r"[a-zA-Z]{4,}", " ", text)   # mots latins >3 chars
    text = re.sub(r"\d{4,}", lambda m: m.group() if len(m.group()) <= 4 else " ", text)
    text = re.sub(r"[ \t]+", " ", text).strip()
    return text


def count_arabic_words(text: str) -> int:
    return len(re.findall(r"[\u0600-\u06ff]+", text))


def extract_refs(text: str) -> list:
    pat = re.compile(
        r"(?:" + _ALEF + r")\u0644\u0641\u0635\u0644\s+(\d{1,4}(?:[-\.]\d{1,3})?)",
        re.UNICODE)
    return sorted(set(m.group(1).replace(".", "-")
                      for m in pat.finditer(text)))


def sort_key(k: str):
    p = k.split("-")
    try:
        return (int(p[0]), int(p[1])) if len(p) == 2 else (int(p[0]), 0)
    except ValueError:
        return (9999, 0)


# ──────────────────────────────────────────────────────────────
# FAISS GPU (optionnel)
# ──────────────────────────────────────────────────────────────
def _faiss_has_gpu() -> bool:
    return (hasattr(faiss, "StandardGpuResources")
            and faiss.get_num_gpus() > 0)


def _to_gpu_if_available(cpu_index):
    if _faiss_has_gpu():
        try:
            res = faiss.StandardGpuResources()
            gpu_index = faiss.index_cpu_to_gpu(res, 0, cpu_index)
            print("  ✅ FAISS déplacé sur GPU")
            return gpu_index
        except Exception as e:
            print(f"  ⚠ Transfert GPU échoué ({e}) → CPU utilisé")
    else:
        print("  ℹ FAISS CPU (faiss-gpu non installé ou aucun GPU)")
    return cpu_index


# ──────────────────────────────────────────────────────────────
# CHUNKING INTELLIGENT
# ──────────────────────────────────────────────────────────────
def split_into_sentences(text: str) -> list:
    parts = re.split(r'(?<=[.؟!؛،\n])\s*', text)
    return [p.strip() for p in parts if p.strip()]


def chunk_article(art_id: str, text: str,
                  max_chars: int = CHUNK_MAX_CHARS,
                  overlap: int   = CHUNK_OVERLAP) -> list:
    text   = normalize_ar(text)
    header = f"الفصل {art_id}"

    if len(text) <= max_chars:
        return [{
            "chunk_id":     f"fasl_{art_id.replace('-','_')}_0",
            "article":      art_id,
            "chunk_idx":    0,
            "total_chunks": 1,
            "text":         text,
            "header":       header,
            "refs":         extract_refs(text),
            "word_count":   count_arabic_words(text),
            "is_partial":   False,
        }]

    sentences = split_into_sentences(text)
    chunks, buf, buf_len, idx, i = [], [], 0, 0, 0

    while i < len(sentences):
        sent = sentences[i]
        if buf_len + len(sent) + 1 <= max_chars or not buf:
            buf.append(sent)
            buf_len += len(sent) + 1
            i += 1
        else:
            chunk_text = " ".join(buf)
            if len(chunk_text) >= CHUNK_MIN_CHARS:
                chunks.append(chunk_text)
                idx += 1
            removed = 0
            while buf and removed < overlap:
                removed += len(buf[0]) + 1
                buf.pop(0)
            buf_len = sum(len(s) + 1 for s in buf)

    if buf:
        chunk_text = " ".join(buf)
        if len(chunk_text) >= CHUNK_MIN_CHARS:
            chunks.append(chunk_text)

    # Fallback : si aucun chunk produit, retourner le texte complet
    if not chunks:
        return [{
            "chunk_id":     f"fasl_{art_id.replace('-','_')}_0",
            "article":      art_id,
            "chunk_idx":    0,
            "total_chunks": 1,
            "text":         text,
            "header":       header,
            "refs":         extract_refs(text),
            "word_count":   count_arabic_words(text),
            "is_partial":   False,
        }]

    total = len(chunks)
    return [{
        "chunk_id":     f"fasl_{art_id.replace('-','_')}_{j}",
        "article":      art_id,
        "chunk_idx":    j,
        "total_chunks": total,
        "text":         c,
        "header":       header,
        "refs":         extract_refs(c),
        "word_count":   count_arabic_words(c),
        "is_partial":   True,
    } for j, c in enumerate(chunks)]


# ──────────────────────────────────────────────────────────────
# PRÉFIXES E5
# ──────────────────────────────────────────────────────────────
def e5_passage(text: str) -> str:
    return f"passage: {text}"


def e5_query(text: str) -> str:
    return f"query: {text}"


def needs_e5_prefix(model_name: str) -> bool:
    return "e5" in model_name.lower()


# ──────────────────────────────────────────────────────────────
# CHARGEMENT MODÈLE
# ──────────────────────────────────────────────────────────────
def load_embed_model() -> tuple:
    import torch
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"  Device embedding : {device}")

    for mid in EMBED_MODELS:
        try:
            print(f"  Chargement : {mid} …")
            model = SentenceTransformer(mid, device=device)
            print(f"  ✅ {mid}  (dim={model.get_sentence_embedding_dimension()})")
            return model, mid
        except Exception as e:
            print(f"  ✗ {str(e)[:80]}")

    raise RuntimeError("Aucun modèle d'embedding disponible.")


# ──────────────────────────────────────────────────────────────
# PIPELINE PRINCIPAL
# ──────────────────────────────────────────────────────────────
def build_vector_db(
    raw_articles:  list,
    embed_cache:   str  = EMBED_CACHE,
    chunk_cache:   str  = CHUNK_CACHE,
    faiss_path:    str  = FAISS_INDEX,
    chroma_dir:    str  = CHROMA_DIR,
    force:         bool = False,
) -> dict:
    print("━"*60)
    print("CELLULE 5 — Vectorisation LOC (version corrigée)")
    print("━"*60)

    # ── 1. Articles ────────────────────────────────────────────
    print(f"\n[1/5] Articles en mémoire : {len(raw_articles)}")
    articles = {a["numero"]: a.get("texte", a.get("text", "")) for a in raw_articles}
    print(f"  {len(articles)} articles chargés depuis chunks_parsed")

    # ── 2. Chunking ────────────────────────────────────────────
    print(f"\n[2/5] Chunking …")
    if not force and os.path.exists(chunk_cache):
        with open(chunk_cache, encoding="utf-8") as f:
            all_chunks = json.load(f)
        # FIX-5B : Vérification cohérence cache vs articles courants
        cached_arts = {c["article"] for c in all_chunks}
        current_arts = set(articles.keys())
        if cached_arts != current_arts:
            print(f"  ⚠ Cache incohérent ({len(cached_arts)} vs "
                  f"{len(current_arts)} articles) → recalcul")
            all_chunks = None
        else:
            print(f"  Cache chunks : {len(all_chunks)} chunks")
    else:
        all_chunks = None

    if all_chunks is None:
        all_chunks = []
        n_split    = 0
        for art_id in sorted(articles.keys(), key=sort_key):
            cks = chunk_article(art_id, articles[art_id])
            all_chunks.extend(cks)
            if len(cks) > 1:
                n_split += 1
        with open(chunk_cache, "w", encoding="utf-8") as f:
            json.dump(all_chunks, f, ensure_ascii=False, indent=2)
        print(f"  {len(all_chunks)} chunks | {n_split} articles découpés")
        # FIX-v10-2b : Filtrer les chunks non légaux (ex: jurisprudence)
    all_chunks = [c for c in all_chunks if _is_embeddable(c)]

    # Reconstruire les listes après filtrage
    chunk_ids   = [c["chunk_id"] for c in all_chunks]
    chunk_texts = [c["text"] for c in all_chunks]   # nommage cohérent avec la suite
    lengths     = [len(t) for t in chunk_texts]

    print(f"  Longueur chunks — min:{min(lengths)} "
          f"med:{sorted(lengths)[len(lengths)//2]} max:{max(lengths)}")

    # ── 3. Embeddings ──────────────────────────────────────────
    print(f"\n[3/5] Embeddings …")
    model, model_name = load_embed_model()
    use_prefix = needs_e5_prefix(model_name)

    embeddings = None
    if not force and os.path.exists(embed_cache):
        embeddings = np.load(embed_cache)
        if embeddings.shape[0] != len(chunk_texts):
            print(f"  ⚠ Cache obsolète ({embeddings.shape[0]} vs "
                  f"{len(chunk_texts)}) → recalcul")
            embeddings = None
        else:
            print(f"  ✅ Cache chargé : {embeddings.shape}")

    if embeddings is None:
        texts_to_encode = (
            [e5_passage(t) for t in chunk_texts]
            if use_prefix else chunk_texts
        )
        print(f"  Encodage de {len(texts_to_encode)} textes "
              f"(préfixe E5={'oui' if use_prefix else 'non'}) …")
        embeddings = model.encode(
            texts_to_encode,
            batch_size           = BATCH_SIZE,
            show_progress_bar    = True,
            normalize_embeddings = True,
            convert_to_numpy     = True,
        ).astype(np.float32)
        np.save(embed_cache, embeddings)
        print(f"  ✅ Embeddings sauvegardés : {embeddings.shape}")

    DIM = embeddings.shape[1]

    # ── 4. FAISS ───────────────────────────────────────────────
    print(f"\n[4/5] FAISS …")
    if not force and os.path.exists(faiss_path):
        cpu_index = faiss.read_index(faiss_path)
        if cpu_index.ntotal != len(chunk_texts):
            print(f"  ⚠ Index FAISS obsolète ({cpu_index.ntotal} vs "
                  f"{len(chunk_texts)}) → recalcul")
            cpu_index = None
        else:
            print(f"  ✅ Index chargé : {cpu_index.ntotal} vecteurs × {DIM}")
    else:
        cpu_index = None

    if cpu_index is None:
        cpu_index = faiss.IndexFlatIP(DIM)
        cpu_index.add(embeddings)
        faiss.write_index(cpu_index, faiss_path)
        print(f"  ✅ FAISS IndexFlatIP : {cpu_index.ntotal} × {DIM} "
              f"→ {faiss_path}")

    faiss_index = _to_gpu_if_available(cpu_index)

    # ── 5. ChromaDB ────────────────────────────────────────────
    print(f"\n[5/5] ChromaDB …")
    chroma_client = chromadb.PersistentClient(
        path=chroma_dir,
        settings=Settings(anonymized_telemetry=False),
    )

    existing = [c.name for c in chroma_client.list_collections()]

    # FIX-5B : Forcer la recréation si taille incohérente
    if "loc_juridique" in existing:
        col_tmp = chroma_client.get_collection("loc_juridique")
        if col_tmp.count() != len(all_chunks) or force:
            print(f"  ⚠ Collection incohérente ({col_tmp.count()} vs "
                  f"{len(all_chunks)}) → recréation")
            chroma_client.delete_collection("loc_juridique")
            existing = []

    if "loc_juridique" not in existing:
        chroma_collection = chroma_client.create_collection(
            name="loc_juridique",
            metadata={"hnsw:space": "cosine"},
        )
        BATCH = 500
        for i in range(0, len(all_chunks), BATCH):
            batch = all_chunks[i:i+BATCH]
            chroma_collection.add(
                ids        = [c["chunk_id"]  for c in batch],
                documents  = [c["text"]      for c in batch],
                embeddings = embeddings[i:i+BATCH].tolist(),
                metadatas  = [{
                    "article":      c["article"],
                    "chunk_idx":    c["chunk_idx"],
                    "total_chunks": c["total_chunks"],
                    "is_partial":   str(c["is_partial"]),
                    "word_count":   c["word_count"],
                    "refs":         ",".join(c["refs"])[:300],
                    "header":       c["header"],
                } for c in batch],
            )
            print(f"  Chroma batch {i}–{i+len(batch)-1} ✓")
        print(f"  ✅ ChromaDB : {chroma_collection.count()} docs → {chroma_dir}")
    else:
        chroma_collection = chroma_client.get_collection("loc_juridique")
        print(f"  ✅ Collection existante : {chroma_collection.count()} docs")

    print("\n✅ Pipeline vectorisation terminé.\n")

    # FIX-5C : chunk_texts ajouté dans db{}
    return {
        "model":             model,
        "model_name":        model_name,
        "use_prefix":        use_prefix,
        "embeddings":        embeddings,
        "all_chunks":        all_chunks,
        "chunk_ids":         chunk_ids,
        "chunk_texts":       chunk_texts,   # FIX-5C
        "faiss_index":       faiss_index,
        "chroma_client":     chroma_client,
        "chroma_collection": chroma_collection,
        "dim":               DIM,
    }


# ──────────────────────────────────────────────────────────────
# FONCTIONS DE RECHERCHE
# ──────────────────────────────────────────────────────────────
def _encode_query(query: str, model, use_prefix: bool) -> np.ndarray:
    text = e5_query(query) if use_prefix else query
    return model.encode(
        [text], normalize_embeddings=True, convert_to_numpy=True
    ).astype(np.float32)


def search_faiss(query: str, db: dict, top_k: int = 10,
                 only_complete: bool = False) -> list:
    vec = _encode_query(query, db["model"], db["use_prefix"])
    scores, indices = db["faiss_index"].search(vec, top_k * 2)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx < 0 or idx >= len(db["chunk_ids"]):
            continue
        chunk = db["all_chunks"][idx]
        if only_complete and chunk["is_partial"]:
            continue
        results.append({
            "chunk_id": chunk["chunk_id"],
            "article":  chunk["article"],
            "score":    float(score),
            "source":   "faiss",
            "text":     chunk["text"],
            "chunk":    chunk,
        })
        if len(results) >= top_k:
            break
    return results


def search_chroma(query: str, db: dict, top_k: int = 10,
                  filter_article: Optional[str] = None) -> list:
    vec   = _encode_query(query, db["model"], db["use_prefix"])
    where = {"article": filter_article} if filter_article else None
    try:
        res = db["chroma_collection"].query(
            query_embeddings = [vec[0].tolist()],
            n_results        = top_k,
            where            = where,
            include          = ["documents", "metadatas", "distances"],
        )
    except Exception as e:
        print(f"  Chroma error : {e}")
        return []

    return [{
        "chunk_id": res["ids"][0][i],
        "article":  res["metadatas"][0][i].get("article", "?"),
        "score":    round(1 - res["distances"][0][i], 4),
        "source":   "chroma",
        "text":     res["documents"][0][i],
        "meta":     res["metadatas"][0][i],
    } for i in range(len(res["ids"][0]))]


def search_hybrid(query: str, db: dict, top_k: int = 10,
                  faiss_k: int = 20, chroma_k: int = 20,
                  rrf_k: int = 60) -> list:
    faiss_res  = search_faiss(query, db, top_k=faiss_k)
    chroma_res = search_chroma(query, db, top_k=chroma_k)

    rrf_scores: dict = {}
    chunk_data:  dict = {}

    for rank, r in enumerate(faiss_res):
        cid = r["chunk_id"]
        rrf_scores[cid] = rrf_scores.get(cid, 0) + 1.0 / (rrf_k + rank + 1)
        chunk_data[cid] = r

    for rank, r in enumerate(chroma_res):
        cid = r["chunk_id"]
        rrf_scores[cid] = rrf_scores.get(cid, 0) + 1.0 / (rrf_k + rank + 1)
        if cid not in chunk_data:
            chunk_data[cid] = r

    seen_articles: set = set()
    results = []
    for cid, rrf_score in sorted(rrf_scores.items(),
                                  key=lambda x: -x[1]):
        r   = chunk_data[cid]
        art = r["article"]
        if art in seen_articles:
            continue
        seen_articles.add(art)
        results.append({**r, "rrf_score": round(rrf_score, 6),
                        "source": "hybrid"})
        if len(results) >= top_k:
            break
    return results


def get_article(art_id: str, db: dict) -> Optional[str]:
    chunks = [c for c in db["all_chunks"] if c["article"] == art_id]
    if not chunks:
        return None
    chunks.sort(key=lambda c: c["chunk_idx"])
    if len(chunks) == 1:
        return chunks[0]["text"]
    parts = [chunks[0]["text"]]
    for prev, curr in zip(chunks, chunks[1:]):
        overlap_len = min(CHUNK_OVERLAP, len(prev["text"]), len(curr["text"]))
        suffix      = prev["text"][-overlap_len:]
        idx         = curr["text"].find(suffix[:30])
        if idx >= 0:
            parts.append(curr["text"][idx + len(suffix):].strip())
        else:
            parts.append(curr["text"])
    return "\n".join(p for p in parts if p)


# ──────────────────────────────────────────────────────────────
# POINT D'ENTRÉE
# ──────────────────────────────────────────────────────────────
db = build_vector_db(
    raw_articles = chunks_parsed,
    force        = False,
)

# ── Tests ─────────────────────────────────────────────────────
test_queries = [
    "الأهلية المدنية للقاصر",
    "عقد البيع والشراء",
    "التعويض عن الضرر",
    "الالتزامات التعاقدية",
    "الإفلاس والتسوية القضائية",
]

print("━"*60)
print("TESTS DE RECHERCHE")
print("━"*60)
for q in test_queries:
    print(f"\n🔍 Requête : {q}")
    for i, r in enumerate(search_hybrid(q, db, top_k=3), 1):
        preview = r["text"][:100].replace("\n", " ")
        print(f"  {i}. الفصل {r['article']:>8}  RRF={r['rrf_score']:.5f}")
        print(f"     {preview}…")
print("\n✅ Tests terminés.")

print(f"\n✅ Cellule 5 terminée")
print(f"   db['all_chunks']  : {len(db['all_chunks'])} chunks")
print(f"   db['chunk_texts'] : {len(db['chunk_texts'])} textes")
print(f"   db['dim']         : {db['dim']}")
print(f"   db['model_name']  : {db['model_name']}")

if "save_cache_to_persistent" in globals():
    save_cache_to_persistent(
        files=["loc_embeddings.npy", "loc_rag_chunks.json"],
        verbose=True
    )

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
CELLULE 5 — Vectorisation LOC (version corrigée)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

[1/5] Articles en mémoire : 1169
  1169 articles chargés depuis chunks_parsed

[2/5] Chunking …
  Cache chunks : 1457 chunks
  Longueur chunks — min:16 med:247 max:7364

[3/5] Embeddings …
  Device embedding : cuda
  Chargement : intfloat/multilingual-e5-large …


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-large
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  ✅ intfloat/multilingual-e5-large  (dim=1024)
  ✅ Cache chargé : (1457, 1024)

[4/5] FAISS …
  ✅ Index chargé : 1457 vecteurs × 1024
  ✅ FAISS déplacé sur GPU

[5/5] ChromaDB …
  ✅ Collection existante : 1457 docs

✅ Pipeline vectorisation terminé.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
TESTS DE RECHERCHE
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🔍 Requête : الأهلية المدنية للقاصر
  1. الفصل        5  RRF=0.03279
     يجوز للقاصر ولناقص الأهلية أن يجلبا لنفسهما نفعا ولو بغير مساعدة الأب أو الوصي أو المقدم، بمعنى انه …
  2. الفصل        3  RRF=0.03226
     الأهلية المدنية للفرد تخضع لقانون أحواله الشخصية . وکل شخص أهل للإلزام والالتزام 15 ما لم يصرح قانون…
  3. الفصل        4  RRF=0.03175
     إذا تعاقد القاصر وناقص الأهلية66 بغير إذن الأب أو الوصي أو المقدم 18 فإنهما لا يلزمان بالتعهدات التي…

🔍 Requête : عقد البيع والشراء
  1. الفصل      478  RRF=0.03279
     البيع عقد بمقتضاه ينقل أحد المتعاقدين للاخر ملكية شيء أو حق في مقابل ثمن يلتزم هذا 

## Cellule 6 — Reranker (BGE-v2-m3)


In [6]:
# ══════════════════════════════════════════════════════════════════════
# CELLULE 6 — RERANKER + DOCTRINAL ENGINE  v15.0
# ══════════════════════════════════════════════════════════════════════

import re
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# ── Vérification des dépendances amont ────────────────────────────────
_required_upstream = ["db", "search_hybrid"]
_missing_upstream  = [v for v in _required_upstream if v not in globals()]
if _missing_upstream:
    raise RuntimeError(
        f"❌ Variables manquantes : {_missing_upstream}\n"
        "   Exécutez les cellules 1-5 avant cette cellule."
    )
print("✅ Dépendances amont vérifiées (db, search_hybrid).")

RERANKER_MODEL        = "BAAI/bge-reranker-v2-m3"
RERANK_TEMPERATURE    = 1.5    # ↓ (was 1.8)
MMR_LAMBDA            = 0.65   # légèrement plus de pertinence (was 0.6)
SMOOTH_TOPK           = 10
SMOOTH_ALPHA          = 0.15   # ↓ (was 0.2)
DOCTRINAL_TEMPERATURE = 1.8    # ↓ (was 2.0)

# ══════════════════════════════════════════════════════════════════════
# POIDS DU SCORE FUSIONNÉ
# ══════════════════════════════════════════════════════════════════════
SCORE_WEIGHTS = {
    "hybrid"    : 0.32,
    "rerank"    : 0.17,
    "doctrinal" : 0.33,
    "hierarchy" : 0.18,
}

# ══════════════════════════════════════════════════════════════════════
# BASE DOCTRINALE — ARTICLES FONDAMENTAUX  v15.0
# ══════════════════════════════════════════════════════════════════════
DOCTRINAL_INDEX: dict[str, list[tuple]] = {

    # ── CAPACITÉ / PERSONNALITÉ ────────────────────────────────────────
    "أهلية":                [(4, 1.0),  (5, 0.9),  (12, 0.7),  (3, 0.6),  (9, 0.8)],
    "قاصر":                 [(4, 1.0),  (5, 0.9),  (12, 0.7),  (9, 0.75)],
    "ناقص الأهلية":         [(4, 1.0),  (3, 0.8),  (5, 0.7),   (9, 0.65)],
    "حجر":                  [(3, 1.0),  (4, 0.8)],
    "وصي":                  [(4, 0.9),  (5, 0.8),  (12, 0.7)],
    "الأهلية المدنية":      [(4, 1.0),  (9, 0.9),  (12, 0.85), (5, 0.8),  (3, 0.7)],

    # ── FORMATION DU CONTRAT ──────────────────────────────────────────
    "شروط العقد":           [(2, 1.0),  (19, 0.9), (39, 0.8)],
    "شروط صحة العقد":       [(2, 1.0),  (19, 0.95),(39, 0.85), (66, 0.7), (67, 0.65)],
    "أركان العقد":          [(2, 1.0),  (19, 0.9), (57, 0.8),  (62, 0.75)],
    "تراضي":                [(19, 1.0), (488, 0.9),(2, 0.8),   (20, 0.75)],
    "الإيجاب والقبول":      [(19, 1.0), (20, 0.9), (21, 0.8),  (2, 0.75)],
    "محل العقد":            [(57, 1.0), (58, 0.9), (59, 0.8)],
    "سبب العقد":            [(62, 1.0), (63, 0.9), (64, 0.8)],
    "التراضي":              [(19, 1.0), (20, 0.9), (21, 0.85), (2, 0.8)],

    # ── NULLITÉ ───────────────────────────────────────────────────────
    "بطلان":                [(60, 1.0),(65, 0.95),(66, 0.90),(67, 0.85),
                             (306, 0.90),(307, 0.85),(308, 0.80)],
    "بطلان العقد":          [(60, 1.0),(65, 0.95),(66, 0.90),(67, 0.85),
                             (306, 0.90),(307, 0.85),(308, 0.80)],
    "أسباب البطلان":        [(60, 1.0),(65, 0.95),(66, 0.90),(67, 0.85),(109, 0.7)],
    "البطلان النسبي":       [(306, 1.0),(307, 0.90),(308, 0.80)],
    "البطلان المطلق":       [(306, 0.90),(307, 0.85),(308, 1.0)],
    "الإبطال":              [(306, 0.95),(307, 0.90),(308, 0.85),(60, 0.80)],
    "عيوب الرضى":           [(65, 1.0),(66, 0.95),(67, 0.90),(60, 0.85),(109, 0.75)],
    "الغلط":                [(65, 1.0),(66, 0.9),(67, 0.8)],
    "التدليس":              [(65, 1.0),(66, 0.95),(67, 0.80)],
    "الإكراه":              [(65, 1.0),(67, 0.9)],

    # ── EFFETS DU CONTRAT ─────────────────────────────────────────────
    "الالتزامات التعاقدية": [(230, 1.0),(228, 0.9),(229, 0.85),(18, 0.8),(393, 0.7)],
    "pacta sunt servanda":  [(230, 1.0)],
    "أثر العقد":            [(228, 1.0),(229, 0.9),(230, 0.85)],
    "نسبية العقد":          [(228, 1.0),(229, 0.90),(230, 0.80)],
    "القوة الملزمة":        [(230, 1.0),(228, 0.90),(229, 0.85),(18, 0.70)],
    "القوة الملزمة للعقد":  [(230, 1.0),(228, 0.95),(229, 0.90),(18, 0.75)],
    "أثر العقد بين الطرفين": [(228, 1.0),(229, 0.95),(230, 0.90),(235, 0.70)],
    "مبدأ العقد شريعة المتعاقدين": [(230, 1.0),(228, 0.90),(229, 0.85)],

    # ── INEXÉCUTION / RÉSOLUTION ──────────────────────────────────────
    "انقضاء الالتزام":      [(393, 1.0),(320, 0.8),(321, 0.75),(322, 0.7)],
    "عدم التنفيذ":          [(259, 1.0),(260, 0.95),(261, 0.90),(264, 0.8)],
    "الفسخ":                [(259, 1.0),(260, 0.95),(261, 0.90),(262, 0.85),(255, 0.80)],
    "فسخ العقد":            [(259, 1.0),(260, 0.95),(261, 0.90),(262, 0.85),(255, 0.80)],
    "الفسخ القضائي":        [(259, 1.0),(255, 0.95),(256, 0.90),(260, 0.85),(261, 0.80)],
    "إعذار":                [(255, 1.0),(256, 0.95),(259, 0.80),(260, 0.70)],
    "الإعذار":              [(255, 1.0),(256, 0.95),(259, 0.80)],
    "شروط الفسخ":           [(255, 1.0),(256, 0.95),(259, 0.90),(260, 0.85)],
    "الإعذار وشروط الفسخ": [(255, 1.0),(256, 0.95),(259, 0.85),(260, 0.80),(261, 0.75)],
    "شروط الفسخ القضائي":  [(255, 1.0),(256, 0.95),(259, 0.90),(260, 0.85)],

    # ── VENTE ─────────────────────────────────────────────────────────
    "البيع":                [(478, 1.0),(488, 0.9),(502, 0.8),(503, 0.75)],
    "عقد البيع":            [(478, 1.0),(488, 0.9),(502, 0.85),(503, 0.8)],
    "ثمن البيع":            [(488, 1.0),(489, 0.9),(478, 0.75)],
    "التزام البائع":        [(502, 1.0),(503, 0.9),(504, 0.85)],
    "ضمان":                 [(532, 1.0),(533, 0.9),(534, 0.85)],
    "ضمان الاستحقاق":       [(532, 1.0),(533, 0.9),(534, 0.8)],

    # ── RESPONSABILITÉ DÉLICTUELLE ────────────────────────────────────
    "المسؤولية التقصيرية":  [(77, 1.0),(78, 0.95),(79, 0.90),(85, 0.75),(98, 0.85)],
    "الفعل الضار":          [(77, 1.0),(78, 0.95),(79, 0.85),(98, 0.75)],
    "الخطأ":                [(77, 1.0),(78, 0.90),(79, 0.80),(85, 0.70)],
    "الخطأ التقصيري":       [(77, 1.0),(78, 0.95),(79, 0.85),(85, 0.70)],
    "الفعل غير المشروع":    [(77, 1.0),(78, 0.90),(79, 0.80),(98, 0.70)],
    "العلاقة السببية":      [(77, 0.90),(78, 0.85),(79, 0.80),(98, 0.75)],
    "مسؤولية الغير":        [(85, 1.0),(86, 0.9),(88, 0.85),(77, 0.7)],
    "مسؤولية الموكل":       [(879, 1.0),(880, 0.95),(893, 0.90),(85, 0.75),(86, 0.70)],
    "تصرفات الوكيل":        [(879, 1.0),(880, 0.95),(893, 0.90),(881, 0.75)],
    "مسؤولية الموكل عن تصرفات الوكيل": [
        (879, 1.0),(880, 0.95),(893, 0.90),(881, 0.80),(85, 0.65)],
    "الكسب غير المشروع":    [(77, 1.0),(78, 0.90),(79, 0.85),(98, 0.80)],
    "شبه العقد":            [(77, 1.0),(78, 0.90),(79, 0.85),(98, 0.80)],
    "الإثراء بلا سبب":      [(77, 1.0),(78, 0.90),(79, 0.80)],
    "الكسب غير المشروع وشبه العقد": [
        (77, 1.0),(78, 0.90),(79, 0.85),(98, 0.80)],

    # ── DOMMAGES ET INTÉRÊTS ──────────────────────────────────────────
    "التعويض":              [(264, 0.95),(77, 1.0),(106, 0.7),(98, 0.8)],
    "الضرر":                [(77, 1.0),(264, 0.9),(98, 0.85),(78, 0.75)],
    "التعويض عن الضرر":     [(77, 1.0),(264, 0.95),(106, 0.75),(98, 0.85)],
    "الضرر المادي":         [(264, 1.0),(98, 0.9),(77, 0.8)],
    "الضرر المعنوي":        [(77, 1.0),(98, 0.95),(264, 0.85)],

    # ── PAIEMENT / EXTINCTION ─────────────────────────────────────────
    "الوفاء":               [(320, 1.0),(321, 0.95),(322, 0.90),(323, 0.80)],
    "انقضاء الالتزامات":    [(393, 1.0),(320, 0.90),(321, 0.85),(322, 0.80)],
    "انقضاء الالتزامات بالوفاء": [(320, 1.0),(321, 0.95),(322, 0.90),(323, 0.80)],
    "الإبراء":              [(393, 1.0),(394, 0.90)],
    "المقاصة":              [(357, 1.0),(358, 0.90)],
    "التقادم":              [(380, 1.0),(381, 0.90),(382, 0.85)],
    "سقوط الحق":            [(380, 1.0),(381, 0.85)],

    # ── BAIL / LOCATION ───────────────────────────────────────────────
    "إيجار":                [(617, 1.0),(618, 0.95),(619, 0.90),(635, 0.85),(663, 0.80)],
    "كراء":                 [(617, 1.0),(618, 0.95),(619, 0.90),(635, 0.80)],
    "عقد الإيجار":          [(617, 1.0),(618, 0.95),(619, 0.90),(635, 0.85),(663, 0.80)],
    "مستأجر":               [(663, 1.0),(664, 0.95),(668, 0.90),(639, 0.80)],
    "مؤجر":                 [(635, 1.0),(636, 0.90),(638, 0.85),(617, 0.80)],
    "أجرة":                 [(618, 1.0),(633, 0.90),(634, 0.85),(664, 0.80)],
    "مدة الإيجار":          [(687, 1.0),(619, 0.90),(689, 0.85)],
    "إكراء من الباطن":      [(668, 1.0),(663, 0.80)],
    "انتهاء الإيجار":       [(687, 1.0),(689, 0.95)],
    "التزامات المستأجر":    [(663, 1.0),(664, 0.95),(668, 0.90),(639, 0.80)],
    "التزامات المؤجر":      [(635, 1.0),(636, 0.90),(638, 0.85),(617, 0.80)],
    "عقد الكراء":           [(617, 1.0),(618, 0.95),(619, 0.90),(635, 0.85)],
    "فسخ عقد الإيجار":      [(687, 1.0),(689, 0.90),(619, 0.80)],
    "عقد الإيجار والتزامات الأطراف": [
        (617, 1.0),(618, 0.95),(619, 0.90),(635, 0.90),
        (663, 0.85),(664, 0.85),(668, 0.80)],

    # ── FAILLITE ──────────────────────────────────────────────────────
    "الإفلاس":              [(1247, 1.0),(1248, 0.90)],
    "التسوية القضائية":     [(1247, 1.0),(1248, 0.95)],
    "الإفلاس والتسوية":     [(1247, 1.0),(1248, 0.95)],
    "التصفية القضائية":     [(1247, 0.95),(1070, 0.70)],
    "الإعسار":              [(1247, 0.85),(1248, 0.80)],

    # ── MANDAT ────────────────────────────────────────────────────────
    "الوكالة":              [(879, 1.0),(880, 0.95),(881, 0.85),(893, 0.90)],
    "الوكيل":               [(879, 0.9),(880, 0.85),(884, 0.80),(885, 0.75),(893, 0.85)],
    "التفويض":              [(879, 0.85),(880, 0.80),(1070, 0.60)],
    "الوكالة والتفويض":     [(879, 1.0),(880, 0.95),(893, 0.95),(881, 0.80)],

    # ── PRÊT / INTÉRÊTS ───────────────────────────────────────────────
    "القرض":                [(856, 1.0),(857, 0.90)],
    "الفائدة":              [(870, 1.0),(871, 0.90)],

    # ── GAGE / HYPOTHÈQUE ─────────────────────────────────────────────
    "الرهن":                [(1170, 1.0),(1171, 0.95),(1172, 0.90)],
    "الرهن الحيازي":        [(1170, 1.0),(1171, 0.95),(1172, 0.85)],

    # ── CAUTIONNEMENT ─────────────────────────────────────────────────
    "الكفالة":              [(1117, 1.0),(1118, 0.95),(1119, 0.90),(1145, 0.80)],
    "الكفيل":               [(1117, 0.90),(1118, 0.85),(1119, 0.80),(1145, 0.75)],
    "التضامن":              [(1117, 0.85),(1119, 0.80),(1118, 0.75)],
    "حق الرجوع":            [(1145, 1.0),(1119, 0.85),(1117, 0.75)],
    "عقد الكفالة":          [(1117, 1.0),(1118, 0.95),(1119, 0.90),(1145, 0.80)],
    "شروط الكفالة":         [(1117, 1.0),(1118, 0.95),(1119, 0.85)],
    "الكفالة التضامنية":    [(1117, 0.90),(1119, 1.0),(1118, 0.85)],
    "عقد الكفالة والضمان":  [(1117, 1.0),(1118, 0.95),(1119, 0.90),(1145, 0.80)],
    "ضمان شخصي":            [(1117, 1.0),(1118, 0.90),(1119, 0.85)],
}

# ══════════════════════════════════════════════════════════════════════
# HIÉRARCHIE NORMATIVE  v15.0
# ══════════════════════════════════════════════════════════════════════
HIERARCHY_SCORES: dict[int, float] = {
    # Principes fondamentaux
    2: 1.0, 19: 1.0, 77: 1.0, 228: 0.95, 230: 1.0,
    # Inexécution / résolution
    255: 0.95, 256: 0.92, 259: 0.95, 264: 0.95,
    260: 0.92, 261: 0.90, 262: 0.85, 263: 0.80,
    # Nullité
    306: 0.95, 307: 0.92, 308: 0.90,
    60: 0.90, 65: 0.88, 66: 0.85, 67: 0.82,
    109: 0.75, 110: 0.70,
    # Vente
    478: 0.90, 488: 0.92, 502: 0.88,
    489: 0.82, 503: 0.80, 504: 0.75,
    532: 0.82, 533: 0.78, 534: 0.75,
    # Capacité
    4: 0.90, 5: 0.88, 9: 0.85, 12: 0.82, 3: 0.72,
    # Formation
    18: 0.88, 20: 0.85, 21: 0.80, 39: 0.82,
    57: 0.82, 62: 0.80, 63: 0.75,
    # Effets
    229: 0.90, 235: 0.80,
    # Extinction
    320: 0.90, 321: 0.88, 322: 0.85, 323: 0.80,
    380: 0.80, 393: 0.88, 394: 0.78,
    357: 0.75, 358: 0.72,
    # Responsabilité
    78: 0.95, 79: 0.92, 85: 0.82, 86: 0.80, 88: 0.78,
    98: 0.88, 106: 0.75,
    # Faillite
    1247: 0.95, 1248: 0.90,
    # Mandat
    879: 0.90, 880: 0.88, 881: 0.82,
    884: 0.72, 893: 0.88,
    # Bail
    617: 0.95, 618: 0.95, 619: 0.92,
    633: 0.82, 635: 0.92, 636: 0.88, 638: 0.82, 639: 0.80,
    663: 0.90, 664: 0.88, 666: 0.75, 668: 0.88,
    687: 0.90, 688: 0.75, 689: 0.85,
    # Cautionnement
    1117: 0.95, 1118: 0.92, 1119: 0.90, 1145: 0.85,
    # Gage
    1170: 0.88, 1171: 0.85, 1172: 0.82,
    # Prêt
    856: 0.82, 870: 0.78,
}

# ══════════════════════════════════════════════════════════════════════
# COHÉRENCE TYPE-DE-CONTRAT ↔ PLAGE D'ARTICLES
# Empêche un article hors du chapitre pertinent de dominer le classement
# par simple chevauchement lexical (OCR bruité, nombres identiques, etc.)
# ══════════════════════════════════════════════════════════════════════
CONTRACT_TYPE_ARTICLE_RANGE: dict[str, range] = {
    "بيع":    range(478, 619),   # vente — bloc général
    "إيجار":  range(617, 723),
    "كراء":   range(617, 723),
    "كفالة":  range(1117, 1170),
    "وكالة":  range(879, 942),
    "رهن":    range(1170, 1247),
}


CONTRACT_TYPE_EXCLUDED_SUFFIX_BLOCKS: dict[str, set[int]] = {
    "بيع": {618},   # exclut tous les "618-N" du domaine vente mobilière
}

OUT_OF_DOMAIN_PENALTY      = 0.55
EXCLUDED_SUFFIX_PENALTY    = 0.30  # pénalité plus forte : hors-sujet quasi certain


def _article_base_and_suffix(raw) -> tuple[int, int | None]:
    """
    Décompose 'الفصل 618-11' / '618-11' en (618, 11).
    Pour un article simple '478', retourne (478, None).
    """
    if raw is None:
        return 0, None
    m = re.match(r"(\d+)(?:-(\d+))?", str(raw).strip())
    if not m:
        return 0, None
    base = int(m.group(1))
    suffix = int(m.group(2)) if m.group(2) else None
    return base, suffix


def _contract_domain_penalty(query: str, article_raw) -> float:

    base, suffix = _article_base_and_suffix(article_raw)

    for kw, art_range in CONTRACT_TYPE_ARTICLE_RANGE.items():
        if kw not in query:
            continue

        excluded_bases = CONTRACT_TYPE_EXCLUDED_SUFFIX_BLOCKS.get(kw, set())
        if suffix is not None and base in excluded_bases:
            return EXCLUDED_SUFFIX_PENALTY

        if base not in art_range:
            return OUT_OF_DOMAIN_PENALTY

        return 1.0

    return 1.0
# ══════════════════════════════════════════════════════════════════════
# UTILITAIRES
# ══════════════════════════════════════════════════════════════════════
def _parse_article_id(raw) -> int:
    if raw is None:
        return 0
    m = re.match(r"(\d+)", str(raw).strip())
    return int(m.group(1)) if m else 0


def _doctrinal_boost(query: str, article: int,
                     temperature: float = DOCTRINAL_TEMPERATURE) -> float:
    q = query.strip()
    best = 0.0
    for concept, entries in DOCTRINAL_INDEX.items():
        if concept in q or any(w in q for w in concept.split()):
            for art, score in entries:
                if art == article:
                    best = max(best, score)
    if best > 0 and temperature > 0:
        best = best ** (1.0 / temperature)
    return best


def _hierarchy_boost(article: int) -> float:
    return HIERARCHY_SCORES.get(article, 0.0)


def _normalize(values: list[float]) -> list[float]:
    mn, mx = min(values), max(values)
    if mx == mn:
        return [1.0] * len(values)
    return [(v - mn) / (mx - mn) for v in values]


# ══════════════════════════════════════════════════════════════════════
# SEUILS ADAPTATIFS
# ══════════════════════════════════════════════════════════════════════
def compute_adaptive_thresholds(query: str) -> dict:
    q = query.strip()

    is_tort        = any(k in q for k in ["مسؤولية","ضرر","خطأ","ضار","الفعل"])
    is_mandate     = any(k in q for k in ["وكالة","وكيل","موكل","تفويض"])
    is_surety      = any(k in q for k in ["كفالة","كفيل","ضمان","التضامن"])
    is_lease       = any(k in q for k in ["إيجار","كراء","مكتري","مكري"])
    is_nullity     = any(k in q for k in ["بطلان","إبطال","فساد"])
    is_payment     = any(k in q for k in ["وفاء","انقضاء","إبراء","مقاصة"])
    is_rescission  = any(k in q for k in ["فسخ","إعذار","عدم التنفيذ"])
    is_capacity    = any(k in q for k in ["أهلية","قاصر","ناقص","حجر","وصي"])
    is_condition   = any(k in q for k in ["شروط","أركان","صحة العقد","التراضي"])
    is_quasi       = any(k in q for k in ["الكسب","غير المشروع","شبه العقد","الإثراء"])
    is_binding     = any(k in q for k in ["القوة الملزمة","أثر العقد","بين الطرفين"])

    thresholds = {
        "RERANK_HARD_FLOOR": 0.01,
        "BGE_TOP_MIN":       0.03,
        "DOCTRINAL_MIN":     0.03,
        "BGE_SOFT_CENTER":   0.07,
        "BGE_SOFT_SLOPE":    4.0,
        "BGE_CTX_THRESHOLD": 0.12,
        "RRF_MIN_SCORE":     0.004,
    }

    if is_tort or is_mandate or is_surety:
        thresholds["DOCTRINAL_MIN"] = 0.02
        thresholds["BGE_TOP_MIN"]   = 0.03

    if is_lease:
        thresholds["DOCTRINAL_MIN"] = 0.02
        thresholds["BGE_TOP_MIN"]   = 0.03

    if is_nullity or is_condition:
        thresholds["DOCTRINAL_MIN"] = 0.03
        thresholds["BGE_TOP_MIN"]   = 0.04

    if is_rescission:
        thresholds["DOCTRINAL_MIN"] = 0.02
        thresholds["BGE_TOP_MIN"]   = 0.03

    if is_capacity:
        thresholds["DOCTRINAL_MIN"] = 0.04
        thresholds["BGE_TOP_MIN"]   = 0.05

    if is_quasi or is_binding:
        thresholds["DOCTRINAL_MIN"] = 0.02
        thresholds["BGE_TOP_MIN"]   = 0.03

    return thresholds


# ══════════════════════════════════════════════════════════════════════
# CHARGEMENT DU RERANKER
# ══════════════════════════════════════════════════════════════════════
def load_reranker(device: str = None) -> dict:
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"  Chargement reranker : {RERANKER_MODEL}  (device={device}) …")
    tokenizer = AutoTokenizer.from_pretrained(RERANKER_MODEL)
    model = AutoModelForSequenceClassification.from_pretrained(
        RERANKER_MODEL,
        torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    ).to(device).eval()
    print(f"   Reranker chargé  (dtype={next(model.parameters()).dtype})")
    return {"model": model, "tokenizer": tokenizer, "device": device}


# ══════════════════════════════════════════════════════════════════════
# RERANK AVEC MMR, SMOOTHING ET CALIBRATION
# ══════════════════════════════════════════════════════════════════════
def rerank(
    query: str,
    candidates: list,
    reranker: dict,
    top_k: int              = 15,
    batch_size: int         = 16,
    max_length: int         = 512,
    text_key: str           = "text",
    weights: dict           = None,
    temperature: float      = RERANK_TEMPERATURE,
    mmr_lambda: float       = MMR_LAMBDA,
    smooth_topk: int        = SMOOTH_TOPK,
    smooth_alpha: float     = SMOOTH_ALPHA,
    doctrinal_temperature: float = DOCTRINAL_TEMPERATURE,
) -> list:
    if not candidates:
        return []
    w = weights or SCORE_WEIGHTS
    valid = [c for c in candidates if c.get(text_key, "").strip()]
    if not valid:
        print("    Aucun candidat avec texte valide.")
        return []

    hybrid_raw = [c.get("rrf_score", 0.0) for c in valid]
    hybrid_n   = _normalize(hybrid_raw)

    pairs = [[query, c[text_key]] for c in valid]
    bge_raw = []
    for i in range(0, len(pairs), batch_size):
        encoded = reranker["tokenizer"](
            [p[0] for p in pairs[i:i+batch_size]],
            [p[1] for p in pairs[i:i+batch_size]],
            padding=True, truncation=True,
            max_length=max_length,
            return_tensors="pt",
        ).to(reranker["device"])
        with torch.no_grad():
            logits = reranker["model"](**encoded).logits
            if logits.shape[-1] == 1:
                logits = logits.squeeze(-1)
            elif logits.ndim == 2:
                logits = logits[:, 0]
            logits = logits / temperature
            scores = torch.sigmoid(logits)
        bge_raw.extend(scores.float().cpu().tolist())

    if smooth_topk > 0 and len(bge_raw) > smooth_topk:
        sorted_scores = sorted(bge_raw, reverse=True)
        top_mean = np.mean(sorted_scores[:smooth_topk])
        bge_smoothed = [smooth_alpha * top_mean + (1 - smooth_alpha) * s for s in bge_raw]
    else:
        bge_smoothed = bge_raw
    bge_n = _normalize(bge_smoothed)

    doc_raw = [
        _doctrinal_boost(query, _parse_article_id(c.get("article")), doctrinal_temperature)
        for c in valid
    ]
    doc_n = _normalize(doc_raw) if max(doc_raw) > 0 else doc_raw

    hier_raw = [_hierarchy_boost(_parse_article_id(c.get("article"))) for c in valid]
    hier_n = _normalize(hier_raw) if max(hier_raw) > 0 else hier_raw

    results = []
    for i, c in enumerate(valid):
        final = (
            w["hybrid"]    * hybrid_n[i]
          + w["rerank"]    * bge_n[i]
          + w["doctrinal"] * doc_n[i]
          + w["hierarchy"] * hier_n[i]
        )
        art_num = _parse_article_id(c.get("article"))
        domain_factor = _contract_domain_penalty(query, c.get("article"))
        if domain_factor < 1.0:
            final *= domain_factor
            print(f"   ⚠️ Pénalité domaine : فصل {c.get('article','?')} "
                  f"facteur ×{domain_factor:.2f}")

        if doc_n[i] > 0.55 and 0.12 <= bge_n[i] < 0.40:
            boost = 1.0 + min(doc_n[i] - 0.55, 0.30) * 0.6
            final = final * boost
            print(f"   Boost : فصل {c.get('article','?')} "
                  f"(doc={doc_n[i]:.3f}, bge={bge_n[i]:.3f}) ×{boost:.2f}")
        results.append({
            **c,
            "rerank_score"    : round(bge_smoothed[i], 6) if smooth_topk > 0 else round(bge_raw[i], 6),
            "doctrinal_score" : round(doc_raw[i], 4),
            "hierarchy_score" : round(hier_raw[i], 4),
            "final_score"     : round(final, 6),
            "_scores": {
                "hybrid_n"    : round(hybrid_n[i], 4),
                "bge_n"       : round(bge_n[i], 4),
                "doctrinal_n" : round(doc_n[i], 4),
                "hierarchy_n" : round(hier_n[i], 4),
            },
        })

    # ── Tri interne par final_score (sélection MMR) ────────────────────
    results.sort(key=lambda x: x["final_score"], reverse=True)

    if mmr_lambda > 0.0 and top_k < len(results):
        if all("embedding" in r for r in results):
            embeddings = [np.array(r["embedding"]) for r in results]
            scores = np.array([r["final_score"] for r in results])
            selected_indices = _mmr_select(embeddings, scores, top_k, mmr_lambda)
            results = [results[i] for i in selected_indices]
        else:
            results = results[:top_k]
    else:
        results = results[:top_k]

    # ══════════════════════════════════════════════════════════════════
    # ✅ TRI FINAL PAR rerank_score — du plus pertinent au moins pertinent
    # ══════════════════════════════════════════════════════════════════
    results = sorted(results, key=lambda x: x["final_score"], reverse=True)

    return results

def _mmr_select(embeddings, scores, k, lam):
    n = len(embeddings)
    if k >= n:
        return list(range(n))
    selected = []
    candidates = list(range(n))

    norms = [np.linalg.norm(e) for e in embeddings]
    if any(abs(nn - 1.0) > 1e-4 for nn in norms):
        embeddings = [e / (np.linalg.norm(e) + 1e-9) for e in embeddings]

    best_idx = int(np.argmax(scores))
    selected.append(best_idx)
    candidates.remove(best_idx)

    while len(selected) < k:
        mmr_scores = []
        for idx in candidates:
            sims = [np.dot(embeddings[idx], embeddings[s]) for s in selected]
            max_sim = max(sims)
            mmr = lam * scores[idx] - (1 - lam) * max_sim
            mmr_scores.append(mmr)
        best_candidate = candidates[int(np.argmax(mmr_scores))]
        selected.append(best_candidate)
        candidates.remove(best_candidate)
    return selected


# ══════════════════════════════════════════════════════════════════════
# PIPELINE COMPLET
# ══════════════════════════════════════════════════════════════════════
def search_hybrid_reranked(
    query: str,
    db: dict,
    reranker: dict,
    retrieval_k: int = 40,
    final_k: int     = 15,
    weights: dict    = None,
    **rerank_kwargs
) -> list:
    candidates = search_hybrid(query, db, top_k=retrieval_k)
    return rerank(query, candidates, reranker, top_k=final_k, weights=weights, **rerank_kwargs)


# ── Chargement du reranker ────────────────────────────────────────────
reranker = load_reranker()

# ══════════════════════════════════════════════════════════════════════
# EXPORTS
# ══════════════════════════════════════════════════════════════════════
if "chunks_parsed" not in globals():
    chunks_parsed = db.get("all_chunks", [])
    if not chunks_parsed:
        raise RuntimeError("❌ chunks_parsed introuvable…")
    print(f"  ℹ️  chunks_parsed reconstruit depuis db : {len(chunks_parsed)} chunks")
else:
    print(f"  ✅ chunks_parsed : {len(chunks_parsed)} chunks")

if "_encode_query" not in globals():
    _embed_model = db.get("model")
    if _embed_model is None:
        raise RuntimeError("❌ db['model'] introuvable.")
    _use_prefix = db.get("use_prefix", False)
    def _encode_query(text: str) -> np.ndarray:
        prefixed = f"query: {text}" if _use_prefix else text
        return _embed_model.encode(
            [prefixed], normalize_embeddings=True, convert_to_numpy=True
        ).astype(np.float32)
    print("  ✅ _encode_query défini")
else:
    print("  ✅ _encode_query déjà disponible")

if "e5_query" not in globals():
    e5_query = db.get("model", None)
    print("  ✅ e5_query défini")
else:
    print("  ✅ e5_query déjà disponible")


# ══════════════════════════════════════════════════════════════════════
# FONCTION D'ÉVALUATION
# ══════════════════════════════════════════════════════════════════════
def evaluate_rerank(
    queries: list,
    ground_truth: list[list[int]],
    db: dict,
    reranker: dict,
    retrieval_k: int  = 40,
    final_k: int      = 15,
    mmr_lambda: float = MMR_LAMBDA,
    **kwargs
) -> dict:
    recalls = {k: [] for k in [1, 3, 5]}
    mrr_list = []
    ndcg_list = []

    for q, gt in zip(queries, ground_truth):
        if not gt:
            continue
        results = search_hybrid_reranked(
            query=q, db=db, reranker=reranker,
            retrieval_k=retrieval_k, final_k=final_k,
            mmr_lambda=mmr_lambda, **kwargs
        )
        retrieved_articles = [_parse_article_id(r.get("article")) for r in results]

        for k in recalls:
            hits = len(set(retrieved_articles[:k]) & set(gt))
            recalls[k].append(hits / len(gt))

        for rank, art in enumerate(retrieved_articles, 1):
            if art in gt:
                mrr_list.append(1.0 / rank)
                break
        else:
            mrr_list.append(0.0)

        dcg = 0.0
        idcg = sum(1.0 / np.log2(i+2) for i in range(min(len(gt), 5)))
        for i, art in enumerate(retrieved_articles[:5]):
            if art in gt:
                dcg += 1.0 / np.log2(i+2)
        ndcg_list.append(dcg / idcg if idcg > 0 else 0.0)

    metrics = {}
    for k, vals in recalls.items():
        metrics[f"Recall@{k}"] = np.mean(vals) if vals else 0.0
    metrics["MRR"]    = np.mean(mrr_list)  if mrr_list  else 0.0
    metrics["NDCG@5"] = np.mean(ndcg_list) if ndcg_list else 0.0
    return metrics


# ══════════════════════════════════════════════════════════════════════
# TESTS DE DÉMONSTRATION
# ══════════════════════════════════════════════════════════════════════
print("\n" + "━"*60)
print("TESTS — CELLULE 6 v15.0")
print("━"*60)

test_queries_c6 = [
    "الأهلية المدنية للقاصر",
    "عقد البيع والشراء",
    "التعويض عن الضرر",
    "الإفلاس والتسوية القضائية",
    "المسؤولية التقصيرية",
    "الفسخ القضائي والإعذار",
    "عقد الكفالة والضمان",
    "القوة الملزمة للعقد وأثره بين الطرفين",
    "الكسب غير المشروع وشبه العقد",
    "مسؤولية الموكل عن تصرفات الوكيل",
    "الإعذار وشروط الفسخ القضائي",
]

for q in test_queries_c6:
    print(f"\n🔍 {q}")
    thr = compute_adaptive_thresholds(q)
    print(f"   BGE_TOP_MIN={thr['BGE_TOP_MIN']} | HARD_FLOOR={thr['RERANK_HARD_FLOOR']} "
          f"| DOCTRINAL_MIN={thr['DOCTRINAL_MIN']}")
    try:
        results = search_hybrid_reranked(
            query=q, db=db, reranker=reranker,
            retrieval_k=40, final_k=7
        )
        if not results:
            print("  ⚠️  Aucun résultat.")
            continue
        for i, r in enumerate(results, 1):
            s = r.get("_scores", {})
            print(f"  {i}. فصل {str(r.get('article','?')):>6}  "
                  f"rerank={r['rerank_score']:.6f}  "       # ← affiché en premier
                  f"final={r['final_score']:.4f}  "
                  f"[hyb={s.get('hybrid_n',0):.3f} "
                  f"bge={s.get('bge_n',0):.3f} "
                  f"doc={s.get('doctrinal_n',0):.3f} "
                  f"hier={s.get('hierarchy_n',0):.3f}]")
    except Exception as e:
        print(f"  ❌ Erreur : {e}")

print("\n✅ Cellule 6 v15.0 terminée.")
print("\n📦 Exports vers Cellule 7 :")
print("   reranker, rerank, search_hybrid_reranked, RERANK_TEMPERATURE")
print("   chunks_parsed, _encode_query, e5_query")
print("   DOCTRINAL_INDEX, HIERARCHY_SCORES, SCORE_WEIGHTS")
print("   _parse_article_id, _doctrinal_boost, _hierarchy_boost")
print("   compute_adaptive_thresholds")
print("   MMR_LAMBDA, SMOOTH_TOPK, SMOOTH_ALPHA, DOCTRINAL_TEMPERATURE")
print("   evaluate_rerank")

✅ Dépendances amont vérifiées (db, search_hybrid).
  Chargement reranker : BAAI/bge-reranker-v2-m3  (device=cuda) …


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

   Reranker chargé  (dtype=torch.float16)
  ✅ chunks_parsed : 1169 chunks
  ✅ _encode_query déjà disponible
  ✅ e5_query déjà disponible

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
TESTS — CELLULE 6 v15.0
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🔍 الأهلية المدنية للقاصر
   BGE_TOP_MIN=0.05 | HARD_FLOOR=0.01 | DOCTRINAL_MIN=0.04
   Boost : فصل 3 (doc=0.883, bge=0.283) ×1.18
  1. فصل      5  rerank=0.695364  final=0.9680  [hyb=1.000 bge=1.000 doc=0.943 hier=0.926]
  2. فصل      4  rerank=0.685403  final=0.9450  [hyb=0.866 bge=0.984 doc=1.000 hier=0.947]
  3. فصل      3  rerank=0.247849  final=0.9137  [hyb=0.932 bge=0.283 doc=0.883 hier=0.758]
  4. فصل     12  rerank=0.559232  final=0.8466  [hyb=0.803 bge=0.782 doc=0.914 hier=0.863]
  5. فصل    984  rerank=0.326810  final=0.3067  [hyb=0.741 bge=0.409 doc=0.000 hier=0.000]
  6. فصل   1247  rerank=0.419571  final=0.2749  [hyb=0.000 bge=0.558 doc=0.000 hier=1.000]
  7. فصل     85  rerank=0.071337  final

## Cellule 7 — Hybrid Retriever (BM25 + FAISS + ChromaDB + RRF)


In [7]:
# ══════════════════════════════════════════════════════════════
# CELLULE 7 — HybridRetriever v5.1  [CORRIGÉ]
# ══════════════════════════════════════════════════════════════

from rank_bm25 import BM25Okapi
from dataclasses import dataclass as _dc
import numpy as np
import re
import math
import unicodedata

# ──────────────────────────────────────────────────────────────
# VÉRIFICATIONS AMONT
# ──────────────────────────────────────────────────────────────
for varname in ["db", "reranker", "rerank", "chunks_parsed",
                "_encode_query", "e5_query"]:
    assert varname in globals(), \
        f"❌ {varname} non défini — relancez les cellules précédentes"

if "compute_adaptive_thresholds" not in globals():
    raise RuntimeError(
        "❌ compute_adaptive_thresholds non défini — exécutez la cellule 6 d'abord."
    )

chunk_ids         = db["chunk_ids"]
all_chunks        = db["all_chunks"]
faiss_index       = db["faiss_index"]
chroma_collection = db["chroma_collection"]
embed_model       = db["model"]
use_prefix        = db["use_prefix"]

RRF_K = 60

# ══════════════════════════════════════════════════════════════
# CONSTANTES DE SCORING
# ══════════════════════════════════════════════════════════════
RERANK_HARD_FLOOR  = 0.008
BGE_TOP_MIN        = 0.03
RRF_MIN_SCORE      = 0.003
MAX_SELECTED       = 7
GAP_SIGMOID_CENTER = 0.45
GAP_SIGMOID_SLOPE  = 5.0
BGE_CTX_THRESHOLD  = 0.10

BGE_SOFT_CENTER    = 0.07
BGE_SOFT_SLOPE     = 4.0
DOCTRINAL_MIN      = 0.03

BGE_PAIR_GAP       = 0.25
BGE_WEAK_REJECT    = 0.40
CENTRALITY_SCALE   = 0.25
CENTRALITY_CAP     = 0.02
MAX_PRIORITY_DELTA = 0.15
TOP_K_FIXED        = 5

RETRIEVAL_N_FAISS  = 200
RETRIEVAL_N_CHROMA = 100
RETRIEVAL_N_BM25   = 200

FALLBACK_FACTORS = [1.0, 0.70, 0.45, 0.25]

CONTRACT_DOMAIN_BOOST = {
    "البيع":   0.20,
    "الكفالة": 0.22,
    "الوكالة": 0.20,
    "الإيجار": 0.20,
    "الكراء":  0.20,
    "الرهن":   0.15,
    "الإفلاس": 0.15,
    "الفسخ":   0.15,
    "البطلان": 0.15,
}

SAME_SECTION_BONUS    = 0.06
SAME_CHAPTER_BONUS    = 0.04
NEIGHBOR_BONUS_D1     = 0.06
NEIGHBOR_BONUS_D2     = 0.03
CROSS_CHAPTER_PENALTY = 0.02
MAX_CONTEXT_BONUS     = 0.14
PROPAGATION_FACTOR    = 0.07
SEMANTIC_GRAPH_BONUS  = 0.07
NEGATIVE_PENALTY      = 0.12
MAX_NEGATIVE_PENALTY  = 0.25

ALWAYS_PAIR_SECTIONS = {
    "الأهلية والتراضي",
    "تعريف البيع وشروطه",
    "شروط الكفالة والتزامات الكفيل",
    "المسؤولية التقصيرية والضرر",
    "الوفاء",
    "الفسخ",
    "أسباب البطلان",
    "التزامات المؤجر والمستأجر",
}

_STRUCTURAL_FP_ARTICLES = {754, 1250, 749, 121, 763}
_STRUCTURAL_FP_PENALTY  = 0.60   # pénalité de base
_STRUCTURAL_FP_RELIEF   = 0.40   # levée partielle si keyword présent


# ══════════════════════════════════════════════════════════════
# UTILITAIRES TEXTE
# ══════════════════════════════════════════════════════════════
def fix_ocr(text: str) -> str:
    if not text:
        return ""
    text = re.sub(r"[a-zA-Z]{5,}", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def tokenize_ar(text: str) -> list:
    text = fix_ocr(text)
    return [w for w in re.findall(r"[\u0600-\u06ff]+", text) if len(w) >= 2]

def enrich_text(chunk: dict) -> str:
    header = chunk.get("header", f"الفصل {chunk.get('article', '')}")
    text   = fix_ocr(chunk.get("text", ""))
    return f"{header} {text}".strip()

def deduplicate_chunks(chunks: list, key: str = "chunk_id") -> list:
    seen, result = set(), []
    for c in chunks:
        cid = c.get(key)
        if cid not in seen:
            seen.add(cid)
            result.append(c)
    return result

# ══════════════════════════════════════════════════════════════
# NORMALISATION ARABE JURIDIQUE
# ══════════════════════════════════════════════════════════════
_ARABIC_SYNONYMS = {
    "الكراء":      "الإيجار",
    "المكتري":     "المستأجر",
    "المكري":      "المؤجر",
    "ضمان":        "كفالة",
    "إعذار":       "إنذار",
    "فسخ":         "حل العقد",
    "بطلان":       "إبطال",
    "الضرر":       "الأضرار",
    "تعويض":       "التعويضات",
    "التزام":      "الالتزامات",
    "قاصر":        "ناقص الأهلية",
    "وكيل":        "موكل",
}

def normalize_arabic_legal(text: str) -> str:
    words = tokenize_ar(text)
    new_words = []
    for w in words:
        new_words.append(_ARABIC_SYNONYMS.get(w, w))
    return " ".join(new_words)

# ══════════════════════════════════════════════════════════════
# GRAPHES LÉGAUX  v5.0
# ══════════════════════════════════════════════════════════════
LEGAL_GRAPH: dict[int, list[int]] = {
    # Résolution / inexécution
    255: [256, 259, 260],
    256: [255, 259, 260],
    259: [260, 261, 262, 230, 255, 256],
    260: [259, 261, 262, 255],
    261: [259, 260, 262, 263, 255],
    262: [259, 260, 261],
    # Responsabilité délictuelle
    77:  [78, 79, 264, 98],
    78:  [77, 79, 98],
    79:  [77, 78, 98],
    98:  [77, 78, 79, 264],
    # Cautionnement
    1117:[1118, 1119, 1121, 1145],
    1118:[1117, 1119, 1145],
    1119:[1117, 1118, 1145],
    1145:[1117, 1118, 1119],
    # Mandat
    879: [880, 881, 893],
    880: [879, 881, 893],
    881: [879, 880, 893],
    893: [879, 880, 881],
    # Vente
    478: [488, 502],
    488: [478, 502],
    502: [478, 488],
    # Extinction
    320: [321, 322, 323],
    321: [320, 322, 323],
    322: [320, 321, 323],
    323: [320, 321, 322],
    # Capacité
    4:   [5, 9, 12],
    5:   [4, 9, 12],
    9:   [4, 5, 12],
    # Dommages-intérêts
    264: [77, 265, 98],
    229: [228, 230, 235],
    228: [229, 230],
    230: [228, 229, 259, 18],
    # Faillite
    1247:[1248, 1249],
    1248:[1247, 1249],
    # Bail
    617: [618, 619, 635, 663, 664, 668],
    618: [617, 619, 635],
    619: [617, 618],
    635: [617, 618, 636, 638],
    636: [635, 638],
    663: [617, 664, 668, 635],
    664: [617, 663, 668, 635],
    668: [617, 663, 664],
    # Nullité
    60:  [61, 65, 66, 67, 109, 306, 307],
    65:  [60, 66, 67, 306],
    66:  [60, 65, 67],
    67:  [60, 65, 66],
    109: [60, 110],
    306: [60, 65, 307, 308],
    307: [60, 65, 306, 308],
    308: [60, 306, 307],
    # Gage
    1170:[1171, 1172],
    1171:[1170, 1172],
    1172:[1170, 1171],
}

LEGAL_SEMANTIC_GRAPH: dict[int, list[int]] = {
    255: [256, 259, 260],
    256: [255, 259, 260],
    259: [260, 261, 262, 230, 255, 256],
    260: [259, 261, 262, 255],
    261: [259, 260, 262, 263, 255],
    77:  [78, 79, 264, 98],
    78:  [77, 79, 98],
    79:  [77, 78, 98],
    98:  [77, 78, 79, 264],
    1117:[1118, 1119, 1121, 1145],
    1118:[1117, 1119, 1145],
    1119:[1117, 1118, 1145],
    879: [880, 881, 893],
    880: [879, 881, 893],
    881: [879, 880, 893],
    893: [879, 880, 881],
    478: [488, 502],
    488: [478, 502],
    320: [321, 322, 323],
    321: [320, 322, 323],
    322: [320, 321, 323],
    4:   [5, 9, 12],
    5:   [4, 9, 12],
    9:   [4, 5, 12],
    264: [77, 265, 98],
    229: [228, 230],
    228: [229, 230],
    230: [228, 229, 259],
    1247:[1248, 1249],
    1248:[1247, 1249],
    617: [618, 619, 635, 663, 664, 668],
    618: [617, 619, 635],
    635: [617, 636, 638],
    663: [617, 664, 668, 635],
    664: [617, 663, 668, 635],
    668: [617, 663, 664],
    60:  [61, 65, 66, 67, 109, 306, 307],
    65:  [60, 66, 67, 306],
    306: [60, 65, 307, 308],
    307: [60, 306, 308],
    308: [60, 306, 307],
    1170:[1171, 1172],
    1171:[1170, 1172],
    1172:[1170, 1171],
}

def get_graph_neighbors(article_int: int, depth: int = 2) -> set:
    visited  = set()
    frontier = {article_int}
    for _ in range(depth):
        next_frontier = set()
        for node in frontier:
            for neighbor in LEGAL_GRAPH.get(node, []):
                if neighbor not in visited and neighbor != article_int:
                    next_frontier.add(neighbor)
        visited.update(next_frontier)
        frontier = next_frontier
    return visited

def get_semantic_neighbors(article_int: int) -> set:
    return set(LEGAL_SEMANTIC_GRAPH.get(article_int, []))

# ══════════════════════════════════════════════════════════════
# TAXONOMIE
# ══════════════════════════════════════════════════════════════
_CHAPTER_MAP: list[tuple[range, str]] = [
    (range(1,   78),   "العقود والالتزامات التعاقدية — أحكام عامة"),
    (range(77,  100),  "المسؤولية التقصيرية والضرر"),
    (range(100, 230),  "آثار الالتزامات التعاقدية"),
    (range(230, 259),  "العقود والالتزامات التعاقدية — أحكام عامة"),
    (range(259, 320),  "العقود والالتزامات التعاقدية — البطلان والفسخ"),
    (range(264, 330),  "التعويض والضرر"),
    (range(320, 478),  "انقضاء الالتزامات"),
    (range(478, 540),  "عقد البيع — أحكام عامة"),
    (range(540, 617),  "عقد البيع — التزامات الأطراف"),
    (range(617, 700),  "عقد الكراء — أحكام عامة"),
    (range(856, 895),  "عقد الوكالة — أحكام عامة"),
    (range(895, 920),  "عقد الوكالة — انتهاء الوكالة"),
    (range(1117,1145), "عقد الكفالة — أحكام عامة"),
    (range(1145,1170), "عقد الكفالة — انقضاء الكفالة"),
    (range(1170,1247), "الرهن الحيازي والرهن الرسمي"),
    (range(1247,1300), "الإفلاس والتسوية القضائية"),
    (range(60,   77),  "العقود والالتزامات التعاقدية — البطلان والفسخ"),
    (range(255,  265), "العقود والالتزامات التعاقدية — البطلان والفسخ"),
]

_SECTION_MAP: list[tuple[range, str]] = [
    (range(1,   10),   "أحكام عامة"),
    (range(19,  25),   "الأهلية والتراضي"),
    (range(57,  65),   "تعريف البيع وشروطه"),
    (range(60,   77),  "أسباب البطلان"),
    (range(77,  100),  "المسؤولية التقصيرية والضرر"),
    (range(230, 265),  "الالتزامات التعاقدية"),
    (range(255, 265),  "الفسخ"),
    (range(259, 265),  "الفسخ"),
    (range(264, 270),  "التعويض والضرر"),
    (range(306, 315),  "أسباب البطلان"),
    (range(320, 395),  "الوفاء"),
    (range(478, 540),  "تعريف البيع وشروطه"),
    (range(617, 627),  "عقد الكراء — أحكام عامة"),
    (range(627, 700),  "التزامات المؤجر والمستأجر"),
    (range(1117,1145), "شروط الكفالة والتزامات الكفيل"),
    (range(1170,1200), "أحكام الرهن وآثاره"),
    (range(1247,1300), "الإفلاس والتسوية القضائية"),
]

def get_chapter(article_int: int) -> str:
    for r, chapter in _CHAPTER_MAP:
        if article_int in r:
            return chapter
    return ""

def get_section(article_int: int) -> str:
    for r, section in _SECTION_MAP:
        if article_int in r:
            return section
    return ""

def get_subdomain(article_int: int) -> str:
    return get_chapter(article_int)

def get_centrality(article_int: int) -> float:
    degree = len(LEGAL_GRAPH.get(article_int, []))
    return round(min(degree * CENTRALITY_SCALE / 10, CENTRALITY_CAP), 4)

# ══════════════════════════════════════════════════════════════
# PRIORITÉ D'ARTICLE PAR REQUÊTE
# ══════════════════════════════════════════════════════════════
_ARTICLE_PRIORITY_TABLE: list[tuple[str, int, float]] = [
    ("الأهلية",              4,    0.10),
    ("قاصر",                 4,    0.10),
    ("تراضي",               19,    0.08),
    ("الإيجاب والقبول",     19,    0.08),
    ("عدم التنفيذ",         259,   0.10),
    ("الفسخ",               259,   0.10),
    ("إعذار",               255,   0.12),
    ("الإعذار",             255,   0.12),
    ("الإعذار",             256,   0.10),
    ("إعذار",               256,   0.10),       # [CORRECTION] doublon explicite
    ("إعذار",               261,   0.08),       # [CORRECTION] 261 lié au gate فسخ
    ("الإعذار",             261,   0.08),       # [CORRECTION]
    ("شروط الفسخ",          255,   0.12),
    ("الفسخ القضائي",       255,   0.12),
    ("الفسخ القضائي",       256,   0.10),
    ("التعويض",             264,   0.08),
    ("الضرر",                77,   0.10),
    ("المسؤولية التقصيرية", 77,    0.12),
    ("البيع",               478,   0.08),
    ("عقد البيع",           478,   0.08),
    ("الوكالة",             879,   0.08),
    ("الكفالة",            1117,   0.12),
    ("الكفالة",            1118,   0.10),
    ("الإفلاس",            1247,   0.10),
    ("الإفلاس",            1248,   0.08),       # [CORRECTION] 1248 souvent manquant
    ("الرهن",              1170,   0.08),
    ("الإيجار",             617,   0.10),
    ("الإيجار",             663,   0.10),
    ("الإيجار",             664,   0.08),
    ("كراء",                617,   0.10),
    ("الالتزامات",          230,   0.06),
    ("القوة الملزمة",       230,   0.12),
    ("القوة الملزمة",       228,   0.10),
    ("القوة الملزمة",       229,   0.08),
    ("أثر العقد",           228,   0.10),
    ("أثر العقد",           229,   0.08),
    ("البطلان",              60,   0.10),
    ("أسباب البطلان",        60,   0.10),
    ("مسؤولية الموكل",      879,   0.12),
    ("مسؤولية الموكل",      880,   0.10),
    ("مسؤولية الموكل",      893,   0.10),
    ("تصرفات الوكيل",       879,   0.12),
    ("تصرفات الوكيل",       880,   0.10),
    # [CORRECTION] الكسب غير المشروع → articles 77-79-98 avec priorité haute
    ("الكسب غير المشروع",   77,    0.12),
    ("الكسب غير المشروع",   78,    0.10),
    ("الكسب غير المشروع",   79,    0.10),
    ("الكسب غير المشروع",   98,    0.08),
    ("شبه العقد",            77,   0.10),
    ("شبه العقد",            78,   0.08),
    ("شبه العقد",            79,   0.08),
    ("شبه العقد",            98,   0.06),
]

def get_article_priority(query: str, article: str) -> float:
    try:
        art_int = int(str(article).split("-")[0])
    except Exception:
        return 0.0
    best = 0.0
    for kw, art_num, delta in _ARTICLE_PRIORITY_TABLE:
        if kw in query and art_num == art_int:
            best = max(best, delta)
    return round(min(best, MAX_PRIORITY_DELTA), 4)

# ══════════════════════════════════════════════════════════════
# FILTRE NÉGATIF
# ══════════════════════════════════════════════════════════════
_NEGATIVE_PAIRS: list[tuple[str, set[int]]] = [
    ("الإفلاس",   {77, 78, 79, 478, 879, 1117}),
    ("التعويض",   {1247, 617, 879}),
    ("الكفالة",   {77, 259, 478, 617}),
    ("الوكالة",   {77, 259, 478, 617, 1117}),
    ("البيع",     {77, 879, 1117, 1247}),
    ("الإيجار",   {77, 879, 1117, 1247, 478}),
    ("البطلان",   {1247, 879, 1117}),
]

def apply_negative_filter(query: str, text: str) -> float:
    try:
        art_matches = re.findall(r"الفصل\s+(\d+)", text)
        art_ints    = {int(m) for m in art_matches}
    except Exception:
        art_ints = set()

    # [CORRECTION] Pénalité pour faux positifs structurels
    penalty = 0.0
    for art_int in art_ints:
        if art_int in _STRUCTURAL_FP_ARTICLES:
            penalty += _STRUCTURAL_FP_PENALTY
            query_terms = set(tokenize_ar(query))
            text_tokens = set(tokenize_ar(text))
            if query_terms & text_tokens:
                penalty -= _STRUCTURAL_FP_RELIEF
            print(f"  🚧 FP structurel détecté : فصل {art_int} "
                  f"(pénalité nette={penalty:.2f})")

    for kw, unrelated_arts in _NEGATIVE_PAIRS:
        if kw in query:
            overlap = art_ints & unrelated_arts
            if overlap:
                penalty += NEGATIVE_PENALTY * len(overlap)

    return round(min(penalty, MAX_NEGATIVE_PENALTY), 4)

# ══════════════════════════════════════════════════════════════
# CLASSIFICATION DE REQUÊTE
# ══════════════════════════════════════════════════════════════
_QUERY_BOOST_TABLE: dict[str, dict[int, float]] = {
    "الأهلية":               {4: 0.08, 5: 0.06, 9: 0.06, 12: 0.06},
    "قاصر":                  {4: 0.08, 5: 0.06},
    "الفسخ":                 {259: 0.10, 260: 0.08, 261: 0.06, 255: 0.08, 256: 0.06},
    "عدم التنفيذ":           {259: 0.10, 260: 0.06, 255: 0.06},
    # [CORRECTION] boost élevé pour إعذار → 255/256/261
    "إعذار":                 {255: 0.14, 256: 0.12, 259: 0.08, 261: 0.08},
    "الإعذار":               {255: 0.14, 256: 0.12, 259: 0.08, 261: 0.08},
    "التعويض":               {264: 0.08, 77: 0.06, 98: 0.06},
    "المسؤولية التقصيرية":   {77: 0.12, 78: 0.10, 79: 0.10, 98: 0.07},
    "البيع":                 {478: 0.08, 488: 0.06, 502: 0.04},
    "عقد البيع":             {478: 0.08, 488: 0.06, 502: 0.04},
    "الوكالة":               {879: 0.08, 880: 0.06, 893: 0.08},
    "مسؤولية الموكل":        {879: 0.12, 880: 0.10, 893: 0.10},
    "تصرفات الوكيل":         {879: 0.12, 880: 0.10, 893: 0.10},
    "الكفالة":              {1117: 0.12, 1118: 0.10, 1119: 0.08},
    "الإفلاس":             {1247: 0.10, 1248: 0.08},
    "الرهن":               {1170: 0.08, 1171: 0.06, 1172: 0.06},
    "الإيجار":              {617: 0.10, 618: 0.08, 663: 0.08, 664: 0.06, 668: 0.06, 635: 0.06},
    "كراء":                  {617: 0.10, 618: 0.08, 663: 0.06},
    "الالتزامات التعاقدية": {230: 0.08, 228: 0.06, 229: 0.06},
    "القوة الملزمة":         {230: 0.12, 228: 0.10, 229: 0.08},
    "أثر العقد":             {228: 0.10, 229: 0.08, 230: 0.10},
    "الوفاء":               {320: 0.08, 321: 0.06, 322: 0.06},
    "البطلان":              {60: 0.10, 65: 0.08, 66: 0.06, 67: 0.06, 306: 0.06, 307: 0.05},
    # [CORRECTION] الكسب غير المشروع / شبه العقد → boost fort sur 77-98
    "الكسب غير المشروع":     {77: 0.14, 78: 0.12, 79: 0.12, 98: 0.10},
    "شبه العقد":             {77: 0.12, 78: 0.10, 79: 0.10, 98: 0.08},
}

def classify_query(query: str) -> dict:
    boosts: dict[int, float] = {}
    for kw, art_map in _QUERY_BOOST_TABLE.items():
        if kw in query:
            for art_num, delta in art_map.items():
                boosts[art_num] = max(boosts.get(art_num, 0.0), delta)
    return boosts

# ══════════════════════════════════════════════════════════════
# CHAPTER_PRIOR & QUERY_CHAPTER_BOOST
# ══════════════════════════════════════════════════════════════
CHAPTER_PRIOR: dict[str, float] = {
    "العقود والالتزامات التعاقدية — أحكام عامة":          0.06,
    "العقود والالتزامات التعاقدية — البطلان والفسخ":       0.08,
    "آثار الالتزامات التعاقدية":                           0.06,
    "التعويض والضرر":                                      0.08,
    "انقضاء الالتزامات":                                   0.06,
    "عقد البيع — أحكام عامة":                             0.06,
    "عقد البيع — التزامات الأطراف":                       0.05,
    "عقد البيع — أنواع خاصة":                             0.04,
    "عقد الكراء — أحكام عامة":                            0.05,
    "عقد الوكالة — أحكام عامة":                           0.05,
    "عقد الوكالة — انتهاء الوكالة":                       0.04,
    "عقد الكفالة — أحكام عامة":                           0.06,
    "عقد الكفالة — انقضاء الكفالة":                       0.05,
    "الرهن الحيازي والرهن الرسمي":                        0.05,
    "الإفلاس والتسوية القضائية":                          0.06,
    "المسؤولية التقصيرية والضرر":                         0.07,
}

QUERY_CHAPTER_BOOST: dict[str, dict[str, float]] = {
    "الفسخ": {
        "العقود والالتزامات التعاقدية — البطلان والفسخ": 0.12,
    },
    "إعذار": {
        "العقود والالتزامات التعاقدية — البطلان والفسخ": 0.14,  # [CORRECTION] +0.02
    },
    "الإعذار": {
        "العقود والالتزامات التعاقدية — البطلان والفسخ": 0.14,  # [CORRECTION] +0.02
    },
    "الكفالة": {
        "عقد الكفالة — أحكام عامة": 0.12,
    },
    "المسؤولية التقصيرية": {
        "التعويض والضرر": 0.12,
        "المسؤولية التقصيرية والضرر": 0.12,
    },
    # [CORRECTION] الكسب غير المشروع → chapitre responsabilité délictuelle
    "الكسب غير المشروع": {
        "المسؤولية التقصيرية والضرر": 0.16,
        "التعويض والضرر": 0.12,
    },
    "شبه العقد": {
        "المسؤولية التقصيرية والضرر": 0.14,  # [CORRECTION] +0.02
        "التعويض والضرر": 0.10,
    },
    "البيع": {
        "عقد البيع — أحكام عامة": 0.08,
    },
    "الوفاء": {
        "انقضاء الالتزامات": 0.08,
    },
    "الأهلية": {
        "العقود والالتزامات التعاقدية — أحكام عامة": 0.08,
    },
    "الإفلاس": {
        "الإفلاس والتسوية القضائية": 0.12,
    },
    "الوكالة": {
        "عقد الوكالة — أحكام عامة": 0.08,
    },
    "مسؤولية الموكل": {
        "عقد الوكالة — أحكام عامة": 0.12,
    },
    "التعويض": {
        "التعويض والضرر": 0.08,
        "المسؤولية التقصيرية والضرر": 0.06,
    },
    "الرهن": {
        "الرهن الحيازي والرهن الرسمي": 0.08,
    },
    "الإيجار": {
        "عقد الكراء — أحكام عامة": 0.08,
        "التزامات المؤجر والمستأجر": 0.12,
    },
    "كراء": {
        "عقد الكراء — أحكام عامة": 0.08,
    },
    "القوة الملزمة": {
        "آثار الالتزامات التعاقدية": 0.12,
        "العقود والالتزامات التعاقدية — أحكام عامة": 0.08,
    },
    "أثر العقد": {
        "آثار الالتزامات التعاقدية": 0.10,
    },
    "انقضاء الالتزامات": {
        "انقضاء الالتزامات": 0.10,
    },
    "الالتزامات التعاقدية": {
        "آثار الالتزامات التعاقدية": 0.08,
        "العقود والالتزامات التعاقدية — أحكام عامة": 0.06,
    },
    "الضرر": {
        "التعويض والضرر": 0.08,
        "المسؤولية التقصيرية والضرر": 0.10,
    },
    "البطلان": {
        "العقود والالتزامات التعاقدية — البطلان والفسخ": 0.12,
    },
}

def get_chapter_boost(query: str, chapter: str) -> float:
    prior = CHAPTER_PRIOR.get(chapter, 0.0)
    query_boost = 0.0
    for topic, ch_map in QUERY_CHAPTER_BOOST.items():
        if topic in query:
            query_boost = max(query_boost, ch_map.get(chapter, 0.0))
    return round(prior + query_boost, 4)

# ══════════════════════════════════════════════════════════════
# LEGAL_PRIOR
# ══════════════════════════════════════════════════════════════
LEGAL_PRIOR: dict[int, float] = {
    230: 0.08, 77: 0.07, 259: 0.07, 478: 0.05,
    320: 0.05, 1117: 0.06, 1247: 0.05, 4: 0.05,
    255: 0.06, 256: 0.05,
    260: 0.06, 261: 0.05, 262: 0.04,
    78: 0.05, 79: 0.05,
    1118: 0.05, 1119: 0.04,
    264: 0.04, 265: 0.03, 879: 0.04, 880: 0.04,
    893: 0.04,
    321: 0.05, 322: 0.04,
    488: 0.04, 502: 0.03,
    617: 0.05, 618: 0.04, 619: 0.04,
    635: 0.04, 663: 0.04, 664: 0.04, 668: 0.04,
    98: 0.04,
    60: 0.04, 65: 0.03, 66: 0.03, 67: 0.03,
    228: 0.04, 229: 0.04,
    306: 0.04, 307: 0.03, 308: 0.03,
    1171: 0.03, 1172: 0.03, 1248: 0.04,
}

# ══════════════════════════════════════════════════════════════
# SCORING UTILS
# ══════════════════════════════════════════════════════════════
def _encode(text: str, is_query: bool = True) -> np.ndarray:
    if use_prefix:
        text = f"query: {text}" if is_query else f"passage: {text}"
    return embed_model.encode(
        [text], normalize_embeddings=True, convert_to_numpy=True
    ).astype(np.float32)

def maxsim_bonus(query: str, text: str,
                 bonus_per_term: float = 0.04,
                 max_bonus: float = 0.20) -> float:
    qt = set(tokenize_ar(query))
    if not qt:
        return 0.0
    text_tokens = tokenize_ar(text)
    if not text_tokens:
        return 0.0
    matched  = sum(1 for t in qt if t in text_tokens)
    coverage = matched / len(qt)               # % termes requête couverts
    density  = matched / len(text_tokens)      # densité — pénalise textes longs
    score    = coverage * 0.7 + density * 0.3
    return round(min(score * bonus_per_term * len(qt), max_bonus), 4)

_SOURCE_TYPE_WEIGHT: dict[str, float] = {
    "legal_text":    1.0,
    "annotation":    0.55,
    "jurisprudence": 0.40,
}

def compute_hybrid_score(rerank_score, bm25_score, vector_score,
                          article_num, query_boosts, bm25_max=1.0,
                          vector_max=1.0, w_rerank=0.50,
                          w_bm25=0.28, w_vector=0.22,
                          source_type: str = "legal_text") -> float:
    s = (w_rerank * rerank_score
         + w_bm25   * bm25_score   / (bm25_max   + 1e-9)
         + w_vector * vector_score / (vector_max  + 1e-9))
    structural_weight = _SOURCE_TYPE_WEIGHT.get(source_type, 0.5)
    s *= structural_weight
    s += LEGAL_PRIOR.get(article_num, 0)
    s += query_boosts.get(article_num, 0)
    s += get_centrality(article_num)
    return round(s, 6)

def compute_contextual_bonus(art_int: int, top1_art_int: int,
                              bge_score: float = 1.0) -> float:
    if art_int == top1_art_int:
        return 0.0
    if bge_score < BGE_CTX_THRESHOLD:
        return 0.0
    bonus = 0.0
    neighbors_d1 = set(LEGAL_GRAPH.get(top1_art_int, []))
    neighbors_d2 = get_graph_neighbors(top1_art_int, depth=2)
    if art_int in neighbors_d1:
        bonus += NEIGHBOR_BONUS_D1
    sec_art  = get_section(art_int)
    sec_top1 = get_section(top1_art_int)
    if sec_art and sec_top1 and sec_art == sec_top1:
        bonus += SAME_SECTION_BONUS
    elif get_chapter(art_int) == get_chapter(top1_art_int):
        bonus += SAME_CHAPTER_BONUS
    if art_int in neighbors_d2 and art_int not in neighbors_d1:
        bonus += NEIGHBOR_BONUS_D2
    if (get_chapter(art_int) != get_chapter(top1_art_int)
            and art_int not in neighbors_d2):
        bonus -= CROSS_CHAPTER_PENALTY
    return round(min(bonus, MAX_CONTEXT_BONUS), 4)

def compute_semantic_graph_bonus(art_int: int, top1_art_int: int,
                                  bge_score: float = 1.0) -> float:
    if art_int == top1_art_int:
        return 0.0
    if bge_score < BGE_CTX_THRESHOLD:
        return 0.0
    if art_int in get_semantic_neighbors(top1_art_int):
        return SEMANTIC_GRAPH_BONUS
    return 0.0

def extract_article_num_from_text(text: str) -> int:
    m = re.search(r'الفصل\s+(\d+)', text)
    if m:
        try:
            return int(m.group(1))
        except ValueError:
            pass
    return 0

def bge_soft_weight(bge: float,
                    center: float = BGE_SOFT_CENTER,
                    slope:  float = BGE_SOFT_SLOPE) -> float:
    return 1.0 / (1.0 + math.exp(-slope * (bge - center)))

def gap_weight(ratio: float) -> float:
    return 1.0 / (1.0 + math.exp(
        -GAP_SIGMOID_SLOPE * (ratio - GAP_SIGMOID_CENTER)))

def adaptive_top_k(top_bge: float) -> int:
    return TOP_K_FIXED


# ══════════════════════════════════════════════════════════════
# DATACLASS RetrievedChunk
# ══════════════════════════════════════════════════════════════
@_dc
class RetrievedChunk:
    chunk_id:    str
    chunk:       dict
    dense_score: float
    bm25_score:  float
    rrf_score:   float
    dense_rank:  int
    bm25_rank:   int

    @property
    def article_num(self):
        return self.chunk.get("article", self.chunk_id)

    @property
    def article_int(self):
        try:
            return int(self.article_num.split("-")[0])
        except Exception:
            return 0

    def to_dict(self) -> dict:
        return {
            "text":        fix_ocr(self.chunk.get("text", "")),
            "article":     self.article_num,
            "rrf_score":   self.rrf_score,
            "dense_score": self.dense_score,
            "bm25_score":  self.bm25_score,
            "dense_rank":  self.dense_rank,
            "bm25_rank":   self.bm25_rank,
            "chunk_id":    self.chunk_id,
            "chunk":       self.chunk,
            "chapter":     get_chapter(self.article_int),
            "subdomain":   get_subdomain(self.article_int),
            "source_type": self.chunk.get("source_type", "legal_text"),
        }

W_RERANK = 0.40
W_BM25   = 0.38
W_VECTOR = 0.22


# ══════════════════════════════════════════════════════════════
# EXPANSION GRAPHE POST-RERANK — multi-top (top-3)
# ══════════════════════════════════════════════════════════════
def expand_with_graph_post_rerank(passed_gap, chunks_dict, articles_by_num,
                                   top_score, top_arts_int: list):
    if not passed_gap:
        return passed_gap
    existing_arts = {r["article"] for r in passed_gap}
    added = []

    for rank_idx, art_int in enumerate(top_arts_int):
        inherit_factor = 0.75 - rank_idx * 0.10
        inherit_factor = max(inherit_factor, 0.40)
        neighbors = LEGAL_GRAPH.get(art_int, [])

        for n in neighbors:
            found_key = None
            for art_key in articles_by_num:
                try:
                    if int(art_key.split("-")[0]) == n and art_key not in existing_arts:
                        found_key = art_key
                        break
                except Exception:
                    continue
            if not found_key:
                direct_key = str(n)
                if direct_key in articles_by_num and direct_key not in existing_arts:
                    found_key = direct_key

            if found_key and found_key not in existing_arts:
                chunk_list = articles_by_num.get(found_key, [])
                if not chunk_list:
                    continue
                chunk = next(
                    (c for c in chunk_list if c.get("source_type","legal_text") == "legal_text"),
                    chunk_list[0]
                )
                new_score = round(top_score * inherit_factor, 6)
                new_item = {
                    "article":           found_key,
                    "text":              fix_ocr(chunk.get("text", "")),
                    "rerank_score":      round(passed_gap[0].get("rerank_score", 0) * inherit_factor, 6),
                    "final_score":       new_score,   # ← AJOUTÉ
                    "hybrid_score":      new_score,
                    "chapter":           get_chapter(n),
                    "subdomain":         get_subdomain(n),
                    "source_type":       chunk.get("source_type", "legal_text"),
                    "chunk":             chunk,
                    "propagation_bonus": 0.0,
                    "_graph_expanded":   True,
                    "_from_art":         art_int,
                }
                added.append(new_item)
                existing_arts.add(found_key)
                print(f"  🔗 Graphe post-rerank : voisin فصل {found_key} "
                      f"← فصل {art_int} (score={new_score:.4f}, ×{inherit_factor:.2f})")

    return passed_gap + added


# ══════════════════════════════════════════════════════════════
# MMR DIVERSIFICATION
# ══════════════════════════════════════════════════════════════
def mmr_diversify(items, top_k, lambda_param=0.72):
    if not items:
        return []
    if len(items) <= 1:
        return items[:top_k]
    selected   = [items[0]]
    candidates = list(items[1:])

    def safe_art_int(r):
        try:
            art_str = str(r.get("article", "0"))
            return int(art_str.split("-")[0]) if art_str.split("-")[0].isdigit() else 0
        except Exception:
            return 0

    while len(selected) < top_k and candidates:
        mmr_scores = []
        for c in candidates:
            c_chapter = get_chapter(safe_art_int(c))
            sim = max(
                [1.0 if get_chapter(safe_art_int(s)) == c_chapter else 0.0
                 for s in selected],
                default=0.0
            )
            score_c = c.get("hybrid_score", 0.0)
            mmr = lambda_param * score_c - (1 - lambda_param) * sim
            mmr_scores.append(mmr)

        best_idx = int(np.argmax(mmr_scores))
        selected.append(candidates[best_idx])
        candidates.pop(best_idx)

    if len(selected) < top_k and candidates:
        selected += candidates[:top_k - len(selected)]
    return selected


# ══════════════════════════════════════════════════════════════
# BOOST CONTRACTUEL
# ══════════════════════════════════════════════════════════════
def get_contract_boost(query: str, chapter: str) -> float:
    for topic, boost in CONTRACT_DOMAIN_BOOST.items():
        if topic in query:
            if topic in chapter or any(
                syn in chapter for syn in _ARABIC_SYNONYMS.get(topic, [])
            ):
                return boost
    return 0.0


# ══════════════════════════════════════════════════════════════
# HYBRID RETRIEVER v5.1  [CORRIGÉ]
# ══════════════════════════════════════════════════════════════
class HybridRetriever:
    def __init__(self, cids, chunks_dict, faiss_idx, chroma_col,
                 emb_model, bm25_model, bm25_cids, k=RRF_K):
        self.cids      = cids
        self.data      = chunks_dict
        self.faiss     = faiss_idx
        self.chroma    = chroma_col
        self.embed     = emb_model
        self.bm25      = bm25_model
        self.k         = k
        _global_idx    = {cid: i for i, cid in enumerate(cids)}
        self._bm25_to_global = [
            _global_idx[cid] for cid in bm25_cids if cid in _global_idx
        ]

    def _dense_faiss(self, query_vec: np.ndarray, n: int = RETRIEVAL_N_FAISS):
        scores, ids = self.faiss.search(query_vec, n)
        return list(zip(ids[0].tolist(), scores[0].tolist()))

    def _dense_chroma(self, query_vec: np.ndarray, n: int = RETRIEVAL_N_CHROMA,
                  filter_dict=None):
        try:
            q_vec = query_vec[0].tolist()
            res   = self.chroma.query(
                query_embeddings=[q_vec], n_results=n,
                where=filter_dict, include=["ids", "distances"])
            return [(self.cids.index(d), 1 - dist)
                for d, dist in zip(res["ids"][0], res["distances"][0])
                if d in self.cids]
        except Exception:
            return []

    def _sparse(self, query: str, n: int = RETRIEVAL_N_BM25):
        tokens = tokenize_ar(fix_ocr(query))
        if not tokens:
            return []
        sc  = self.bm25.get_scores(tokens)
        top = np.argsort(sc)[::-1][:n]
        result = []
        for j in top:
            if j < len(self._bm25_to_global):
                global_i = self._bm25_to_global[j]
                result.append((global_i, float(sc[j])))
        return result

    def _rrf_merge(self, *ranked_lists, n: int = 10,
                   rrf_min_score: float = None):
        if rrf_min_score is None:
            rrf_min_score = RRF_MIN_SCORE
        rrf_sc     = {}
        rank_maps  = [{} for _ in ranked_lists]
        score_maps = [{} for _ in ranked_lists]
        for li, ranked in enumerate(ranked_lists):
            for rk, (i, s) in enumerate(ranked):
                rrf_sc[i] = rrf_sc.get(i, 0) + 1.0 / (self.k + rk + 1)
                rank_maps[li][i]  = rk + 1
                score_maps[li][i] = s
        top = [i for i in sorted(rrf_sc, key=rrf_sc.get, reverse=True)
               if rrf_sc[i] >= rrf_min_score][:n]
        out = []
        for i in top:
            if not (0 <= i < len(self.cids)):
                continue
            cid = self.cids[i]
            if cid not in self.data:
                continue
            out.append(RetrievedChunk(
                chunk_id    = cid,
                chunk       = self.data[cid],
                dense_score = score_maps[0].get(i, 0.0),
                bm25_score  = score_maps[-1].get(i, 0.0),
                rrf_score   = rrf_sc[i],
                dense_rank  = rank_maps[0].get(i, 999),
                bm25_rank   = rank_maps[-1].get(i, 999),
            ))
        return out

    def retrieve(self, query: str, top_k: int = 5, chroma_filter=None,
             rrf_min_score: float = None,
             n_faiss: int = None, n_chroma: int = None,
             n_bm25: int = None):
        n_faiss  = n_faiss  or RETRIEVAL_N_FAISS
        n_chroma = n_chroma or RETRIEVAL_N_CHROMA
        n_bm25   = n_bm25   or RETRIEVAL_N_BM25
        
        query_vec = _encode(query, is_query=True)

        return self._rrf_merge(
            self._dense_faiss(query_vec, n_faiss),
            self._dense_chroma(query_vec, n_chroma, chroma_filter),
            self._sparse(query, n_bm25),
            n=top_k,
            rrf_min_score=rrf_min_score,
        )

    def retrieve_filtered(self, query: str, top_k: int = 5,
                          chroma_filter=None):
        return self.retrieve(query, top_k=top_k, chroma_filter=chroma_filter)

    @staticmethod
    def _compute_section_limit(section: str, passed_articles: list,
                                bge_top: float) -> int:
        if section in ALWAYS_PAIR_SECTIONS:
            return 3
        section_articles = [
            r for r in passed_articles
            if (get_section(
                    int(r["article"].split("-")[0])
                    if r["article"].split("-")[0].isdigit() else 0
                ) or get_chapter(
                    int(r["article"].split("-")[0])
                    if r["article"].split("-")[0].isdigit() else 0
                )) == section
        ]
        if len(section_articles) < 2:
            return 1
        bges = sorted([r.get("rerank_score", 0)
                       for r in section_articles], reverse=True)
        if bges[0] > 0 and bges[1] > 0:
            if abs(bges[0] - bges[1]) < BGE_PAIR_GAP:
                return 3
        return 2

    @staticmethod
    def _aggregate_by_article(reranked: list) -> list:
        best: dict = {}
        for r in reranked:
            art = r.get("article", "?")
            if (art not in best
                    or r.get("final_score", 0) > best[art].get("final_score", 0)):
                best[art] = r
        return sorted(best.values(),
                      key=lambda x: x.get("final_score", 0), reverse=True)

    def retrieve_reranked(self, query: str, top_k: int = 5,
                      retrieval_k: int = 200,
                      chroma_filter=None,
                      query_for_retrieval: str = None) -> list:

        normalized_query = normalize_arabic_legal(query)
        retrieval_query  = query_for_retrieval or normalized_query
        query_boosts     = classify_query(query)

    # ── Seuils de base (calculés une fois) ────────────────────
        thresholds_base = compute_adaptive_thresholds(query)
        _RRF_MIN_SCORE  = thresholds_base.get("RRF_MIN_SCORE", RRF_MIN_SCORE)

    # ── Retrieval (une fois) ───────────────────────────────────
        candidates = self.retrieve(
            retrieval_query, top_k=retrieval_k,
            chroma_filter=chroma_filter,
            rrf_min_score=_RRF_MIN_SCORE,
        )
        if not candidates:
            print("  ⛔ Aucun candidat récupéré.")
            return []

        candidates.sort(
            key=lambda c: (
                0 if c.chunk.get("source_type", "legal_text") == "legal_text" else 1,
                -c.rrf_score,
            )
        )

        cand_dicts = []
        for c in candidates:
            d = c.to_dict()
            d["chunk_id"] = c.chunk_id
            d["chunk"]    = c.chunk
            cand_dicts.append(d)

        bm25_max = max((c.get("bm25_score", 0) for c in cand_dicts), default=1.0)
        vec_max  = max((c.get("dense_score", 0) for c in cand_dicts), default=1.0)

    # ── BGE calculé UNE SEULE FOIS (hors boucle fallback) ─────
        reranked_base = rerank(query, cand_dicts, reranker, top_k=retrieval_k)
        reranked_base = self._aggregate_by_article(reranked_base)

    # ── Boucle fallback (seuils uniquement, pas de recompute) ──
        for factor in FALLBACK_FACTORS:
            thresholds = thresholds_base
            _RERANK_HARD_FLOOR = max(0.003, thresholds["RERANK_HARD_FLOOR"] * factor)
            _BGE_TOP_MIN       = max(0.01,  thresholds["BGE_TOP_MIN"]       * factor)
            _DOCTRINAL_MIN     = max(0.005, thresholds["DOCTRINAL_MIN"]     * factor)
            _BGE_SOFT_CENTER   = thresholds["BGE_SOFT_CENTER"] * factor
            _BGE_CTX_THRESHOLD = thresholds["BGE_CTX_THRESHOLD"] * factor

            print(f"  🎚️ Paliers facteur={factor:.2f} : "
                  f"HARD_FLOOR={_RERANK_HARD_FLOOR:.3f} "
                  f"TOP_MIN={_BGE_TOP_MIN:.3f} "
                  f"DOCTRINAL_MIN={_DOCTRINAL_MIN:.3f}")

        # Copie légère — pas de recompute BGE
            reranked = [dict(r) for r in reranked_base]
            passed_floor = []
            for r in reranked:
                bge = r.get("rerank_score", 0)
                r["bge_soft_w"]    = round(bge_soft_weight(bge, center=_BGE_SOFT_CENTER,
                                                        slope=BGE_SOFT_SLOPE), 4)
                r["bge_attenuated"] = round(bge * r["bge_soft_w"], 4)
                if bge >= _RERANK_HARD_FLOOR:
                    passed_floor.append(r)
                else:
                    print(f"  🔒 BGE<{_RERANK_HARD_FLOOR:.3f} : rejeté فصل "
                      f"{r['article']} (BGE={bge:.4f})")
            if not passed_floor:
                continue
            top_bge = passed_floor[0].get("rerank_score", 0)
            if top_bge < _BGE_TOP_MIN:
                print(f"  ⛔ GATE TOP : BGE_top={top_bge:.4f} < {_BGE_TOP_MIN}")
                continue

            adapt_k    = min(TOP_K_FIXED, MAX_SELECTED)
            passed_gap = passed_floor[:adapt_k]

            try:
                top1_art_int = int(passed_gap[0].get("article", "0").split("-")[0])
            except Exception:
                top1_art_int = 0

            for r in passed_gap:
                bge_n = r.get("rerank_score", 0)
                ratio = bge_n / top_bge if top_bge > 0 else 0.0
                r["_gap_ratio"]  = round(ratio, 4)
                r["_gap_weight"] = round(gap_weight(ratio), 4)

            for r in passed_gap:
                try:
                    art_int = int(r["article"].split("-")[0])
                except Exception:
                    art_int = 0

                bge_score      = r.get("bge_attenuated", r.get("rerank_score", 0))
                chapter        = r.get("chapter", get_chapter(art_int))
                text           = r.get("text", "")
                chap_boost     = get_chapter_boost(query, chapter)
                contract_boost = get_contract_boost(query, chapter)
                lex_bonus      = maxsim_bonus(query, text)

                base = compute_hybrid_score(
                rerank_score = bge_score,
                bm25_score   = r.get("bm25_score", 0),
                vector_score = r.get("dense_score", 0),
                article_num  = art_int,
                query_boosts = query_boosts,
                bm25_max     = bm25_max,
                vector_max   = vec_max,
                source_type  = r.get("source_type", "legal_text"),
                )
                ctx_bonus    = compute_contextual_bonus(art_int, top1_art_int, bge_score)
                sem_bonus    = compute_semantic_graph_bonus(art_int, top1_art_int, bge_score)
                art_priority = get_article_priority(query, r.get("article", ""))
                neg_penalty  = apply_negative_filter(query, text)

                raw_score = (base + chap_boost + contract_boost + lex_bonus
                         + ctx_bonus + sem_bonus + art_priority - neg_penalty)
                domain_factor = _contract_domain_penalty(query, r.get("article"))
                if domain_factor < 1.0:
                    raw_score *= domain_factor
                    r["domain_penalty_applied"] = round(domain_factor, 4)
                    print(f"  ⚠️ Pénalité domaine (hybrid) : فصل {r['article']} "
                      f"×{domain_factor:.2f}")

                weighted = raw_score * r["_gap_weight"]

                r["hybrid_score"]    = round(weighted, 6)
                r["chapter_boost"]   = round(chap_boost, 4)
                r["contract_boost"]  = round(contract_boost, 4)
                r["lexical_bonus"]   = round(lex_bonus, 4)
                r["ctx_bonus"]       = round(ctx_bonus, 4)
                r["sem_bonus"]       = round(sem_bonus, 4)
                r["art_priority"]    = round(art_priority, 4)
                r["neg_penalty"]     = round(neg_penalty, 4)
                r["centrality"]      = get_centrality(art_int)
                r["gap_weight"]      = r["_gap_weight"]
                r["gap_ratio"]       = r["_gap_ratio"]
                r["doctrinal_score"] = round(
                r.get("bge_attenuated", bge_score) + art_priority
                + get_centrality(art_int) + query_boosts.get(art_int, 0), 4
            )

        # ── Gate doctrinal avec bypass élargi ─────────────────
            post_gate = []
            for r in passed_gap:
                doc_score = r.get("doctrinal_score", 0)
                art_prio  = r.get("art_priority", 0)
                try:
                    art_int_r = int(r["article"].split("-")[0])
                except Exception:
                    art_int_r = 0
                q_boost_r = query_boosts.get(art_int_r, 0)

                if doc_score >= _DOCTRINAL_MIN:
                    post_gate.append(r)
                elif art_prio >= 0.08:
                    post_gate.append(r)
                    print(f"  ⚡ Gate bypass-A (art_prio={art_prio:.3f}) : "
                      f"فصل {r['article']}")
                elif art_prio >= 0.06 and q_boost_r >= 0.10:
                    post_gate.append(r)
                    print(f"  ⚡ Gate bypass-B (art_prio={art_prio:.3f}, "
                      f"q_boost={q_boost_r:.3f}) : فصل {r['article']}")
                else:
                    print(f"  🚫 Gate doctrinal : rejeté فصل {r['article']} "
                      f"(doc={doc_score:.3f} < {_DOCTRINAL_MIN:.3f})")

            if not post_gate:
                continue
            passed_gap = post_gate

        # ── Propagation sémantique ─────────────────────────────
            top1_score         = passed_gap[0]["hybrid_score"]
            sem_neighbors_top1 = get_semantic_neighbors(top1_art_int)
            seen_pairs         = set()
            for r in passed_gap:
                try:
                    art_int = int(r["article"].split("-")[0])
                except Exception:
                    continue
                pair = (top1_art_int, art_int)
                if (art_int in sem_neighbors_top1
                        and art_int != top1_art_int
                        and pair not in seen_pairs):
                    seen_pairs.add(pair)
                    prop_value = round(top1_score * PROPAGATION_FACTOR, 6)
                    current    = r.get("propagation_bonus", 0.0)
                    if prop_value > current:
                        r["hybrid_score"]      = round(r["hybrid_score"] + prop_value - current, 6)
                        r["propagation_bonus"] = prop_value
                        print(f"  🔗 Propagation : فصل {art_int} ← +{prop_value:.4f}")

        # ── Expansion graphe multi-top (top-3) ─────────────────
            top_arts_int = []
            for r in passed_gap[:3]:
                try:
                    top_arts_int.append(int(r["article"].split("-")[0]))
                except Exception:
                    pass

            passed_gap = expand_with_graph_post_rerank(
                passed_gap, chunks_dict, articles_by_num,
                top1_score, top_arts_int
        )

        # ── Déduplication ──────────────────────────────────────
            seen = {}
            for r in sorted(passed_gap, key=lambda x: x["hybrid_score"], reverse=True):
                if r["article"] not in seen:
                    seen[r["article"]] = r
            sorted_unique = sorted(seen.values(),
                               key=lambda x: x["hybrid_score"], reverse=True)

        # ── Diversification MMR ────────────────────────────────
            diversified = mmr_diversify(sorted_unique, top_k)

            if diversified:
                arts_ret = [r["article"] for r in diversified]
                print(f"  ✅ Pipeline v5.1 : {len(diversified)} articles "
                  f"[BGE_top={top_bge:.3f}] → {arts_ret}")
                return diversified

        print("  ⛔ Aucun résultat après fallback.")
        return []


# ══════════════════════════════════════════════════════════════
# VÉRIFICATION DES ARTICLES CLÉS
# ══════════════════════════════════════════════════════════════
print("\n" + "━"*60)
print("VÉRIFICATION DES ARTICLES CLÉS DANS LE RAG")
print("━"*60)
missing_articles = [
    "2", "9", "39", "66", "67", "77", "78", "79", "98",
    "229", "255", "256", "261", "306", "307", "308",
    "321", "322", "488", "628", "629", "635", "664", "668",
    "893", "1117", "1118", "1120", "1171", "1172", "1248",
]
articles_by_num_verif: dict = {}
for c in all_chunks:
    art = c.get("article", "")
    if art:
        articles_by_num_verif.setdefault(art, []).append(c)

_chunks_dict_preview = {c["chunk_id"]: c for c in deduplicate_chunks(all_chunks)}
for art in missing_articles:
    if art in articles_by_num_verif:
        chs    = articles_by_num_verif[art]
        types  = sorted({c.get("source_type", "legal_text") for c in chs})
        n_legal = sum(1 for c in chs if c.get("source_type","legal_text") == "legal_text")
        flag   = "✅" if n_legal > 0 else "⚠️ (aucun chunk legal_text!)"
        print(f"{flag} Article {art} : {len(chs)} chunks, "
              f"{n_legal} legal_text — types={types}")
    else:
        print(f"❌ Article {art} NON TROUVÉ dans le RAG ← à corriger en amont!")
print("━"*60)

# ══════════════════════════════════════════════════════════════
# BM25 — restreint aux legal_text
# ══════════════════════════════════════════════════════════════
_deduped_chunks  = deduplicate_chunks(all_chunks)
chunks_dict      = {c["chunk_id"]: c for c in _deduped_chunks}
chunk_ids        = [cid for cid in chunk_ids if cid in chunks_dict]
_cid_to_global_idx: dict[str, int] = {cid: i for i, cid in enumerate(chunk_ids)}
chunk_ids_legal = [
    cid for cid in chunk_ids
    if chunks_dict[cid].get("source_type", "legal_text") == "legal_text"
]
bm25_texts  = [enrich_text(chunks_dict[cid]) for cid in chunk_ids_legal]
tokenized   = [tokenize_ar(t) for t in bm25_texts]
bm25        = BM25Okapi(tokenized)
_n_excluded = len(chunk_ids) - len(chunk_ids_legal)
print(f"  ✅ BM25 : {len(tokenized)} docs indexés ({_n_excluded} non-legal_text exclus)")

# ══════════════════════════════════════════════════════════════
# articles_by_num
# ══════════════════════════════════════════════════════════════
articles_by_num: dict = {}
for c in _deduped_chunks:
    articles_by_num.setdefault(c.get("article", ""), []).append(c)
print(f"  ✅ articles_by_num : {len(articles_by_num)} articles")

# ══════════════════════════════════════════════════════════════
# INSTANCIATION
# ══════════════════════════════════════════════════════════════
retriever = HybridRetriever(
    cids        = chunk_ids,
    chunks_dict = chunks_dict,
    faiss_idx   = faiss_index,
    chroma_col  = chroma_collection,
    emb_model   = embed_model,
    bm25_model  = bm25,
    bm25_cids   = chunk_ids_legal,
)
print(f"  ✅ HybridRetriever v5.1 prêt – {len(chunk_ids)} chunks globaux"
      f" | {len(chunk_ids_legal)} dans BM25"
      f" | retrieval_k(faiss/chroma/bm25)="
      f"{RETRIEVAL_N_FAISS}/{RETRIEVAL_N_CHROMA}/{RETRIEVAL_N_BM25}")
print(f"  🔧 TOP_K_FIXED={TOP_K_FIXED} | MAX_SELECTED={MAX_SELECTED} "
      f"| FALLBACK={FALLBACK_FACTORS}")

# ══════════════════════════════════════════════════════════════
# TESTS v5.1
# ══════════════════════════════════════════════════════════════
print("\n" + "━"*68)
print("TESTS v5.1 — requêtes ayant échoué (MRR=0)")
print("━"*68)
_test_queries = [
    ("عقد الكفالة والضمان",                    [1117, 1118, 1119]),
    ("عقد الإيجار والتزامات الأطراف",          [617, 618, 663, 664]),
    ("بطلان العقد وأسبابه",                    [60, 65, 66, 67]),
    ("مسؤولية الموكل عن تصرفات الوكيل",        [879, 880, 893]),
    ("الإعذار وشروط الفسخ القضائي",             [255, 256, 261]),   # requête MRR=0
    ("الكسب غير المشروع وشبه العقد",            [77, 78, 79, 98]),  # requête MRR=0
    ("القوة الملزمة للعقد وأثره بين الطرفين",   [228, 229, 230]),
]
for q, gold in _test_queries:
    gold_set = set(str(g) for g in gold)
    print(f"\n🔍 {q}  [gold={gold}]")
    results = retriever.retrieve_reranked(q, top_k=5, retrieval_k=200)
    if not results:
        print(f"  ⛔ Aucun résultat")
        continue
    returned = [r["article"] for r in results]
    hits = set(returned) & gold_set
    mrr = 0.0
    for rank, art in enumerate(returned, 1):
        if art in gold_set:
            mrr = 1.0/rank
            break
    status = "✅" if mrr == 1.0 else ("⚠️" if mrr > 0 else "❌")
    print(f"  {status} Retourné : {returned}")
    print(f"     Hits={sorted(hits)}  MRR={mrr:.3f}")

# ══════════════════════════════════════════════════════════════
# BENCHMARKS STRATIFIÉS
# ══════════════════════════════════════════════════════════════
print("\n" + "━"*70)
print("BENCHMARKS STRATIFIÉS — v5.1")
print("━"*70)

EVAL_SET = [
    {"query": "الأهلية المدنية للقاصر",               "gold": [4, 9, 12]},
    {"query": "عقد البيع والشراء",                     "gold": [478, 488, 502]},
    {"query": "التعويض عن الضرر",                      "gold": [264, 77, 98]},
    {"query": "الالتزامات التعاقدية",                  "gold": [230, 18, 393]},
    {"query": "الإفلاس والتسوية القضائية",             "gold": [1247, 1248]},
    {"query": "فسخ العقد",                             "gold": [259, 260, 261]},
    {"query": "عقد الكفالة والضمان",                   "gold": [1117, 1118, 1119]},
    {"query": "الوكالة والتفويض",                      "gold": [879, 880, 893]},
    {"query": "انقضاء الالتزامات بالوفاء",            "gold": [320, 321, 322]},
    {"query": "المسؤولية التقصيرية",                  "gold": [77, 78, 79]},
    {"query": "بطلان العقد وأسبابه",                   "gold": [60, 65, 66, 67]},
    {"query": "عقد الإيجار والتزامات الأطراف",         "gold": [617, 618, 663, 664]},
    {"query": "الإعذار وشروط الفسخ القضائي",           "gold": [255, 256, 261]},
    {"query": "مسؤولية الموكل عن تصرفات الوكيل",       "gold": [879, 880, 893]},
    {"query": "الكسب غير المشروع وشبه العقد",          "gold": [77, 78, 79, 98]},
    {"query": "القوة الملزمة للعقد وأثره بين الطرفين", "gold": [228, 229, 230]},
]

KS = [1, 3, 5, 10]

def evaluate_retrieval_function(retrieve_fn, eval_set, ks, label=""):
    metrics = {k: {'recall':[],'precision':[],'f1':[],'ndcg':[],'hit':[]} for k in ks}
    all_mrr = []
    for item in eval_set:
        query  = item["query"]
        gold   = set(str(g) for g in item["gold"])
        results = retrieve_fn(query, max(ks))
        returned = [str(r) for r in results]
        for k in ks:
            top_k_arts = returned[:k]
            recall    = len(set(top_k_arts) & gold) / len(gold) if gold else 0
            precision = len(set(top_k_arts) & gold) / k if k else 0
            f1 = 2*recall*precision/(recall+precision) if (recall+precision)>0 else 0.0
            hit  = 1 if len(set(top_k_arts) & gold) > 0 else 0
            dcg  = sum(1.0/np.log2(idx+2) for idx, art in enumerate(top_k_arts) if art in gold)
            idcg = sum(1.0/np.log2(idx+2) for idx in range(min(len(gold), k)))
            ndcg = dcg/idcg if idcg>0 else 0.0
            metrics[k]['recall'].append(recall)
            metrics[k]['precision'].append(precision)
            metrics[k]['f1'].append(f1)
            metrics[k]['hit'].append(hit)
            metrics[k]['ndcg'].append(ndcg)
        mrr = 0.0
        for rank, art in enumerate(returned, 1):
            if art in gold:
                mrr = 1.0/rank
                break
        all_mrr.append(mrr)

    avg = {k: {m: np.mean(metrics[k][m]) for m in ['recall','precision','f1','ndcg','hit']} for k in ks}
    avg_mrr = np.mean(all_mrr)
    print(f"\n▶ {label}")
    print(f"{'Métrique':<12}", end="")
    for k in ks:
        print(f"  K={k:<3}", end="")
    print()
    for m in ['recall','precision','f1','ndcg','hit']:
        print(f"{m.capitalize():<12}", end="")
        for k in ks:
            print(f"  {avg[k][m]:.3f}", end="")
        print()
    print(f"{'MRR':<12} {'':>6}  {avg_mrr:.3f}")
    composite = (0.35*avg[3]['recall'] + 0.25*avg[3]['ndcg']
                 + 0.25*avg_mrr + 0.15*avg[3]['precision'])
    print(f"Score composite (K=3) : {composite:.4f}")
    return avg, avg_mrr, composite

def full_pipeline(query, top_k):
    results = retriever.retrieve_reranked(query, top_k=top_k, retrieval_k=200)
    return [r["article"] for r in results]

bench = evaluate_retrieval_function(full_pipeline, EVAL_SET, KS,
                                    label="(C) Full pipeline v5.1")

print(f"\n✅ Cellule 7 v5.1 terminée")
print(f"   Variables exportées : retriever, articles_by_num, chunks_dict, etc.")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
VÉRIFICATION DES ARTICLES CLÉS DANS LE RAG
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
❌ Article 2 NON TROUVÉ dans le RAG ← à corriger en amont!
❌ Article 9 NON TROUVÉ dans le RAG ← à corriger en amont!
✅ Article 39 : 1 chunks, 1 legal_text — types=['legal_text']
✅ Article 66 : 1 chunks, 1 legal_text — types=['legal_text']
✅ Article 67 : 1 chunks, 1 legal_text — types=['legal_text']
✅ Article 77 : 1 chunks, 1 legal_text — types=['legal_text']
✅ Article 78 : 1 chunks, 1 legal_text — types=['legal_text']
✅ Article 79 : 1 chunks, 1 legal_text — types=['legal_text']
✅ Article 98 : 1 chunks, 1 legal_text — types=['legal_text']
✅ Article 229 : 1 chunks, 1 legal_text — types=['legal_text']
✅ Article 255 : 1 chunks, 1 legal_text — types=['legal_text']
✅ Article 256 : 1 chunks, 1 legal_text — types=['legal_text']
✅ Article 261 : 1 chunks, 1 legal_text — types=['legal_text']
✅ Article 306 : 1 chunks, 1 legal_text — ty

## Cellule 8+9 — Ontologie


In [8]:
# ══════════════════════════════════════════════════════════════
# CELLULE 8+9 — Doctrinal Dominance v6.6
# Améliorations vs v6.5 :
#   [v6.6-A]  Article 393 faux positif : boost réduit + filtre négatif ciblé
#   [v6.6-B]  Articles 2, 39, 1120 ajoutés aux cibles d'injection
#   [v6.6-C]  الكسب غير المشروع : articles 99/100 ajoutés à l'ontologie
#   [v6.6-D]  Topic "شروط العقد" nouveau pour récupérer 2, 19, 39, 57
#   [v6.6-E]  Requêtes multi-concept : pondération chapitre principal renforcée
#   [v6.6-F]  Diagnostic de l'article 1248 (absent du RAG ?)
# ══════════════════════════════════════════════════════════════

import re
import math
import numpy as np
from types import MethodType

# ──────────── Dépendances ────────────
for varname in [
    "retriever", "reranker", "rerank",
    "fix_ocr", "tokenize_ar",
    "classify_query", "get_chapter", "get_chapter_boost",
    "compute_hybrid_score", "maxsim_bonus",
    "gap_weight", "adaptive_top_k",
    "chunks_dict", "all_chunks",
    "CHAPTER_PRIOR", "QUERY_CHAPTER_BOOST",
    "apply_negative_filter",
    "LEGAL_GRAPH", "LEGAL_SEMANTIC_GRAPH",
    "get_article_priority", "get_centrality", "get_section",
    "articles_by_num",
    "normalize_arabic_legal",
    "extract_article_num_from_text",
    "_STRUCTURAL_FP_ARTICLES",
    "_STRUCTURAL_FP_PENALTY",
    "_STRUCTURAL_FP_RELIEF",
    "_contract_domain_penalty",
]:
    assert varname in globals(), f"❌ {varname} non défini"

print("✅ Dépendances C7 v5.1 vérifiées.")

# ══════════════════════════════════════════════════════════════
# [v6.6-F] DIAGNOSTIC ARTICLE 1248
# ══════════════════════════════════════════════════════════════
_art_1248_chunks = articles_by_num.get("1248", [])
_art_1248_legal  = [c for c in _art_1248_chunks
                    if c.get("source_type", "legal_text") == "legal_text"]
if not _art_1248_legal:
    print("⚠️  [v6.6-F] الفصل 1248 ABSENT du RAG ou sans chunk legal_text "
          f"({len(_art_1248_chunks)} chunks total) "
          "→ l'injection ne pourra pas le récupérer. "
          "Corriger l'indexation en amont.")
else:
    print(f"✅ [v6.6-F] الفصل 1248 présent : {len(_art_1248_legal)} chunk(s) legal_text")

# ══════════════════════════════════════════════════════════════
# 1. EXPANSION DE REQUÊTE (legacy, filet de sécurité)
# ══════════════════════════════════════════════════════════════
LEGAL_EXPANSION = {
    "الإفلاس":          ["التصفية القضائية", "السنديك", "الإعسار", "الدائنين", "التفليسة"],
    "التسوية القضائية": ["الإفلاس", "السنديك", "الدائنين", "الإعسار"],
    "قاصر":             ["ناقص الأهلية", "الوصي", "المقدم", "الأهلية المدنية"],
    "أهلية":            ["القاصر", "ناقص الأهلية", "الوصي", "الأب"],
    "تعويض":            ["الضرر", "المسؤولية المدنية", "الخسارة", "الفعل الضار"],
    "ضرر":              ["التعويض", "المسؤولية", "الخسارة", "الفعل غير المشروع"],
    "بيع":              ["ملكية", "ثمن", "المبيع", "البائع", "المشتري", "التسليم"],
    "شراء":             ["عقد البيع", "المبيع", "الثمن", "التسليم"],
    "إيجار":            ["المؤجر", "المستأجر", "الكراء", "الأجرة", "المدة",
                         "تسليم العين", "الضمان", "الفسخ", "مدة محددة",
                         "مكري", "مكتري"],
    "كراء":             ["إيجار", "المكري", "المكتري", "الأجرة", "العين المكتراة"],
    "مكتري":            ["مستأجر", "الكراء", "الأجرة", "التزامات المكتري"],
    "مكري":             ["مؤجر", "التسليم", "الضمان", "الصيانة"],
    "كفالة":            ["الكفيل", "المدين الأصلي", "الدائن", "الضامن"],
    "رهن":              ["الدائن المرتهن", "الراهن", "الضمان", "التأمين العيني"],
    "وكالة":            ["الوكيل", "الموكل", "التفويض", "النيابة"],
    "عقد":              ["الالتزام", "التراضي", "الإيجاب", "القبول", "الأطراف"],
    "التزام":           ["العقد", "الدائن", "المدين", "الوفاء", "الإخلال"],
    "فسخ":              ["حل العقد", "انحلال الالتزام", "عدم التنفيذ",
                         "الشرط الفاسخ", "الفسخ القضائي", "البطلان", "الإلغاء"],
    "بطلان":            ["الفسخ", "الإبطال", "النسبي", "المطلق", "الإلغاء",
                         "أسباب البطلان", "عيوب الرضى"],
    "مسؤولية":          ["الضرر", "الخطأ", "العلاقة السببية", "التعويض",
                         "الفعل الضار", "شبه الجريمة"],
    "شركة":             ["الشركاء", "رأس المال", "الأرباح", "الذمة المالية"],
    "إعذار":            ["الإخطار", "التنبيه", "الإنذار", "مهلة التنفيذ",
                         "الفسخ القضائي", "عدم التنفيذ", "شروط الفسخ",
                         "الفصل 255", "الفصل 256", "الفصل 261"],
    "الإعذار":          ["الإخطار", "التنبيه", "الإنذار", "مهلة التنفيذ",
                         "الفسخ القضائي", "عدم التنفيذ", "شروط الفسخ",
                         "الفصل 255", "الفصل 256", "الفصل 261"],
    "الكسب غير المشروع": ["الإثراء بلا سبب", "الفعل الضار", "شبه الجرائم",
                           "المسؤولية التقصيرية", "الجريمة المدنية",
                           "الفعل غير المشروع", "دعوى الاسترداد",
                           "الفصل 77", "الفصل 78", "الفصل 79", "الفصل 98"],
    "شبه العقد":         ["الإثراء بلا سبب", "الفضالة", "الدفع غير المستحق",
                           "الكسب غير المشروع", "المسؤولية التقصيرية",
                           "الفصل 77", "الفصل 78", "الفصل 79"],
    "عمل":              ["إجارة الخدمة", "عقد العمل", "أجير", "رب العمل", "مدة العمل"],
    "عقد العمل":         ["إجارة الخدمة", "أجير", "رب العمل", "مدة محددة"],
}

def expand_query_v1(query: str) -> str:
    expansions = []
    for kw, terms in LEGAL_EXPANSION.items():
        if kw in query:
            expansions.extend(terms)
    if not expansions:
        return query
    unique = list(dict.fromkeys(expansions))[:8]
    return query + " " + " ".join(unique)

# ══════════════════════════════════════════════════════════════
# 2. HIÉRARCHIE COMPLÈTE
# ══════════════════════════════════════════════════════════════
LOC_HIERARCHY = [
    {"book": "الكتاب الأول: الالتزامات بوجه عام",
     "title": "الباب الأول: مصادر الالتزامات",
     "chapter": "العقود والالتزامات التعاقدية — أحكام عامة",
     "section": "الأهلية والتراضي",          "articles": range(1,   16)},
    {"book": "الكتاب الأول: الالتزامات بوجه عام",
     "title": "الباب الأول: مصادر الالتزامات",
     "chapter": "العقود والالتزامات التعاقدية — أحكام عامة",
     "section": "شروط انعقاد العقد",          "articles": range(16,  60)},
    {"book": "الكتاب الأول: الالتزامات بوجه عام",
     "title": "الباب الأول: مصادر الالتزامات",
     "chapter": "العقود والالتزامات التعاقدية — البطلان والفسخ",
     "section": "أسباب البطلان (عيوب الرضى)",  "articles": range(60,  68)},
    {"book": "الكتاب الأول: الالتزامات بوجه عام",
     "title": "الباب الأول: مصادر الالتزامات",
     "chapter": "العقود والالتزامات التعاقدية — أحكام عامة",
     "section": "البطلان والفسخ",             "articles": range(68, 109)},
    {"book": "الكتاب الأول: الالتزامات بوجه عام",
     "title": "الباب الأول: مصادر الالتزامات",
     "chapter": "العقود والالتزامات التعاقدية — البطلان والفسخ",
     "section": "أسباب البطلان",              "articles": range(109, 160)},
    {"book": "الكتاب الأول: الالتزامات بوجه عام",
     "title": "الباب الأول: مصادر الالتزامات",
     "chapter": "العقود والالتزامات التعاقدية — البطلان والفسخ",
     "section": "الفسخ القضائي وأثاره",       "articles": range(160, 230)},
    {"book": "الكتاب الأول: الالتزامات بوجه عام",
     "title": "الباب الثاني: آثار الالتزامات",
     "chapter": "آثار الالتزامات التعاقدية",
     "section": "القوة الملزمة للعقد",         "articles": range(230, 265)},
    {"book": "الكتاب الأول: الالتزامات بوجه عام",
     "title": "الباب الثاني: آثار الالتزامات",
     "chapter": "التعويض والضرر",
     "section": "المسؤولية التقصيرية والضرر", "articles": range(77,  109)},
    {"book": "الكتاب الأول: الالتزامات بوجه عام",
     "title": "الباب الثاني: آثار الالتزامات",
     "chapter": "التعويض والضرر",
     "section": "تقدير التعويض",              "articles": range(265, 330)},
    {"book": "الكتاب الأول: الالتزامات بوجه عام",
     "title": "الباب الثالث: انقضاء الالتزامات",
     "chapter": "انقضاء الالتزامات",
     "section": "الوفاء",                      "articles": range(320, 360)},
    {"book": "الكتاب الأول: الالتزامات بوجه عام",
     "title": "الباب الثالث: انقضاء الالتزامات",
     "chapter": "انقضاء الالتزامات",
     "section": "التجديد والإبراء والمقاصة",  "articles": range(360, 380)},
    {"book": "الكتاب الأول: الالتزامات بوجه عام",
     "title": "الباب الثالث: انقضاء الالتزامات",
     "chapter": "انقضاء الالتزامات",
     "section": "التقادم",                     "articles": range(380, 478)},
    {"book": "الكتاب الثاني: العقود المسماة", "title": "عقد البيع",
     "chapter": "عقد البيع — أحكام عامة",
     "section": "تعريف البيع وشروطه",         "articles": range(478, 510)},
    {"book": "الكتاب الثاني: العقود المسماة", "title": "عقد البيع",
     "chapter": "عقد البيع — التزامات الأطراف",
     "section": "التسليم والضمان",             "articles": range(510, 580)},
    {"book": "الكتاب الثاني: العقود المسماة", "title": "عقد البيع",
     "chapter": "عقد البيع — أنواع خاصة",
     "section": "الشرط الفاسخ وخيار الفسخ",  "articles": range(580, 619)},
    {"book": "الكتاب الثاني: العقود المسماة", "title": "عقد الكراء",
     "chapter": "عقد الكراء — أحكام عامة",
     "section": "التزامات المؤجر والمستأجر",  "articles": range(627, 700)},
    {"book": "الكتاب الثاني: العقود المسماة", "title": "عقد إجارة الخدمة أو العمل",
     "chapter": "عقد إجارة الخدمة أو العمل",
     "section": "تعريف العقد وشروطه وإثباته", "articles": range(723, 780)},
    {"book": "الكتاب الثاني: العقود المسماة", "title": "عقد القرض",
     "chapter": "عقد القرض",
     "section": "القرض بالاستهلاك والاستعمال","articles": range(856, 879)},
    {"book": "الكتاب الثاني: العقود المسماة", "title": "عقد الوكالة",
     "chapter": "عقد الوكالة — أحكام عامة",
     "section": "التفويض والنيابة",            "articles": range(879, 920)},
    {"book": "الكتاب الثاني: العقود المسماة", "title": "عقد الوكالة",
     "chapter": "عقد الوكالة — انتهاء الوكالة",
     "section": "انقضاء الوكالة وآثاره",       "articles": range(920, 942)},
    {"book": "الكتاب الثاني: العقود المسماة", "title": "عقد الشركة",
     "chapter": "عقد الشركة",
     "section": "أنواع الشركات وأحكامها",      "articles": range(982, 1065)},
    {"book": "الكتاب الثاني: العقود المسماة", "title": "عقد الكفالة",
     "chapter": "عقد الكفالة — أحكام عامة",
     "section": "شروط الكفالة والتزامات الكفيل","articles": range(1117, 1145)},
    {"book": "الكتاب الثاني: العقود المسماة", "title": "عقد الكفالة",
     "chapter": "عقد الكفالة — انقضاء الكفالة",
     "section": "انقضاء الكفالة ورجوع الكفيل","articles": range(1145, 1170)},
    {"book": "الكتاب الثالث: التأمينات العينية",
     "title": "الرهن والتأمينات",
     "chapter": "الرهن الحيازي والرهن الرسمي",
     "section": "أحكام الرهن وآثاره",          "articles": range(1170, 1220)},
    {"book": "الكتاب الثالث: التأمينات العينية",
     "title": "الرهن والتأمينات",
     "chapter": "الإفلاس والتسوية القضائية",
     "section": "الدائنون الممتازون والسنديك", "articles": range(1220, 1260)},
]

def get_full_hierarchy(article_num: int) -> dict:
    best, best_len = None, 9999
    for entry in LOC_HIERARCHY:
        if article_num in entry["articles"]:
            span = len(entry["articles"])
            if span < best_len:
                best, best_len = entry, span
    return best or {
        "book": "قانون الالتزامات والعقود",
        "title": "", "chapter": get_chapter(article_num), "section": "",
    }

# ══════════════════════════════════════════════════════════════
# 3. ONTOLOGIE JURIDIQUE  v6.6
# [v6.6-C] الكسب غير المشروع : ajout articles 99/100
# [v6.6-D] Nouveau topic "شروط العقد" pour récupérer 2, 19, 39, 57
# ══════════════════════════════════════════════════════════════
LEGAL_ONTOLOGY = {
    "البيع": {
        "synonyms": ["شراء","مبيع","بائع","مشترٍ","ثمن","تسليم المبيع",
                     "عقد البيع","نقل الملكية"],
        "articles": [478,488,490,502,510,519,530,577,601],
        "sections": ["تعريف البيع وشروطه","التسليم والضمان",
                     "الشرط الفاسخ وخيار الفسخ"],
        "main_chapter": "عقد البيع — أحكام عامة",
        "boost": 0.14,
    },
    "الفسخ": {
        "synonyms": ["فسخ العقد","إلغاء العقد","انحلال العقد","الشرط الفاسخ",
                     "الفسخ القضائي","فسخ البيع","التفاسخ","إقالة",
                     "حل العقد","انحلال الالتزام","عدم التنفيذ"],
        "articles": [259,260,261,262,263,230,235,601,602],
        "sections": ["الفسخ القضائي وأثاره","الشرط الفاسخ وخيار الفسخ"],
        "main_chapter": "العقود والالتزامات التعاقدية — البطلان والفسخ",
        "boost": 0.16,
    },
    "الوكالة": {
        "synonyms": ["وكيل","موكل","تفويض","نيابة","إنابة","الوكالة بالعمولة",
                     "وكالة عامة","وكالة خاصة","تفويض السلطة",
                     "مسؤولية الموكل","تصرفات الوكيل"],
        "articles": [879,880,881,882,893,908,920,926,938],
        "sections": ["التفويض والنيابة","انقضاء الوكالة وآثاره"],
        "main_chapter": "عقد الوكالة — أحكام عامة",
        "boost": 0.14,
    },
    "الوفاء": {
        "synonyms": ["أداء الالتزام","تنفيذ الالتزام","الدفع","إيفاء الدين",
                     "الوفاء بالالتزام","انقضاء بالوفاء","براءة الذمة"],
        "articles": [320,321,322,323,324,325,326,327],
        "sections": ["الوفاء"],
        "main_chapter": "انقضاء الالتزامات",
        "boost": 0.16,
    },
    "الكفالة": {
        "synonyms": ["كفيل","ضامن","ضمان شخصي","مدين أصلي","الكفالة التضامنية",
                     "الكفالة البسيطة","حق الرجوع","دعوى الكفيل",
                     "الضمان","الكفيل","الرجوع","التضامن"],
        # [v6.6-B] ajout de 1120 (renonciation au bénéfice de discussion)
        "articles": [1117,1118,1119,1120,1121,1129,1145,1151],
        "sections": ["شروط الكفالة والتزامات الكفيل","انقضاء الكفالة ورجوع الكفيل"],
        "main_chapter": "عقد الكفالة — أحكام عامة",
        "boost": 0.14,
    },
    "المسؤولية التقصيرية": {
        "synonyms": ["الفعل الضار","الجرائم وأشباه الجرائم","شبه الجريمة",
                     "الخطأ التقصيري","المسؤولية المدنية","الفعل غير المشروع",
                     "الضرر المباشر","العلاقة السببية","الكسب غير المشروع",
                     "شبه العقد"],
        "articles": [77,78,79,80,81,82,83,84,85,98,99,100],
        "sections": ["المسؤولية التقصيرية والضرر"],
        "main_chapter": "التعويض والضرر",
        "boost": 0.18,
    },
    "التعويض": {
        "synonyms": ["تقدير التعويض","الضرر المادي","الضرر المعنوي","الخسارة",
                     "الكسب الفائت","التعويض القضائي","الضرر المباشر","رد الاعتبار"],
        "articles": [264,265,266,267,268,269,270,98,77],
        "sections": ["تقدير التعويض","المسؤولية التقصيرية والضرر"],
        "main_chapter": "التعويض والضرر",
        "boost": 0.14,
    },
    "الإفلاس": {
        "synonyms": ["التصفية القضائية","السنديك","الإعسار","التفليسة",
                     "الدائنون الممتازون","توقف عن الدفع","الحجز","غرماء"],
        "articles": [1247,1248,1249,1230,1231,1232],
        "sections": ["الدائنون الممتازون والسنديك"],
        "main_chapter": "الإفلاس والتسوية القضائية",
        "boost": 0.18,
    },
    "الأهلية": {
        "synonyms": ["قاصر","ناقص الأهلية","محجور عليه","الولي","الوصي",
                     "المقدم","رفع الحجر","أهلية التعاقد","أهلية الأداء"],
        # [v6.6-B] ajout article 2 (personnalité juridique)
        "articles": [1,2,3,4,5,9,12,13,14,15],
        "sections": ["الأهلية والتراضي"],
        "main_chapter": "العقود والالتزامات التعاقدية — أحكام عامة",
        "boost": 0.16,
    },
    "الرهن": {
        "synonyms": ["رهن حيازي","رهن رسمي","الدائن المرتهن","الراهن",
                     "التأمين العيني","الرهن التجاري","فك الرهن","حق التتبع"],
        "articles": [1170,1171,1172,1180,1190,1200,1210],
        "sections": ["أحكام الرهن وآثاره"],
        "main_chapter": "الرهن الحيازي والرهن الرسمي",
        "boost": 0.14,
    },
    "الإيجار": {
        "synonyms": ["كراء","مؤجر","مستأجر","أجرة","عقد الكراء","المدة",
                     "انتهاء الإيجار","الإخلاء","الكراء السكني",
                     "مكري","مكتري","التزامات المكتري","التزامات المكري",
                     "تسليم العين المكتراة"],
        "articles": [617,618,619,663,664,668],
        "sections": ["التزامات المؤجر والمستأجر"],
        "main_chapter": "عقد الكراء — أحكام عامة",
        "boost": 0.16,
    },
    "انقضاء الالتزامات": {
        "synonyms": ["الإبراء","المقاصة","التجديد","اتحاد الذمة",
                     "استحالة التنفيذ","التقادم","سقوط الحق"],
        "articles": [320,335,360,370,380,393,400,430],
        "sections": ["الوفاء","التجديد والإبراء والمقاصة","التقادم"],
        "main_chapter": "انقضاء الالتزامات",
        "boost": 0.14,
    },
    "الالتزامات التعاقدية": {
        "synonyms": ["القوة الملزمة","حسن النية","تنفيذ العقد","المدين",
                     "الدائن","آثار العقد","عدم التنفيذ","نظرية الالتزام",
                     "الالتزام التعاقدي","القانون العقدي","أثر العقد بين الطرفين"],
        "articles": [230,228,229,235,18,264,259,260],  # [v6.6-A] 393 retiré de la liste
        "sections": ["القوة الملزمة للعقد","الفسخ القضائي وأثاره"],
        "main_chapter": "آثار الالتزامات التعاقدية",
        "boost": 0.14,
    },
    "البطلان": {
        "synonyms": ["بطلان العقد","أسباب البطلان","الإبطال","البطلان النسبي",
                     "البطلان المطلق","عيوب الرضى","الغلط","التدليس","الغرر",
                     "الإكراه","عدم المشروعية"],
        "articles": [60,61,65,66,67,109,110],
        "sections": ["أسباب البطلان (عيوب الرضى)","أسباب البطلان"],
        "main_chapter": "العقود والالتزامات التعاقدية — البطلان والفسخ",
        "boost": 0.16,
    },
    # [v6.6-C] الكسب غير المشروع : ajout 99/100 (responsabilité du fait d'autrui)
    "الكسب غير المشروع": {
        "synonyms": ["الإثراء بلا سبب","الفعل الضار","شبه الجرائم",
                     "المسؤولية التقصيرية","الجريمة المدنية",
                     "الفعل غير المشروع","دعوى الاسترداد","شبه العقد"],
        "articles": [77,78,79,98,99,100],
        "sections": ["المسؤولية التقصيرية والضرر"],
        "main_chapter": "التعويض والضرر",
        "boost": 0.20,
    },
    "الإعذار": {
        "synonyms": ["إعذار","التنبيه","الإخطار","الإنذار","مهلة التنفيذ",
                     "الفسخ القضائي","شروط الفسخ","عدم التنفيذ"],
        "articles": [255,256,259,261],
        "sections": ["الفسخ"],
        "main_chapter": "العقود والالتزامات التعاقدية — البطلان والفسخ",
        "boost": 0.20,
    },
    "إجارة الخدمة": {
        "synonyms": ["عقد عمل","عقد العمل","أجير","رب العمل","صاحب العمل",
                     "إجارة الصنعة","مدة التجربة","خدماته الشخصية"],
        "articles": [723,724,727,728,730,731],
        "sections": ["تعريف العقد وشروطه وإثباته"],
        "main_chapter": "عقد إجارة الخدمة أو العمل",
        "boost": 0.18,
    },
    # [v6.6-D] Nouveau topic : formation/conditions du contrat
    # Cible : récupérer الفصل 2, 19, 39, 57 pour les requêtes
    # "أركان العقد", "شروط صحة العقد", "شروط انعقاد العقد"
    "شروط العقد": {
        "synonyms": ["أركان العقد","شروط الصحة","شروط انعقاد العقد",
                     "صحة العقد","التراضي","محل العقد","سبب العقد",
                     "الإيجاب والقبول","انعقاد العقد"],
        "articles": [2,19,39,57,58,62,63],
        "sections": ["الأهلية والتراضي","شروط انعقاد العقد"],
        "main_chapter": "العقود والالتزامات التعاقدية — أحكام عامة",
        "boost": 0.16,
    },
}

# ══════════════════════════════════════════════════════════════
# 4. DOCTRINAL_CORE, FOUNDATIONAL_BOOST, etc.
# [v6.6-A] 393 : boost réduit de 0.06 → 0.02 dans الالتزامات التعاقدية
# [v6.6-B] 1120 ajouté à الكفالة ; 2 ajouté à الأهلية ; 39 ajouté
# [v6.6-C] 99/100 ajoutés à الكسب غير المشروع
# [v6.6-D] شروط العقد : nouveau core
# ══════════════════════════════════════════════════════════════
DOCTRINAL_CORE: dict[str, dict[str, float]] = {
    "البيع":                 {"478": 0.12, "488": 0.08, "502": 0.06},
    "الفسخ":                 {"259": 0.20, "260": 0.18, "261": 0.15,
                              "262": 0.10, "263": 0.08, "230": 0.06},
    "المسؤولية التقصيرية": {"77":  0.16, "78":  0.14, "79":  0.14, "98": 0.10},
    "التعويض":               {"264": 0.10, "265": 0.07, "77":  0.06},
    # [v6.6-B] ajout article 2
    "الأهلية":               {"2":   0.10, "4":   0.12, "12":  0.10,
                               "9":   0.06, "5":   0.08},
    # [v6.6-B] ajout 1120
    "الكفالة":               {"1117":0.18, "1118":0.16, "1119":0.10,
                               "1120":0.08, "1121":0.08, "1145":0.10},
    "الإفلاس":               {"1247":0.12, "1248":0.10, "1249":0.05},
    "الوكالة":               {"879": 0.10, "880": 0.07, "893": 0.10},
    "الوفاء":                {"320": 0.12, "321": 0.10, "322": 0.08},
    # [v6.6-A] 393 réduit à 0.02 (quasi-inactif sauf requête explicite)
    "الالتزامات التعاقدية": {"230": 0.12, "18":  0.08, "393": 0.02, "228": 0.08},
    "الإيجار":               {"617": 0.14, "618": 0.10, "663": 0.10,
                              "664": 0.10, "668": 0.10},
    "البطلان":               {"60": 0.16, "65": 0.12, "66": 0.10, "67": 0.10},
    # [v6.6-C] ajout 99/100
    "الكسب غير المشروع":     {"77": 0.20, "78": 0.16, "79": 0.16,
                               "98": 0.12, "99": 0.08, "100": 0.06},
    "الإعذار":               {"255": 0.20, "256": 0.16, "261": 0.12, "259": 0.10},
    "إجارة الخدمة":          {"727": 0.20, "723": 0.14, "730": 0.12, "728": 0.08},
    # [v6.6-D] nouveau
    "شروط العقد":            {"2":   0.14, "19":  0.12, "39":  0.10,
                               "57":  0.08, "62":  0.06},
}

CORE_ARTICLE_MULTIPLIER = 1.35
FOUNDATIONAL_BOOST: dict[str, float] = {
    "230": 0.10, "77":  0.08, "259": 0.08, "478": 0.06,
    "320": 0.06, "1117":0.06, "1247":0.06, "4":   0.06,
    "617": 0.05, "60":  0.05,
    "255": 0.06, "256": 0.05,
    "78":  0.05, "79":  0.05,
    # [v6.6-D] 2 et 19 fondamentaux pour tout le droit des contrats
    "2":   0.04, "19":  0.04,
}

CHAPTER_DISTANCE_PENALTY = 0.92
BOOST_GLOBAL_CAP = 0.55

HARD_DOCTRINAL_PRIORITY: dict[str, list[int]] = {
    "الفسخ":                 [259, 260, 261, 262, 230],
    # [v6.6-B] 1120 ajouté
    "الكفالة":               [1117, 1118, 1119, 1120, 1145],
    "المسؤولية التقصيرية": [77, 78, 79, 98],
    # [v6.6-B] 2 ajouté en tête
    "الأهلية":               [2, 4, 12, 9, 5],
    "البيع":                 [478, 488, 502],
    "الوفاء":                [320, 321, 322],
    "التعويض":               [264, 265, 77],
    "الإفلاس":               [1247, 1248, 1249],
    "الوكالة":               [879, 880, 893],
    # [v6.6-A] 393 retiré (faux positif) → remplacé par 229
    "الالتزامات التعاقدية": [230, 18, 228, 229],
    "الإيجار":               [617, 618, 663, 664, 668],
    "البطلان":               [60, 65, 66, 67],
    # [v6.6-C] 99/100 ajoutés
    "الكسب غير المشروع":     [77, 78, 79, 98, 99, 100],
    "الإعذار":               [255, 256, 261, 259],
    "إجارة الخدمة":          [727, 723, 730, 728],
    # [v6.6-D] nouveau
    "شروط العقد":            [2, 19, 39, 57, 62],
}

# ══════════════════════════════════════════════════════════════
# [v6.6-A] FILTRE NÉGATIF POUR الفصل 393
# 393 ne doit apparaître que pour des requêtes
# explicitement sur l'extinction ou la confusion des dettes.
# ══════════════════════════════════════════════════════════════
_ART_393_ALLOWED_KEYWORDS = {
    "انقضاء", "انقضاء الالتزام", "الإبراء", "المقاصة",
    "اتحاد الذمة", "التقادم", "إنقضاء",
}

def _should_penalize_393(query: str) -> bool:
    """Retourne True si 393 est hors-sujet pour cette requête."""
    return not any(kw in query for kw in _ART_393_ALLOWED_KEYWORDS)

# ══════════════════════════════════════════════════════════════
# [v6.6-E] PONDÉRATION CHAPITRE PRINCIPAL POUR REQUÊTES MULTI-CONCEPT
# Pour "مسؤولية الموكل عن تصرفات الوكيل", le chapitre وكالة
# doit dominer sur المسؤولية التقصيرية.
# ══════════════════════════════════════════════════════════════
_MULTI_CONCEPT_PRIMARY_CHAPTER: dict[str, tuple[str, float]] = {
    "مسؤولية الموكل":   ("عقد الوكالة — أحكام عامة", 0.10),
    "تصرفات الوكيل":   ("عقد الوكالة — أحكام عامة", 0.10),
    "فسخ عقد الإيجار": ("عقد الكراء — أحكام عامة", 0.10),
    "فسخ عقد البيع":   ("عقد البيع — أنواع خاصة", 0.10),
}

def _multi_concept_chapter_boost(query: str, chapter: str) -> float:
    for trigger, (target_chapter, bonus) in _MULTI_CONCEPT_PRIMARY_CHAPTER.items():
        if trigger in query and chapter == target_chapter:
            return bonus
    return 0.0

# ══════════════════════════════════════════════════════════════
# [FIX-D4] INTENT DE RÉDACTION + SOCLE DROIT COMMUN
# ══════════════════════════════════════════════════════════════
CONTRACT_DRAFTING_KEYWORDS = ["حرّر", "حرر", "صياغة", "اكتب عقد", "تحرير عقد"]

def is_drafting_intent(query: str) -> bool:
    return any(kw in query for kw in CONTRACT_DRAFTING_KEYWORDS)

GENERAL_CONTRACT_DRAFTING_CORE: dict[int, float] = {
    2:   0.10,
    19:  0.10,
    230: 0.16,
    228: 0.08,
    229: 0.08,
    263: 0.08,
    264: 0.10,
}

# [FIX-D7] DELTA DE FORMATION
DRAFTING_FORMATION_DELTA: dict[str, dict[int, float]] = {
    "البيع":   {488: 0.08, 478: 0.03, 502: -0.04, 503: -0.03},
    "الإيجار": {619: 0.06, 617: 0.03, 663: -0.02},
}

# ──────────── _al_variants + classify_query_v2 ────────────
def _al_variants(phrase: str) -> list:
    words = phrase.split()
    stripped = [w[2:] if w.startswith("ال") and len(w) > 4 else w for w in words]
    return list({phrase, " ".join(stripped)})

def classify_query_v2(query: str) -> dict:
    detected = {}
    normalized = normalize_arabic_legal(query)
    for topic, entry in LEGAL_ONTOLOGY.items():
        for trigger in [topic] + entry["synonyms"]:
            if any(v in query or v in normalized for v in _al_variants(trigger)):
                detected[topic] = entry
                break
    return detected

def get_article_boosts_v2(query: str) -> dict:
    boosts: dict = {}
    detected = classify_query_v2(query)
    for topic, entry in detected.items():
        for art in entry["articles"]:
            boosts[art] = max(boosts.get(art, 0), entry["boost"])
    for topic in detected:
        if topic in DOCTRINAL_CORE:
            for art_str, val in DOCTRINAL_CORE[topic].items():
                try:
                    art_int = int(art_str)
                    boosts[art_int] = max(boosts.get(art_int, 0), val)
                except ValueError:
                    pass

    # [FIX-D1] Garde de pertinence FOUNDATIONAL_BOOST
    for art_str, val in FOUNDATIONAL_BOOST.items():
        try:
            art_int = int(art_str)
        except ValueError:
            continue
        if art_int in boosts:
            boosts[art_int] = max(boosts[art_int], val)

    for art, b in classify_query(query).items():
        boosts[art] = min(BOOST_GLOBAL_CAP, boosts.get(art, 0) + b * 0.3)

    for art_int in list(boosts.keys()):
        prio = get_article_priority(query, str(art_int))
        if prio != 0:
            boosts[art_int] = boosts.get(art_int, 0) + prio

    # [v6.6-A] Annuler le boost de 393 si requête hors-sujet
    if 393 in boosts and _should_penalize_393(query):
        del boosts[393]
        print("  🚫 [v6.6-A] الفصل 393 supprimé des boosts (hors-sujet pour cette requête)")

    for art_int in list(boosts.keys()):
        boosts[art_int] = round(min(BOOST_GLOBAL_CAP, boosts[art_int]), 6)

    return boosts

# ══════════════════════════════════════════════════════════════
# 5. EXPANSION V2
# ══════════════════════════════════════════════════════════════
DOCTRINAL_HIERARCHY_EXPANSION: dict[str, list[str]] = {
    "الفسخ": ["حل العقد","انحلال الالتزام","عدم التنفيذ","الشرط الفاسخ","الفسخ القضائي"],
    "الالتزامات التعاقدية": ["القوة الملزمة","حسن النية","تنفيذ العقد","المدين","الدائن","آثار العقد","عدم التنفيذ"],
    "المسؤولية التقصيرية": ["الفعل الضار","الخطأ","العلاقة السببية","الضرر المباشر","التعويض","شبه الجريمة"],
    "الكفالة": ["الكفيل","الضامن","الرجوع","التضامن","المدين","حق الرجوع","الضمان الشخصي"],
    "الوفاء": ["أداء الالتزام","انقضاء الالتزام","إيفاء الدين","تنفيذ العقد","براءة الذمة"],
    "الإيجار": ["المكري","المكتري","تسليم العين","الصيانة","الأجرة","التزامات المستأجر"],
    "البطلان": ["عيوب الرضى","الغلط","التدليس","الغرر","الإكراه"],
    "الكسب غير المشروع": ["الإثراء بلا سبب","الفعل الضار","الخطأ","الضرر","المسؤولية","شبه الجريمة","دعوى الاسترداد"],
    "الإعذار": ["إعذار","التنبيه","الإنذار","مهلة التنفيذ","الفسخ القضائي","عدم التنفيذ","شروط الفسخ"],
    # [v6.6-D]
    "شروط العقد": ["التراضي","محل العقد","سبب العقد","الإيجاب والقبول","انعقاد العقد"],
}

def expand_query_v2(query: str) -> str:
    base = normalize_arabic_legal(query)
    detected = classify_query_v2(query)
    if not detected:
        return base
    extra = []
    for topic, entry in detected.items():
        extra.extend(entry["synonyms"][:6])
        if entry.get("boost", 0) >= 0.16:
            extra.extend(entry["sections"][:1])
    extra = list(dict.fromkeys(extra))[:12]
    return (base + " " + " ".join(extra)) if extra else base

# ══════════════════════════════════════════════════════════════
# 6. FONCTIONS UTILITAIRES
# ══════════════════════════════════════════════════════════════
def _get_section_local_index(article_num: int) -> int:
    for entry in LOC_HIERARCHY:
        if article_num in entry["articles"]:
            art_list = list(entry["articles"])
            try:
                return art_list.index(article_num)
            except ValueError:
                return 0
    return 0

def section_position_boost(article_num: int) -> float:
    MAX_POS_BOOST = 0.08
    idx = _get_section_local_index(article_num)
    raw = 1.0 / math.log2(2 + idx)
    return round(min(raw * MAX_POS_BOOST, MAX_POS_BOOST), 5)

def chapter_distance_penalty(article_num: int, detected_topics: dict) -> float:
    if not detected_topics:
        return 1.0
    h = get_full_hierarchy(article_num)
    art_chapter = h.get("chapter", "")
    for topic, entry in detected_topics.items():
        main_ch = entry.get("main_chapter", "")
        if main_ch and art_chapter == main_ch:
            return 1.0
    return CHAPTER_DISTANCE_PENALTY

def is_doctrinal_core(article_str: str, detected_topics: dict) -> bool:
    for topic in detected_topics:
        if topic in DOCTRINAL_CORE and article_str in DOCTRINAL_CORE[topic]:
            return True
    return False

def apply_core_multiplier(score: float, article_str: str, detected_topics: dict) -> float:
    if is_doctrinal_core(article_str, detected_topics):
        return score * CORE_ARTICLE_MULTIPLIER
    return score

MIN_KEPT_AFTER_DOMAIN_FILTER = 10

def filter_by_domain(results: list, detected_topics: dict) -> list:
    if not detected_topics:
        return results
    boosted = [
        r for r in results
        if r.get("boost_v2", 0) > 0
        or r.get("core_multiplied")
        or r.get("_injected")
        or r.get("propagated")
    ]
    if len(boosted) >= MIN_KEPT_AFTER_DOMAIN_FILTER:
        return boosted
    return results

# ══════════════════════════════════════════════════════════════
# 7. FILTRE NÉGATIF V2 + [v6.6-A] pénalité 393
# ══════════════════════════════════════════════════════════════
def apply_negative_filter_v2(query: str, text: str) -> float:
    penalty = 0.0

    art_num_text = extract_article_num_from_text(text)
    if art_num_text in _STRUCTURAL_FP_ARTICLES:
        penalty += _STRUCTURAL_FP_PENALTY
        query_terms = set(tokenize_ar(query))
        text_tokens = set(tokenize_ar(text))
        if query_terms & text_tokens:
            penalty -= _STRUCTURAL_FP_RELIEF
        print(f"  🚧 FP structurel v2 : فصل {art_num_text} (pénalité={penalty:.2f})")

    # [v6.6-A] Pénalité explicite الفصل 393 hors requête انقضاء
    if art_num_text == 393 and _should_penalize_393(query):
        penalty += 0.35
        print(f"  🚫 [v6.6-A] الفصل 393 pénalisé (hors-sujet, pénalité=0.35)")

    if "فسخ" in query and "بطلان" in text and not ("فسخ" in text or "حل" in text):
        penalty += 0.4
    if "كفالة" in query and "ضمان" in text and not ("كفيل" in text or "كفالة" in text):
        penalty += 0.3
    if "فسخ" in query:
        art_num = art_num_text
        if 109 <= art_num <= 160 and "فسخ" not in text and "انحلال" not in text:
            penalty += 0.5
        if art_num == 235 and "فسخ" not in text:
            penalty += 0.3
        if 150 <= art_num <= 155 and "فسخ" not in text:
            penalty += 0.4
    if "كفالة" in query:
        art_num = art_num_text
        if 1170 <= art_num <= 1220:
            penalty += 0.6
        if 856 <= art_num <= 879:
            if "كفيل" not in text and "ضامن" not in text:
                penalty += 0.4
        if 478 <= art_num <= 619 and "كفالة" not in text:
            penalty += 0.3
    if ("إيجار" in query or "كراء" in query):
        art_num = art_num_text
        if art_num in (1247, 1117, 478):
            penalty += 0.4
    if "بطلان" in query:
        art_num = art_num_text
        if art_num in (1247, 1117, 879):
            penalty += 0.4
    if ("الكسب غير المشروع" in query or "شبه العقد" in query):
        art_num = art_num_text
        if art_num > 0 and not (77 <= art_num <= 100):
            if "فعل" not in text and "ضرر" not in text and "مسؤولية" not in text:
                penalty += 0.5

    if penalty <= 0:
        return 0.0
    return round(math.tanh(penalty) * 0.6, 6)

# ══════════════════════════════════════════════════════════════
# 8. PROPAGATION DIRIGÉE
# ══════════════════════════════════════════════════════════════
def directed_doctrinal_propagation(candidates: list, detected_topics: dict, query: str) -> list:
    if not detected_topics:
        return candidates
    cand_dict = {r["article"]: r for r in candidates}
    core_arts = []
    for topic in detected_topics:
        if topic in HARD_DOCTRINAL_PRIORITY:
            core_arts.extend([str(a) for a in HARD_DOCTRINAL_PRIORITY[topic]])
    core_arts = set(core_arts)
    for art_str in core_arts:
        if art_str not in cand_dict:
            continue
        art_int = int(art_str.split("-")[0])
        for neighbor in LEGAL_GRAPH.get(art_int, []):
            n_str = str(neighbor)
            if n_str in cand_dict:
                h_neighbor = get_full_hierarchy(neighbor)
                h_core = get_full_hierarchy(art_int)
                if (h_neighbor.get("chapter") == h_core.get("chapter") or
                    h_neighbor.get("section") == h_core.get("section")):
                    bonus = 0.10
                    current = cand_dict[n_str].get("directed_propagation_bonus", 0.0)
                    if bonus > current:
                        cand_dict[n_str]["hybrid_score"] = round(
                            cand_dict[n_str]["hybrid_score"] + bonus - current, 6)
                        cand_dict[n_str]["directed_propagation_bonus"] = bonus
                        cand_dict[n_str]["propagated"] = True
                        print(f"  🔄 Propagation dirigée : فصل {n_str} "
                              f"(voisin de {art_str}) ← +{bonus:.3f}")
    return list(cand_dict.values())

# ══════════════════════════════════════════════════════════════
# 9. INJECTION DOCTRINAL CORE
# ══════════════════════════════════════════════════════════════
def inject_missing_core_articles(
    result: list,
    detected_topics: dict,
    query: str,
    max_inject: int = 3,
) -> list:
    if not detected_topics:
        return result

    returned_arts = {r["article"] for r in result}
    injected_count = 0

    for topic in detected_topics:
        if injected_count >= max_inject:
            break
        priority_list = HARD_DOCTRINAL_PRIORITY.get(topic, [])
        core_dict = DOCTRINAL_CORE.get(topic, {})
        entry = detected_topics[topic]

        for art_int in priority_list[:5]:      # [:5] au lieu de [:4] pour couvrir 1120/99
            if injected_count >= max_inject:
                break
            art_str = str(art_int)
            if art_str in returned_arts:
                continue

            chunk_list = articles_by_num.get(art_str, [])
            legal_chunks = [
                c for c in chunk_list
                if c.get("source_type", "legal_text") == "legal_text"
            ]
            if not legal_chunks:
                print(f"  ⚠️ Injection impossible : فصل {art_str} absent ou sans chunk legal_text")
                continue

            chunk = legal_chunks[0]
            boost_val = core_dict.get(art_str, entry.get("boost", 0.10))
            inject_score = round(float(boost_val) * 0.85, 6)

            result.append({
                "article":        art_str,
                "text":           fix_ocr(chunk.get("text", "")),
                "hybrid_score":   inject_score,
                "rerank_score":   0.0,
                "bm25_score":     0.0,
                "dense_score":    0.0,
                "chapter":        get_chapter(art_int),
                "subdomain":      get_chapter(art_int),
                "source_type":    "legal_text",
                "chunk":          chunk,
                "chunk_id":       chunk.get("chunk_id", art_str),
                "propagation_bonus": 0.0,
                "_injected":      True,
                "_inject_topic":  topic,
            })
            returned_arts.add(art_str)
            injected_count += 1
            print(f"  💉 Injection doctrinal : فصل {art_str} "
                  f"(topic={topic}, score={inject_score:.4f})")

    return result

# ══════════════════════════════════════════════════════════════
# [FIX-D6] INJECTION DROIT COMMUN CONTRATS
# ══════════════════════════════════════════════════════════════
def inject_missing_general_articles(result: list, query: str, max_inject: int = 4) -> list:
    if not is_drafting_intent(query):
        return result
    returned_arts = {r["article"] for r in result}
    injected = 0
    for art_int, boost_val in sorted(
        GENERAL_CONTRACT_DRAFTING_CORE.items(), key=lambda x: -x[1]
    ):
        if injected >= max_inject:
            break
        art_str = str(art_int)
        if art_str in returned_arts:
            continue
        chunk_list = articles_by_num.get(art_str, [])
        legal_chunks = [c for c in chunk_list
                        if c.get("source_type", "legal_text") == "legal_text"]
        if not legal_chunks:
            continue
        chunk = legal_chunks[0]
        result.append({
            "article":        art_str,
            "text":           fix_ocr(chunk.get("text", "")),
            "hybrid_score":   round(boost_val * 0.85, 6),
            "rerank_score":   0.0, "bm25_score": 0.0, "dense_score": 0.0,
            "chapter":        get_chapter(art_int),
            "subdomain":      get_chapter(art_int),
            "source_type":    "legal_text",
            "chunk":          chunk,
            "chunk_id":       chunk.get("chunk_id", art_str),
            "propagation_bonus": 0.0,
            "_injected":      True,
            "_inject_topic":  "_general_contract_core",
        })
        returned_arts.add(art_str)
        injected += 1
        print(f"  💉 Injection droit commun : فصل {art_str} "
              f"(score={round(boost_val * 0.85, 4)})")
    return result

# ══════════════════════════════════════════════════════════════
# 10. NORMALISATION
# ══════════════════════════════════════════════════════════════
def normalize_scores(results: list) -> list:
    if not results:
        return results
    scores = np.array([r["hybrid_score"] for r in results], dtype=float)
    min_s, max_s = scores.min(), scores.max()
    if max_s - min_s < 1e-6:
        return results
    for r in results:
        r["hybrid_score"] = round(float((r["hybrid_score"] - min_s) / (max_s - min_s)), 6)
    return results

# ══════════════════════════════════════════════════════════════
# 11. RECIPROCAL RANK FUSION
# ══════════════════════════════════════════════════════════════
def reciprocal_rank_fusion(results: list, k: int = 60) -> list:
    if not results:
        return results
    dense_ranked = sorted(results, key=lambda r: r.get("dense_score", 0.0), reverse=True)
    dense_rank   = {r["article"]: i + 1 for i, r in enumerate(dense_ranked)}
    bm25_ranked  = sorted(results, key=lambda r: r.get("bm25_score", 0.0), reverse=True)
    bm25_rank    = {r["article"]: i + 1 for i, r in enumerate(bm25_ranked)}
    for r in results:
        art = r["article"]
        rrf_score = (
            1.0 / (k + dense_rank.get(art, len(results))) +
            1.0 / (k + bm25_rank.get(art, len(results)))
        )
        r["rrf_score"]    = round(rrf_score, 6)
        existing          = r.get("hybrid_score", 0.0)
        r["hybrid_score"] = round(existing * 0.5 + rrf_score * 0.5, 6)
    return results

# ══════════════════════════════════════════════════════════════
# 12. PATCH v6.6 — pipeline principal
# ══════════════════════════════════════════════════════════════
_orig_retrieve_reranked = retriever.retrieve_reranked.__func__

def retrieve_reranked_v2(self, query, top_k=5, retrieval_k=100, chroma_filter=None):
    detected = classify_query_v2(query)
    expanded = expand_query_v2(query) if detected else query
    rk = max(retrieval_k, 300)

    result = _orig_retrieve_reranked(
        self, query, top_k=max(top_k, 5), retrieval_k=rk,
        chroma_filter=chroma_filter,
        query_for_retrieval=expanded,
    )

    if not result and detected:
        print("  ⚠️ Pipeline vide — injection de secours uniquement")
        result = inject_missing_core_articles([], detected, query, max_inject=top_k)
        result.sort(key=lambda x: x["hybrid_score"], reverse=True)
        result = result[:top_k]
        return normalize_scores(result)

    if not result:
        return []

    result = reciprocal_rank_fusion(result)

    if detected or is_drafting_intent(query):
        boosts = get_article_boosts_v2(query) if detected else {}

        if is_drafting_intent(query):
            for art_int, val in GENERAL_CONTRACT_DRAFTING_CORE.items():
                boosts[art_int] = max(boosts.get(art_int, 0), val)

        for r in result:
            try:
                a = int(r["article"].split("-")[0])
            except Exception:
                a = 0
            art_str = str(a)

            extra_neg = apply_negative_filter_v2(query, r.get("text", ""))
            if extra_neg:
                r["hybrid_score"] = round(r["hybrid_score"] - extra_neg, 6)
                r["neg_penalty_v2"] = round(extra_neg, 4)

            if a in boosts:
                b = boosts[a]
                new_score = r["hybrid_score"] + b
                if is_doctrinal_core(art_str, detected):
                    new_score = apply_core_multiplier(new_score, art_str, detected)
                    r["core_multiplied"] = True
                r["hybrid_score"] = round(new_score, 6)
                r["boost_v2"] = round(b, 4)

            ch_factor = chapter_distance_penalty(a, detected)
            if ch_factor < 1.0:
                r["hybrid_score"] = round(r["hybrid_score"] * ch_factor, 6)
            r["ch_factor"] = ch_factor

            pos_b = section_position_boost(a)
            r["hybrid_score"] = round(r["hybrid_score"] + pos_b, 6)
            r["pos_boost"] = pos_b

            # [v6.6-E] Boost chapitre principal multi-concept
            chapter_r = r.get("chapter", get_chapter(a))
            mc_boost = _multi_concept_chapter_boost(query, chapter_r)
            if mc_boost > 0:
                r["hybrid_score"] = round(r["hybrid_score"] + mc_boost, 6)
                r["multi_concept_boost"] = mc_boost

            # [FIX-D5] Boost lexical ratio
            query_tokens  = set(tokenize_ar(query))
            text_tok_list = tokenize_ar(r.get("text", ""))
            if text_tok_list and query_tokens:
                overlap_ratio = len(query_tokens & set(text_tok_list)) / len(text_tok_list)
                lexical_bonus = round(min(0.05, overlap_ratio * 0.5), 4)
            else:
                lexical_bonus = 0.0
            r["hybrid_score"]  = round(r["hybrid_score"] + lexical_bonus, 6)
            r["lexical_bonus"] = lexical_bonus

        before_n = len(result)
        result = filter_by_domain(result, detected)
        if len(result) < before_n:
            print(f"  🧹 filter_by_domain : {before_n} → {len(result)} candidats")

        result = inject_missing_core_articles(result, detected, query, max_inject=3)
        result = inject_missing_general_articles(result, query, max_inject=4)

        for r in result:
            if not r.get("_injected"):
                continue
            try:
                a = int(r["article"].split("-")[0])
            except Exception:
                continue
            art_str = str(a)
            if a in boosts:
                b = boosts[a]
                new_score = r["hybrid_score"] + b * 0.5
                if is_doctrinal_core(art_str, detected):
                    new_score = apply_core_multiplier(new_score, art_str, detected)
                    r["core_multiplied"] = True
                r["hybrid_score"] = round(new_score, 6)
                r["boost_v2"] = round(b * 0.5, 4)
            pos_b = section_position_boost(a)
            r["hybrid_score"] = round(r["hybrid_score"] + pos_b, 6)

        for r in result:
            if r.get("_injected") and r["hybrid_score"] > 0.25:
                r["hybrid_score"] = round(r["hybrid_score"] * 0.9, 6)

        result = directed_doctrinal_propagation(result, detected, query)

        for r in result:
            try:
                a = int(r["article"].split("-")[0])
            except Exception:
                a = 0
            r["is_core"] = is_doctrinal_core(str(a), detected)
            domain_factor_final = _contract_domain_penalty(query, r["article"])
            if domain_factor_final < 1.0:
                r["hybrid_score"] = round(r["hybrid_score"] * domain_factor_final, 6)

        if is_drafting_intent(query):
            for topic in detected:
                for r in result:
                    try:
                        a = int(r["article"].split("-")[0])
                    except Exception:
                        continue
                    delta = DRAFTING_FORMATION_DELTA.get(topic, {}).get(a)
                    if delta:
                        r["hybrid_score"] = round(r["hybrid_score"] + delta, 6)
                        r["drafting_intent_delta"] = delta

        result.sort(key=lambda x: x["hybrid_score"], reverse=True)
        result = result[:top_k]

        core_arts = [r["article"] for r in result if r.get("is_core")]
        if core_arts:
            print(f"  📌 Articles core présents dans le top-{top_k} : {core_arts}")
    else:
        result.sort(key=lambda x: x["hybrid_score"], reverse=True)
        result = result[:top_k]

    result = normalize_scores(result)
    return result

retriever.retrieve_reranked = MethodType(retrieve_reranked_v2, retriever)
print("✅ PATCH v6.6 appliqué")
print("   [v6.6-A] الفصل 393 faux positif : boost annulé + pénalité filtre négatif")
print("   [v6.6-B] Articles 2, 1120 ajoutés aux cibles d'injection")
print("   [v6.6-C] الكسب غير المشروع : articles 99/100 couverts")
print("   [v6.6-D] Nouveau topic 'شروط العقد' : récupère 2, 19, 39, 57")
print("   [v6.6-E] Requêtes multi-concept : boost chapitre principal")
print("   [v6.6-F] Diagnostic article 1248 (voir message ci-dessus)")

# ══════════════════════════════════════════════════════════════
# 13. ÉVALUATION
# ══════════════════════════════════════════════════════════════
EVAL_SET = [
    {"query": "الأهلية المدنية للقاصر",               "gold": [4, 9, 12]},
    {"query": "عقد البيع والشراء",                     "gold": [478, 488, 502]},
    {"query": "التعويض عن الضرر",                      "gold": [264, 77, 98]},
    {"query": "الالتزامات التعاقدية",                  "gold": [230, 18, 393]},
    {"query": "الإفلاس والتسوية القضائية",             "gold": [1247, 1248]},
    {"query": "فسخ العقد",                             "gold": [259, 260, 261]},
    {"query": "عقد الكفالة والضمان",                   "gold": [1117, 1118]},
    {"query": "الوكالة والتفويض",                      "gold": [879, 880, 893]},
    {"query": "انقضاء الالتزامات بالوفاء",            "gold": [320, 321, 322]},
    {"query": "المسؤولية التقصيرية",                  "gold": [77, 78, 79]},
    {"query": "بطلان العقد وأسبابه",                   "gold": [60, 65, 66, 67]},
    {"query": "عقد الإيجار والتزامات الأطراف",         "gold": [617, 618, 663, 664]},
    {"query": "الإعذار وشروط الفسخ القضائي",           "gold": [255, 256, 261]},
    {"query": "مسؤولية الموكل عن تصرفات الوكيل",       "gold": [879, 880, 893]},
    {"query": "الكسب غير المشروع وشبه العقد",          "gold": [77, 78, 79, 98]},
    {"query": "القوة الملزمة للعقد وأثره بين الطرفين", "gold": [228, 229, 230]},
    {"query": "عقد العمل محدد المدة وحقوق الأجير",     "gold": [723, 727, 730]},
    # [v6.6] Requêtes élargies pour couvrir les articles manquants
    {"query": "أركان العقد وشروط صحته",                "gold": [2, 19, 39, 57]},
    {"query": "عقد الكفالة والتنازل عن حق الدفع",      "gold": [1117, 1118, 1120]},
    {"query": "التسوية القضائية وحقوق الدائنين",       "gold": [1247, 1248]},
]

def evaluate(retriever_obj, eval_set: list, k: int = 3) -> dict:
    recall_scores, mrr_scores, p1_scores = [], [], []
    print(f"\n{'─'*60}")
    print(f"ÉVALUATION v6.6 — Recall@{k} | MRR | Precision@1")
    print("─"*60)
    for item in eval_set:
        query = item["query"]
        gold = set(str(g) for g in item["gold"])
        results = retriever_obj.retrieve_reranked(query, top_k=k, retrieval_k=300)
        returned = [r["article"] for r in results]
        hits = sum(1 for a in returned if a in gold)
        recall = hits / len(gold) if gold else 0
        recall_scores.append(recall)
        mrr = 0.0
        for rank, art in enumerate(returned, 1):
            if art in gold:
                mrr = 1.0 / rank
                break
        mrr_scores.append(mrr)
        p1 = 1.0 if returned and returned[0] in gold else 0.0
        p1_scores.append(p1)
        status = "✅" if p1 == 1.0 else ("⚠️" if mrr > 0 else "❌")
        print(f"\n{status} {query}")
        print(f"   Gold    : {sorted(gold)}")
        print(f"   Retourné: {returned}")
        print(f"   Recall@{k}={recall:.2f}  MRR={mrr:.3f}  P@1={p1:.0f}")
    avg_recall = sum(recall_scores) / len(recall_scores)
    avg_mrr = sum(mrr_scores) / len(mrr_scores)
    avg_p1 = sum(p1_scores) / len(p1_scores)
    print(f"\n{'━'*60}")
    print(f"RÉSULTATS GLOBAUX ({len(eval_set)} requêtes, K={k})")
    print(f"  Recall@{k}   : {avg_recall:.3f}")
    print(f"  MRR         : {avg_mrr:.3f}")
    print(f"  Precision@1 : {avg_p1:.3f}")
    composite = (0.35*avg_recall + 0.25*avg_mrr + 0.15*avg_p1)
    print(f"  Score composite : {composite:.4f}")
    print("━"*60)
    return {"recall@k": avg_recall, "mrr": avg_mrr, "precision@1": avg_p1}

metrics = evaluate(retriever, EVAL_SET, k=3)
print(f"\n✅ Cellule 8+9 v6.6 terminée — métriques : {metrics}")

✅ Dépendances C7 v5.1 vérifiées.
⚠️  [v6.6-F] الفصل 1248 ABSENT du RAG ou sans chunk legal_text (0 chunks total) → l'injection ne pourra pas le récupérer. Corriger l'indexation en amont.
✅ PATCH v6.6 appliqué
   [v6.6-A] الفصل 393 faux positif : boost annulé + pénalité filtre négatif
   [v6.6-B] Articles 2, 1120 ajoutés aux cibles d'injection
   [v6.6-C] الكسب غير المشروع : articles 99/100 couverts
   [v6.6-D] Nouveau topic 'شروط العقد' : récupère 2, 19, 39, 57
   [v6.6-E] Requêtes multi-concept : boost chapitre principal
   [v6.6-F] Diagnostic article 1248 (voir message ci-dessus)

────────────────────────────────────────────────────────────
ÉVALUATION v6.6 — Recall@3 | MRR | Precision@1
────────────────────────────────────────────────────────────
   Boost : فصل 3 (doc=0.883, bge=0.289) ×1.18
  🎚️ Paliers facteur=1.00 : HARD_FLOOR=0.010 TOP_MIN=0.050 DOCTRINAL_MIN=0.040
  🔗 Propagation : فصل 5 ← +0.0814
  🔗 Propagation : فصل 12 ← +0.0814
  🔗 Graphe post-rerank : voisin فصل 9-106 ← فصل

## Cellule 10 — LLM 


In [9]:
!pip install llama-cpp-python

In [10]:
# ══════════════════════════════════════════════════════════════
# CELLULE 10 — LLM Qwen2.5-7B-Instruct (GGUF via llama-cpp)
# ══════════════════════════════════════════════════════════════

import os
import subprocess
import json
from huggingface_hub import hf_hub_download 
from llama_cpp import Llama
from langchain_core.language_models.llms import LLM

# ──────────────────────────────────────────────────────────────
# VÉRIFICATIONS
# ──────────────────────────────────────────────────────────────
assert "BASE_DIR" in globals(), "❌ BASE_DIR non défini — relancez Cellule 2"

MODEL_REPO  = "bartowski/Qwen2.5-7B-Instruct-GGUF"
MODEL_FILE  = "Qwen2.5-7B-Instruct-Q4_K_M.gguf"
MODEL_PATH  = f"{BASE_DIR}/{MODEL_FILE}"
MIN_SIZE_GB = 4.0


# ──────────────────────────────────────────────────────────────
# VÉRIFICATION / TÉLÉCHARGEMENT
# ──────────────────────────────────────────────────────────────
def fichier_valide(path: str, min_gb: float = MIN_SIZE_GB) -> bool:
    if not os.path.exists(path):
        return False
    size_gb = os.path.getsize(path) / (1024**3)
    print(f"  Taille modèle : {size_gb:.2f} GB")
    return size_gb >= min_gb


if not fichier_valide(MODEL_PATH):
    print(f"  Téléchargement {MODEL_FILE} depuis {MODEL_REPO} …")
    if os.path.exists(MODEL_PATH):
        os.remove(MODEL_PATH)
    hf_hub_download(
        repo_id        = MODEL_REPO,
        filename       = MODEL_FILE,
        local_dir      = BASE_DIR,
        force_download = True,
    )
    print(f"  ✅ Téléchargement terminé → {MODEL_PATH}")
else:
    print(f"  ✅ Modèle déjà présent : {MODEL_PATH}")


# ──────────────────────────────────────────────────────────────
# DÉTECTION GPU
# ──────────────────────────────────────────────────────────────
def gpu_ok() -> bool:
    try:
        return subprocess.run(
            ["nvidia-smi"], capture_output=True
        ).returncode == 0
    except FileNotFoundError:
        return False


_gpu_available = gpu_ok()
n_gpu_layers   = -1 if _gpu_available else 0
print(f"  GPU disponible : {_gpu_available} → n_gpu_layers={n_gpu_layers}")


# ──────────────────────────────────────────────────────────────
# FIX-10A : n_ctx = 6144  (was 4096)
#           Les contrats complets peuvent dépasser 3000 tokens.
#           6144 couvre prompt + réponse confortablement.
# ──────────────────────────────────────────────────────────────
try:
    llm = Llama(
        model_path   = MODEL_PATH,
        n_gpu_layers = n_gpu_layers,
        n_ctx        = 6144,        # FIX-10A
        n_batch      = 512,
        verbose      = False,
        seed         = 42,
        chat_format  = "chatml",
    )
    print("  ✅ Qwen2.5-7B chargé (GPU)" if _gpu_available else
          "  ✅ Qwen2.5-7B chargé (CPU)")
except (ValueError, RuntimeError) as e:
    print(f"  ⚠ GPU échoué ({e}) — fallback CPU")
    llm = Llama(
        model_path   = MODEL_PATH,
        n_gpu_layers = 0,
        n_ctx        = 4096,
        n_batch      = 256,
        verbose      = False,
        seed         = 42,
        chat_format  = "chatml",
    )
    print("  ✅ Qwen2.5-7B chargé (CPU fallback)")

# Test rapide de santé
_test_resp = llm.create_chat_completion(
    messages=[{"role": "user", "content": "قل: جاهز"}],
    max_tokens=10,
)
print(f"  Test LLM : {_test_resp['choices'][0]['message']['content']}")


# ──────────────────────────────────────────────────────────────
# WRAPPER LANGCHAIN
# ──────────────────────────────────────────────────────────────
class QwenLangChainLLM(LLM):
    @property
    def _llm_type(self): return "qwen2.5-gguf"

    def _call(self, prompt: str, stop=None, **kwargs) -> str:
        resp = llm.create_chat_completion(
            messages       = [{"role": "user", "content": prompt}],
            max_tokens     = kwargs.get("max_tokens", 2000),   # FIX-10B
            temperature    = kwargs.get("temperature", 0.05),
            repeat_penalty = 1.1,
            stop           = stop or [],
        )
        return resp["choices"][0]["message"]["content"].strip()


qwen_lc = QwenLangChainLLM()


# ──────────────────────────────────────────────────────────────
# FIX-10B : max_tokens par défaut = 2000
#           Les contrats peuvent nécessiter 1500+ tokens.
# ──────────────────────────────────────────────────────────────
def _llm_call(prompt: str, max_tokens: int = 2000,    # FIX-10B (was 800)
              temperature: float = 0.05) -> str:
    """Appel direct au LLM — retourne le texte généré."""
    resp = llm.create_chat_completion(
        messages       = [{"role": "user", "content": prompt}],
        max_tokens     = max_tokens,
        temperature    = temperature,
        repeat_penalty = 1.1,
    )
    return resp["choices"][0]["message"]["content"].strip()


def _parse_json(text: str, fallback: dict) -> dict:
    """Extrait le premier objet JSON valide d'une réponse LLM."""
    s = text.find('{')
    e = text.rfind('}')
    if s == -1 or e == -1:
        return fallback
    try:
        return json.loads(text[s:e+1])
    except json.JSONDecodeError:
        return fallback


# ══════════════════════════════════════════════════════════════
# TESTS RAPIDES
# ══════════════════════════════════════════════════════════════
test_queries = [
    "الأهلية المدنية للقاصر",
]

_PROMPT_TEMPLATE = """\
أنت مساعد قانوني متخصص في القانون المغربي.
أجب على السؤال التالي بإيجاز وبدقة في 3 إلى 5 جمل باللغة العربية.

السؤال: {question}
"""

print("━"*60)
print("TESTS DE RÉPONSE LLM")
print("━"*60)

for q in test_queries:
    print(f"\n🔍 Requête : {q}")
    print("─"*50)
    try:
        prompt   = _PROMPT_TEMPLATE.format(question=q)
        response = _llm_call(prompt, max_tokens=300, temperature=0.1)
        for line in response.strip().splitlines():
            if line.strip():
                print(f"  {line.strip()}")
    except Exception as e:
        print(f"  ❌ Erreur : {e}")

print("\n" + "━"*60)
print("✅ Tests LLM terminés.")
print("━"*60)

print("\n✅ Cellule 10 terminée")
print("   Variables exportées : llm, qwen_lc")
print("   Fonctions exportées : _llm_call (max_tokens=2000), _parse_json")

  Taille modèle : 4.36 GB
  ✅ Modèle déjà présent : /kaggle/working/Qwen2.5-7B-Instruct-Q4_K_M.gguf
  GPU disponible : True → n_gpu_layers=-1


llama_context: n_ctx_seq (6144) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


  ✅ Qwen2.5-7B chargé (GPU)
  Test LLM : جاهز
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
TESTS DE RÉPONSE LLM
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🔍 Requête : الأهلية المدنية للقاصر
──────────────────────────────────────────────────
  الأهلية المدنية للقاصر في المغرب تبدأ من سن السابعة إلى سن البلوغ، وهي تبلغ 18 سنة. خلال هذه الفترة، القاصر يعتبر قاصر الأهلية، حيث يمكنه أداء بعض المعاملات القانونية بصفة مستقلة، بينما يبقى البعض الآخر يتطلب موافقة الوصي أو الحاضن.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
✅ Tests LLM terminés.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

✅ Cellule 10 terminée
   Variables exportées : llm, qwen_lc
   Fonctions exportées : _llm_call (max_tokens=2000), _parse_json


## Cellule 11 — LangGraph StateGraph


In [11]:
# ══════════════════════════════════════════════════════════════
# CELLULE 11 — LangGraph v11  (extraction LLM+Regex + collecte adaptative)
# ══════════════════════════════════════════════════════════════

import json, re, time, operator
from typing import List, Annotated, TypedDict
from langgraph.graph import StateGraph, END

# ──────────────────────────────────────────────────────────────
# VÉRIFICATIONS
# ──────────────────────────────────────────────────────────────
for varname in ["retriever", "_llm_call", "_parse_json",
                "articles_by_num", "AGENT_LOG", "TOP_K_RETRIEVAL",
                "rerank", "reranker"]:
    assert varname in globals(), \
        f"❌ {varname} non défini — relancez les cellules précédentes"

print("━"*60)
print("CELLULE 11 — LangGraph v11  (JSON output réorganisé, top-3)")
print("━"*60)

# ══════════════════════════════════════════════════════════════
# CONSTANTES
# ══════════════════════════════════════════════════════════════
RRF_MIN_SCORE    = 0.012
RERANK_MIN_SCORE = 0.20
MAX_SELECTED     = 3
TOP_K_BY_TYPE = {
    "factuelle":       4,
    "interprétative":  6,
    "comparative":     8,
    "procédurale":     6,
}
# [FIX] nombre d'articles réellement affichés dans le JSON final,
# quel que soit le nombre interne sélectionné (contrat ou non).
TOP_N_ARTICLES_DISPLAY = 3

# ══════════════════════════════════════════════════════════════
# OCR → vrais numéros d'articles LOC
# ══════════════════════════════════════════════════════════════
OCR_TO_TRUE_ART = {"617": "627", "618": "628", "619": "629"}
TRUE_TO_OCR_ART = {v: k for k, v in OCR_TO_TRUE_ART.items()}

def true_art_num(ocr_num: str) -> str:
    return OCR_TO_TRUE_ART.get(str(ocr_num), str(ocr_num))

def ocr_art_num(true_num: str) -> str:
    return TRUE_TO_OCR_ART.get(str(true_num), str(true_num))

_canon_tests = [("618", "628"), ("619", "629"), ("617", "627")]
for _ocr, _true in _canon_tests:
    assert true_art_num(_ocr) == _true
print("✅ Canonicalisation articles OCR→LOC vérifiée : 618→628 | 619→629 | 617→627")

# ══════════════════════════════════════════════════════════════
# CATALOGUE DES CONTRATS
# ══════════════════════════════════════════════════════════════
CONTRACT_CATALOGUE = {
    "إيجار": {
        "type_ar": "إيجار", "type_fr": "bail / location",
        "keywords": ["إيجار","كراء","مؤجر","مستأجر","شقة","منزل","محل",
                     "دار","عقار","bail","location","louer","loyer","locataire"],
        "articles": ["617","618","619","633","634","635","636","638","639",
                     "663","664","666","668","687","689"],
    },
    "بيع": {
        "type_ar": "بيع", "type_fr": "vente",
        "keywords": ["بيع","شراء","مشتري","بائع","ثمن","مبيع",
                     "vente","achat","acheteur","vendeur","prix"],
        "articles": ["478","479","481","482","483","488","489",
                     "502","503","504","505","532","534"],
    },
    "قرض": {
        "type_ar": "قرض", "type_fr": "prêt",
        "keywords": ["قرض","سلف","اقتراض","مقترض","مقرض","دين","prêt","emprunt"],
        "articles": ["856","857","858","859","860","861","870","871","872","875"],
    },
    "وكالة": {
        "type_ar": "وكالة", "type_fr": "mandat",
        "keywords": ["وكالة","وكيل","موكل","تفويض","نيابة",
                     "mandat","mandataire","procuration"],
        "articles": ["879","880","881","882","893","894","922","929","930","931","932"],
    },
    "كفالة": {
        "type_ar": "كفالة", "type_fr": "cautionnement",
        "keywords": ["كفالة","كفيل","ضمان","ضامن","caution","cautionnement","garant"],
        "articles": ["1117","1118","1119","1120","1121","1125","1128","1129","1145","1146"],
    },
    "شركة": {
        "type_ar": "شركة", "type_fr": "société",
        "keywords": ["شركة","شريك","حصص","رأس المال","تأسيس شركة",
                     "société","associé","parts","capital"],
        "articles": ["982","983","984","985","986","987","1010","1013","1019"],
    },
    "عمل": {
        "type_ar": "عمل", "type_fr": "louage d'ouvrage",
        "keywords": ["عمل","أجير","رب العمل","أجر","خدمة","travail","louage","ouvrage",
                     "شغل","عقد شغل","عقد العمل","موظف","مستخدم","أجرة"],
        "articles": ["723","724","725","726","727","728","730","731","769","770"],
    },
    "هبة": {
        "type_ar": "هبة", "type_fr": "donation",
        "keywords": ["هبة","تبرع","واهب","موهوب","منحة","don","donation","donataire"],
        "articles": ["826","827","828","829","831","843","845","846"],
    },
}

ARTICLES_GENERAUX_CONTRAT = ["230", "18", "19", "20"]

# ══════════════════════════════════════════════════════════════
# CHAMPS OBLIGATOIRES PAR TYPE DE CONTRAT
# ══════════════════════════════════════════════════════════════
REQUIRED_FIELDS = {
    "إيجار": [
        ("مؤجر",       "الاسم الكامل للمؤجر (صاحب العقار)",               False),
        ("مستأجر",     "الاسم الكامل للمستأجر",                            False),
        ("عنوان",      "عنوان العين المكتراة (المدينة، الحي، الشارع)",     False),
        ("أجرة",       "قيمة الأجرة الشهرية مع العملة (مثال: 3000 درهم)", False),
        ("مدة",        "مدة الإيجار (مثال: سنة واحدة، 6 أشهر، سنتان)",    False),
        ("تاريخ_بدء",  "تاريخ بداية الإيجار (مثال: 01/01/2025)",          False),
        ("ضمان",       "مبلغ التأمين / الضمان (اكتب 'لا' إن لم يوجد)",    True),
        ("شرط_الباطن", "هل يُسمح بالإكراء من الباطن؟ (نعم / لا)",          True),
    ],
    "بيع": [
        ("بائع",        "الاسم الكامل للبائع",                              False),
        ("مشتري",       "الاسم الكامل للمشتري",                             False),
        ("وصف_المبيع",  "وصف الشيء المبيع (العقار، السيارة، البضاعة...)",  False),
        ("ثمن",         "ثمن البيع مع العملة (مثال: 500000 درهم)",          False),
        ("طريقة_الدفع", "طريقة الدفع (نقداً، أقساط، تحويل بنكي...)",       False),
        ("تاريخ_البيع", "تاريخ إبرام عقد البيع",                            False),
        ("ضمان_العيوب", "هل يوجد ضمان عيوب خفية؟ (نعم / لا)",              True),
    ],
    "قرض": [
        ("مقرض",         "الاسم الكامل للمُقرض",                            False),
        ("مقترض",        "الاسم الكامل للمُقترض",                           False),
        ("مبلغ_القرض",   "مبلغ القرض مع العملة (مثال: 100000 درهم)",        False),
        ("فائدة",        "نسبة الفائدة السنوية (أو 'بدون فائدة')",           False),
        ("مدة_السداد",   "مدة السداد (مثال: 24 شهراً، 3 سنوات)",            False),
        ("طريقة_السداد", "طريقة السداد (أقساط شهرية / دفعة واحدة)",          False),
        ("ضمانات",       "الضمانات المقدمة (كفيل، رهن، أو 'لا')",           True),
    ],
    "وكالة": [
        ("موكل",           "الاسم الكامل للموكل",                            False),
        ("وكيل",           "الاسم الكامل للوكيل",                            False),
        ("موضوع_الوكالة",  "موضوع الوكالة بالتفصيل (ما المُفوَّض به؟)",     False),
        ("صلاحيات_الوكيل", "صلاحيات الوكيل (التوقيع، البيع، التقاضي...)",   False),
        ("مدة_الوكالة",    "مدة الوكالة (أو 'غير محددة')",                  True),
        ("أتعاب_الوكيل",   "أتعاب الوكيل إن وجدت (أو 'بدون أتعاب')",        True),
    ],
    "كفالة": [
        ("دائن",          "الاسم الكامل للدائن",                             False),
        ("مدين",          "الاسم الكامل للمدين",                             False),
        ("كفيل",          "الاسم الكامل للكفيل",                             False),
        ("مبلغ_الدين",    "مبلغ الدين المكفول مع العملة",                   False),
        ("طبيعة_الكفالة", "طبيعة الكفالة (تضامنية / بسيطة)",               False),
        ("أجل_الكفالة",   "أجل الكفالة (أو 'غير محدد')",                   True),
    ],
    "شركة": [
        ("الشريك_الأول",  "الاسم الكامل للشريك الأول",                      False),
        ("الشريك_الثاني", "الاسم الكامل للشريك الثاني",                     False),
        ("اسم_الشركة",    "اسم الشركة المراد تأسيسها",                      False),
        ("نوع_الشركة",    "نوع الشركة (ذ.م.م / تضامن / مساهمة...)",          False),
        ("رأس_المال",     "رأس مال الشركة مع العملة",                       False),
        ("موضوع_الشركة",  "موضوع / غرض الشركة",                            False),
        ("مقر_الشركة",    "المقر الاجتماعي للشركة",                         False),
        ("حصص_الشركاء",  "توزيع الحصص بين الشركاء (مثال: 50% / 50%)",      False),
    ],
    "عمل": [
        ("رب_العمل",    "اسم رب العمل (الشخص أو الشركة)",                   False),
        ("الأجير",      "الاسم الكامل للأجير",                               False),
        ("المنصب",      "المنصب أو طبيعة العمل المسند",                      False),
        ("الأجر",       "الأجر الشهري الصافي مع العملة",                    False),
        ("تاريخ_البدء", "تاريخ بداية العمل",                                 False),
        ("نوع_العقد",   "نوع العقد (محدد المدة / غير محدد المدة)",          False),
        ("مكان_العمل",  "مكان العمل (المدينة / العنوان)",                    False),
        ("مدة_التجربة", "فترة التجربة إن وجدت (أو 'لا')",                  True),
    ],
    "هبة": [
        ("الواهب",        "الاسم الكامل للواهب",                             False),
        ("الموهوب_له",    "الاسم الكامل للموهوب له",                         False),
        ("الشيء_الموهوب", "وصف الشيء الموهوب بالتفصيل",                    False),
        ("قيمة_الهبة",    "القيمة التقديرية للشيء الموهوب (أو 'غير محددة')",True),
        ("شروط_الهبة",    "هل توجد شروط أو أعباء مقابل الهبة؟ (أو 'لا')",  True),
    ],
}

# ══════════════════════════════════════════════════════════════
# DÉTECTION DU TYPE DE CONTRAT
# ══════════════════════════════════════════════════════════════
def _detect_contract_type(question: str) -> dict | None:
    q_lower = question.lower()
    for cat_key, cat_info in CONTRACT_CATALOGUE.items():
        if any(kw in q_lower for kw in cat_info["keywords"]):
            return cat_info
    return None

def _is_contract_drafting(question: str) -> bool:
    drafting_verbs = [
        "إنشاء عقد","صياغة عقد","أكتب عقد","اكتب عقد","تحرير عقد",
        "أريد عقد","أريد إنشاء","أريد صياغة","أحتاج عقد",
        "اكتب لي","حرر لي","صِغ لي","أريد تحرير",
        "rédiger","établir","créer","contrat de",
        "عقد إيجار","عقد بيع","عقد قرض","عقد وكالة",
        "عقد كفالة","عقد شركة","عقد عمل","عقد هبة",
        "يؤسسان شركة","يُنشئان شركة","أسسا شركة","يؤسس شركة","تأسيس شركة",
        "يساهم كل واحد","يساهمان","شركاء في","شراكة بين",
        "عقد شغل","إنشاء عقد شغل","صياغة عقد شغل",
        "عقد العمل","إنشاء عقد العمل",
    ]
    q_lower = question.lower()
    return any(kw in q_lower for kw in drafting_verbs)

# ══════════════════════════════════════════════════════════════
# EXTRACTEUR LLM
# ══════════════════════════════════════════════════════════════
def _extract_entities_llm(question: str, contract_type_ar: str) -> dict:
    if not contract_type_ar or contract_type_ar not in REQUIRED_FIELDS:
        return {}

    fields = REQUIRED_FIELDS[contract_type_ar]
    fields_list = "\n".join(
        f'  - "{fk}": {fl}  {"[اختياري]" if fo else "[إلزامي]"}'
        for fk, fl, fo in fields
    )

    prompt = f"""\
أنت مساعد قانوني. استخرج من الجملة التالية المعلومات المذكورة صراحةً فقط.
لا تخترع أي معلومة غير موجودة في النص.
إذا لم تُذكر معلومة، لا تُدرجها في النتيجة.

نوع العقد: {contract_type_ar}
الحقول المطلوبة:
{fields_list}

الجملة:
«{question}»

تعليمات إضافية:
- "شغل" = "عمل" → نوع_العقد: "محدد المدة" إذا وردت
- "مطور معلوماتي" أو "مهندس" أو أي مهنة → المنصب
- "أجرة شهرية قدرها X" أو "أجر شهري X" → الأجر
- "لمدة سنة" أو "لمدة X أشهر" → مدة_التجربة أو مدة العقد
- "محدد المدة" → نوع_العقد: "محدد المدة"

أرجع JSON فقط، بدون أي نص إضافي ولا backticks، بالشكل:
{{"المنصب": "مطور معلوماتي", "الأجر": "10000 درهم", "نوع_العقد": "محدد المدة"}}
أو {{}} إذا لم تُذكر أي معلومة.
"""
    try:
        raw = _llm_call(prompt, max_tokens=600)
        print(f"  [LLM-RAW] → {raw[:300]}")
        raw = re.sub(r"```(?:json)?", "", raw).strip().strip("`").strip()
        result = _parse_json(raw, {})
        valid_keys = {fk for fk, _, _ in fields}
        return {
            k: str(v).strip()
            for k, v in result.items()
            if k in valid_keys and v and str(v).strip() not in ("null", "None", "")
        }
    except Exception as e:
        print(f"  ⚠️ _extract_entities_llm erreur : {e}")
        return {}

# ══════════════════════════════════════════════════════════════
# EXTRACTEUR REGEX
# ══════════════════════════════════════════════════════════════
def _extract_entities_regex(question: str) -> dict:
    ents = {}

    _AMOUNT_RE = (
        r"([\d\s\.,]+(?:\.000|,000)*)"
        r"\s*(درهم|دج|جنيه|ريال|دولار|يورو|MAD|DH)?"
    )
    def _clean_amount(s: str) -> str:
        return re.sub(r"[^\d]", "", s)

    # ── شركة ──────────────────────────────────────────────────
    m = re.search(
        r"^([^\s،,]{2,20})\s+و\s*([^\s،,]{2,20})\s+"
        r"(?:يؤسسان|أسسا|أنشأا|يُنشئان|شركاء\s+في)\s+شركة",
        question, re.UNICODE | re.MULTILINE)
    if m:
        ents.setdefault("الشريك_الأول",  m.group(1).strip())
        ents.setdefault("الشريك_الثاني", m.group(2).strip())

    m = re.search(
        r"(?:يؤسس|أسس|ينشئ)\s+كل\s+من\s+"
        r"([^\sو،,]{2,20})\s+و\s*([^\s،,]{2,20})",
        question, re.UNICODE)
    if m:
        ents.setdefault("الشريك_الأول",  m.group(1).strip())
        ents.setdefault("الشريك_الثاني", m.group(2).strip())

    m = re.search(
        r"(?:شراكة|شركة)\s+بين\s+([^\sو،,]{2,30})\s+و\s*([^\s،,.]{2,30})",
        question, re.UNICODE)
    if m:
        ents.setdefault("الشريك_الأول",  m.group(1).strip())
        ents.setdefault("الشريك_الثاني", m.group(2).strip())

    m = re.search(
        r"(?:يساهم|تساهم)\s+كل\s+(?:واحد\s+)?(?:منهما|منهم)\s+"
        r"(?:ب|بمبلغ\s+)" + _AMOUNT_RE,
        question, re.UNICODE | re.IGNORECASE)
    if m:
        try:
            per_partner = int(_clean_amount(m.group(1)))
            total       = per_partner * 2
            currency    = m.group(2) or "درهم"
            ents.setdefault("رأس_المال", f"{total:,} {currency}".replace(",", "."))
        except ValueError:
            pass

    m = re.search(
        r"(?:رأس\s+(?:المال|مال)\s+(?:الشركة\s+)?(?:البالغ\s+)?(?:قدره\s+)?|"
        r"(?:رأسمال|capital)\s+)" + _AMOUNT_RE,
        question, re.UNICODE | re.IGNORECASE)
    if m and "رأس_المال" not in ents:
        ents.setdefault("رأس_المال",
                        f"{_clean_amount(m.group(1))} {m.group(2) or 'درهم'}")

    m = re.search(
        r"\b(ذ\.م\.م|ش\.م\.م|SARL|SA\b|SNC|SCS"
        r"|شركة\s+تضامن|شركة\s+مساهمة|شركة\s+ذات\s+مسؤولية\s+محدودة)\b",
        question, re.UNICODE | re.IGNORECASE)
    if m:
        ents.setdefault("نوع_الشركة", m.group(1).strip())

    m = re.search(
        r"(?:مقرها|المقر\s+الاجتماعي|يقع\s+(?:مقره|مقرها)\s*(?:ب|في)?"
        r"|الكائن(?:ة)?\s+(?:ب|في))\s*[:：]?\s*([^\،,.\n]{3,60})",
        question, re.UNICODE)
    if m:
        ents.setdefault("مقر_الشركة", m.group(1).strip())

    m = re.search(r"بالتساوي", question, re.UNICODE)
    if m:
        ents.setdefault("حصص_الشركاء", "50% / 50%")
    m = re.search(r"(\d+\s*%)\s*[/و]\s*(\d+\s*%)", question, re.UNICODE)
    if m:
        ents.setdefault("حصص_الشركاء",
                        f"{m.group(1).strip()} / {m.group(2).strip()}")

    # ── إيجار ─────────────────────────────────────────────────
    for pat, key in [
        (r"(?:المؤجر|المالك|صاحب\s+العقار)\s*[:：]?\s*"
         r"((?:[^\s،,.\nوب]{2,20}\s?){1,3}?)(?=\s+(?:و|ب|بأجرة|والمستأجر)|[،,.]|$)",
         "مؤجر"),
        (r"(?:المستأجر)\s*[:：]?\s*"
         r"((?:[^\s،,.\nوب]{2,20}\s?){1,3}?)(?=\s+(?:و|ب|بأجرة|بمبلغ)|[،,.]|$)",
         "مستأجر"),
    ]:
        m = re.search(pat, question, re.UNICODE)
        if m: ents.setdefault(key, m.group(1).strip())

    m = re.search(
        r"(?:أجرة|كراء|إيجار(?:\s+شهري)?)\s+(?:قدرها\s+|ب)?" + _AMOUNT_RE,
        question, re.UNICODE | re.IGNORECASE)
    if m:
        ents.setdefault("أجرة",
                        f"{_clean_amount(m.group(1))} {m.group(2) or 'درهم'}")

    m = re.search(
        r"(?:لمدة|مدة)\s+"
        r"(سنة\s+واحدة|سنتين|\d+\s*سنوات?|\d+\s*أشهر?|\d+\s*شهر(?:ًا)?)",
        question, re.UNICODE)
    if m: ents.setdefault("مدة", m.group(1).strip())

    # ── بيع ───────────────────────────────────────────────────
    for pat, key in [
        (r"(?:البائع|بائع)\s*[:：]?\s*([^\،,.\n]{3,40})", "بائع"),
        (r"(?:المشتري|مشتري)\s*[:：]?\s*([^\،,.\n]{3,40})", "مشتري"),
    ]:
        m = re.search(pat, question, re.UNICODE)
        if m: ents.setdefault(key, m.group(1).strip())

    m = re.search(
        r"([^\s،,]{2,20})\s+(?:يبيع|باع|سيبيع)\s+(?:\w+\s+)*(?:ل|إلى)\s+"
        r"([^\s،,.]{2,20})",
        question, re.UNICODE)
    if m:
        ents.setdefault("بائع",  m.group(1).strip())
        ents.setdefault("مشتري", m.group(2).strip())

    m = re.search(
        r"(?:بثمن|ثمن(?:ه)?(?:\s+البيع)?|بسعر|مقابل\s+مبلغ)"
        r"\s*(?:قدره\s+|يبلغ\s+)?" + _AMOUNT_RE,
        question, re.UNICODE | re.IGNORECASE)
    if m:
        ents.setdefault("ثمن",
                        f"{_clean_amount(m.group(1))} {m.group(2) or 'درهم'}")

    for item in ["سيارة","مركبة","شقة","أرض","منزل","دار","محل تجاري",
                 "محل","عقار","قطعة أرض","بضاعة","أثاث","دراجة","حاسوب","هاتف"]:
        if item in question:
            ents.setdefault("وصف_المبيع", item)
            break

    # ── قرض ───────────────────────────────────────────────────
    for pat, key in [
        (r"(?:المقرض|مقرض)\s*[:：]?\s*"
         r"((?:[^\s،,.\nوب]{2,20}\s?){1,3}?)(?=\s+(?:و|ب|بمبلغ|والمقترض)|[،,.]|$)",
         "مقرض"),
        (r"(?:المقترض|مقترض)\s*[:：]?\s*"
         r"((?:[^\s،,.\nوب]{2,20}\s?){1,3}?)(?=\s+(?:و|ب|بمبلغ)|[،,.]|$)",
         "مقترض"),
    ]:
        m = re.search(pat, question, re.UNICODE)
        if m: ents.setdefault(key, m.group(1).strip())

    m = re.search(
        r"(?:مبلغ\s+(?:القرض|السلف)|قرض(?:\s+بمبلغ)?)\s+" + _AMOUNT_RE,
        question, re.UNICODE | re.IGNORECASE)
    if m:
        ents.setdefault("مبلغ_القرض",
                        f"{_clean_amount(m.group(1))} {m.group(2) or 'درهم'}")

    m = re.search(
        r"(?:فائدة|فوائد)\s+(?:بنسبة\s+)?"
        r"([\d,\.]+\s*%|بدون\s+فائدة|بلا\s+فائدة)",
        question, re.UNICODE | re.IGNORECASE)
    if m: ents.setdefault("فائدة", m.group(1).strip())

    m = re.search(
        r"(?:سداد|تسديد|أقساط)\s+(?:لمدة\s+|خلال\s+)?"
        r"(\d+\s*(?:سنة|سنوات|شهر|أشهر|شهراً))",
        question, re.UNICODE)
    if m: ents.setdefault("مدة_السداد", m.group(1).strip())

    # ── وكالة ─────────────────────────────────────────────────
    for pat, key in [
        (r"(?:الموكل|موكل)\s*[:：]?\s*([^\،,.\n]{3,40})", "موكل"),
        (r"(?:الوكيل|وكيل)\s*[:：]?\s*([^\،,.\n]{3,40})", "وكيل"),
    ]:
        m = re.search(pat, question, re.UNICODE)
        if m: ents.setdefault(key, m.group(1).strip())

    m = re.search(
        r"(?:للقيام\s+ب|لأجل\s+|بشأن|موضوع(?:ها)?)\s*[:：]?\s*([^\،,.\n]{5,80})",
        question, re.UNICODE)
    if m: ents.setdefault("موضوع_الوكالة", m.group(1).strip())

    # ── كفالة ─────────────────────────────────────────────────
    for pat, key in [
        (r"(?:الكفيل|كفيل)\s*[:：]?\s*([^\،,.\n]{3,40})", "كفيل"),
        (r"(?:المدين|مدين)\s*[:：]?\s*([^\،,.\n]{3,40})", "مدين"),
        (r"(?:الدائن|دائن)\s*[:：]?\s*([^\،,.\n]{3,40})", "دائن"),
    ]:
        m = re.search(pat, question, re.UNICODE)
        if m: ents.setdefault(key, m.group(1).strip())

    m = re.search(r"كفالة\s+(تضامنية|بسيطة)", question, re.UNICODE)
    if m: ents.setdefault("طبيعة_الكفالة", m.group(1).strip())

    # ── عمل ───────────────────────────────────────────────────
    for pat, key in [
        (r"(?:رب\s+العمل|صاحب\s+العمل|المشغل)\s*[:：]?\s*([^\،,.\n]{3,40})",
         "رب_العمل"),
        (r"(?:الأجير|أجير|الموظف)\s*[:：]?\s*([^\،,.\n]{3,40})", "الأجير"),
    ]:
        m = re.search(pat, question, re.UNICODE)
        if m: ents.setdefault(key, m.group(1).strip())

    m = re.search(
        r"(?:أجر(?:ة)?|راتب)\s+(?:شهري(?:ة)?\s+)?(?:(?:صافي|إجمالي)\s+)?"
        r"(?:قدره?ا?\s+)?" + _AMOUNT_RE,
        question, re.UNICODE | re.IGNORECASE)
    if m:
        ents.setdefault("الأجر",
                        f"{_clean_amount(m.group(1))} {m.group(2) or 'درهم'}")

    m = re.search(
        r"\b(عقد\s+(?:شغل\s+|عمل\s+)?(?:محدد\s+المدة|غير\s+محدد\s+المدة|CDD|CDI))\b",
        question, re.UNICODE | re.IGNORECASE)
    if m:
        ents.setdefault("نوع_العقد", m.group(1).strip())
    else:
        m = re.search(
            r"\b(محدد\s+المدة|غير\s+محدد\s+المدة|CDD|CDI)\b",
            question, re.UNICODE | re.IGNORECASE)
        if m:
            ents.setdefault("نوع_العقد", m.group(1).strip())

    _METIERS = [
        "مطور\s+(?:ويب|معلوماتي|برمجيات|تطبيقات)?",
        "مهندس\s+(?:معلوماتي|برمجيات|مدني|كهربائي|ميكانيكي)?",
        "محاسب\s+(?:قانوني|عام)?",
        "طبيب\s+(?:عام|أسنان|أخصائي)?",
        "مدير\s+(?:مالي|تنفيذي|مبيعات|مشروع)?",
        "أستاذ\s+(?:\w+)?",
        "محامي",
        "تقني\s+(?:\w+)?",
        "إداري(?:\s+\w+)?",
        "مبرمج\s+(?:\w+)?",
        "مصمم\s+(?:جرافيك|ويب|جافيك)?",
        "موظف\s+(?:\w+)?",
        "سكرتير(?:ة)?",
        "عون\s+(?:\w+)?",
        "تقني(?:\s+متخصص)?",
        "مستشار\s+(?:\w+)?",
    ]
    _metier_pattern = r"(?:لـ?\s*|لمنصب\s+|بصفة\s+|كـ?\s*|بوصفه\s+|يشغل\s+(?:منصب\s+)?)(" + "|".join(_METIERS) + ")"
    m = re.search(_metier_pattern, question, re.UNICODE | re.IGNORECASE)
    if m:
        ents.setdefault("المنصب", m.group(1).strip())
    else:
        for metier_re in _METIERS:
            m = re.search(r"\b(" + metier_re + r")\b", question, re.UNICODE | re.IGNORECASE)
            if m:
                ents.setdefault("المنصب", m.group(1).strip())
                break

    m_duree = re.search(
        r"لمدة\s+(سنة\s+واحدة|سنتين|\d+\s*سنوات?|\d+\s*أشهر?|\d+\s*شهر(?:ًا)?)",
        question, re.UNICODE)
    if m_duree:
        if re.search(r"(?:تجربة|اختبار)", question, re.UNICODE):
            ents.setdefault("مدة_التجربة", m_duree.group(1).strip())
        ents.setdefault("مدة", m_duree.group(1).strip())

    # ── هبة ───────────────────────────────────────────────────
    for pat, key in [
        (r"(?:الواهب|واهب)\s*[:：]?\s*([^\،,.\n]{3,40})", "الواهب"),
        (r"(?:الموهوب\s+له|موهوب\s+له)\s*[:：]?\s*([^\،,.\n]{3,40})",
         "الموهوب_له"),
    ]:
        m = re.search(pat, question, re.UNICODE)
        if m: ents.setdefault(key, m.group(1).strip())

    # ── Pattern "بين X وY" générique ─────────────────────────
    _bw = re.search(
        r"بين\s+([^\s،,و]{2,25})\s+و\s*([^\s،,.]{2,25})(?=\s|[،,.]|$)",
        question, re.UNICODE)
    if _bw:
        p1, p2 = _bw.group(1).strip(), _bw.group(2).strip()
        if any(kw in question for kw in ["بيع","شراء"]):
            ents.setdefault("بائع",          p1)
            ents.setdefault("مشتري",         p2)
        elif any(kw in question for kw in ["قرض","سلف"]):
            ents.setdefault("مقرض",          p1)
            ents.setdefault("مقترض",         p2)
        elif any(kw in question for kw in ["وكالة","وكيل"]):
            ents.setdefault("موكل",          p1)
            ents.setdefault("وكيل",          p2)
        elif any(kw in question for kw in ["كفالة","كفيل"]):
            ents.setdefault("دائن",          p1)
            ents.setdefault("مدين",          p2)
        elif any(kw in question for kw in ["إيجار","كراء"]):
            ents.setdefault("مؤجر",          p1)
            ents.setdefault("مستأجر",        p2)
        elif any(kw in question for kw in ["عمل","أجير","شغل"]):
            ents.setdefault("رب_العمل",      p1)
            ents.setdefault("الأجير",        p2)
        elif any(kw in question for kw in ["هبة","تبرع"]):
            ents.setdefault("الواهب",        p1)
            ents.setdefault("الموهوب_له",    p2)

    return ents

# ══════════════════════════════════════════════════════════════
# EXTRACTEUR UNIFIÉ
# ══════════════════════════════════════════════════════════════
def extract_entities(question: str, contract_type_ar: str) -> dict:
    rx_ents  = _extract_entities_regex(question)
    llm_ents = _extract_entities_llm(question, contract_type_ar)
    print(f"  [REGEX-ENTS] → {rx_ents}")
    print(f"  [LLM-ENTS]   → {llm_ents}")
    merged   = dict(rx_ents)
    merged.update(llm_ents)
    return {
        k: v for k, v in merged.items()
        if v and str(v).strip() not in ("null", "None", "-", "")
    }

# ══════════════════════════════════════════════════════════════
# ANALYSE DU PROMPT
# ══════════════════════════════════════════════════════════════
_P_ANALYSE = """\
أنت محلل قانوني متخصص في ق.ل.ع المغربي.
السؤال: {question}
أرجع JSON فقط:
{{"type":"factuelle|interprétative|comparative|procédurale",
  "domaines":[],"ambiguites":[],"mots_cles_arabes":[],"hors_loc":false,
  "contrat_demande":false}}"""

# ══════════════════════════════════════════════════════════════
# UTILITAIRES RETRIEVAL
# ══════════════════════════════════════════════════════════════
_ART_REF_RE = re.compile(r"الفصل\s*(\d+(?:[.\-]\d+)?)", re.UNICODE)

def _lookup_direct(question: str) -> list:
    found, seen = [], set()
    for raw in _ART_REF_RE.findall(question):
        num = raw.strip().replace('.', '-')
        if num not in seen and num in articles_by_num:
            for chunk in articles_by_num[num]:
                found.append(chunk)
            seen.add(num)
    return found

def _filter_by_rrf(candidates, rrf_map, min_score=RRF_MIN_SCORE):
    return [ch for ch in candidates
            if rrf_map.get(ch.get('article', ''), 0.0) >= min_score
            or rrf_map.get(ch.get('article', ''), 0.0) == 0.0]

def _chunks_to_context(chunks_list, max_chunks=3):
    parts = []
    for ch in chunks_list[:max_chunks]:
        art_id  = true_art_num(ch.get('article', '?'))
        contenu = ch.get('text', ch.get('contenu', ''))
        if contenu:
            parts.append(f"══ الفصل {art_id} من ق.ل.ع ══\n{contenu}")
    return "\n\n".join(parts)

def _format_articles_used(chunks_list):
    if not chunks_list:
        return "لا توجد فصول محددة"
    arts, seen = [], set()
    for c in chunks_list:
        true_num = true_art_num(c.get('article', '?'))
        if true_num not in seen:
            arts.append(f"الفصل {true_num}")
            seen.add(true_num)
    return "، ".join(arts)

def _retrieve_contract_articles_direct(contract_type: dict) -> list:
    found, seen = [], set()
    candidate_nums = list(contract_type.get("articles", [])) + ARTICLES_GENERAUX_CONTRAT
    for art_num in candidate_nums:
        if art_num in articles_by_num:
            for chunk in articles_by_num[art_num]:
                cid = chunk.get("article", art_num)
                if cid not in seen:
                    found.append(chunk)
                    seen.add(cid)
    return found

# ══════════════════════════════════════════════════════════════
# ÉTAT
# ══════════════════════════════════════════════════════════════
class AgentState(TypedDict):
    question:             str
    question_type:        str
    domains:              List[str]
    ambiguities:          List[str]
    out_of_scope:         bool
    keywords:             List[str]
    is_direct_lookup:     bool
    is_contract_drafting: bool
    contract_type:        dict
    contract_entities:    dict
    candidates:           List[dict]
    rrf_map:              dict
    selected:             List[dict]
    rejected:             List[dict]
    final_answer:         str
    json_output:          dict   # ← sortie JSON structurée
    confidence:           str
    mode:                 str
    top_k:                int
    iteration:            int
    logs:                 Annotated[List[str], operator.add]

# ══════════════════════════════════════════════════════════════
# NŒUDS
# ══════════════════════════════════════════════════════════════

def node_analyse(state: AgentState) -> AgentState:
    prompt   = _P_ANALYSE.format(question=state["question"])
    raw      = _llm_call(prompt, max_tokens=300)
    fallback = {"type": "factuelle", "domaines": [], "ambiguites": [],
                "mots_cles_arabes": [], "hors_loc": False, "contrat_demande": False}
    result   = _parse_json(raw, fallback)

    contract_type = _detect_contract_type(state["question"])
    is_contract   = _is_contract_drafting(state["question"]) or bool(contract_type)
    top_k = (20 if is_contract
             else TOP_K_BY_TYPE.get(result.get("type", "factuelle"), TOP_K_RETRIEVAL))

    if is_contract:
        result["contrat_demande"] = True
        result["type"]            = "procédurale"

    type_ar  = contract_type.get("type_ar", "") if contract_type else ""
    entities = extract_entities(state["question"], type_ar) if is_contract else {}

    return {**state,
            "question_type":        result.get("type", "factuelle"),
            "domains":              result.get("domaines", []),
            "ambiguities":          result.get("ambiguites", []),
            "keywords":             result.get("mots_cles_arabes", []),
            "out_of_scope":         bool(result.get("hors_loc", False)),
            "is_direct_lookup":     False,
            "is_contract_drafting": is_contract,
            "contract_type":        contract_type or {},
            "contract_entities":    entities,
            "top_k":                top_k,
            "logs": [
                f"[ANALYSE-v11] type={result.get('type','?')} | top_k={top_k}"
                f" | contrat={is_contract}"
                f" | entités_pré-extraites={list(entities.keys())}"
            ]}

def node_info_collection(state: AgentState) -> AgentState:
    contract_type = state.get("contract_type", {})
    entities      = dict(state.get("contract_entities", {}))

    if not contract_type:
        sep = "─" * 50
        print(f"\n{sep}")
        print("  ⚠️  لم يُتعرَّف على نوع العقد.")
        print("  الأنواع المتاحة :")
        for k, v in CONTRACT_CATALOGUE.items():
            print(f"    • {k} ({v['type_fr']})")
        print(sep)
        choice = input("  📝 أدخل نوع العقد : ").strip()
        contract_type = CONTRACT_CATALOGUE.get(choice, {})
        if not contract_type:
            for k, v in CONTRACT_CATALOGUE.items():
                if choice in k or k in choice:
                    contract_type = v
                    break
        if not contract_type:
            return {**state, "contract_type": {}, "contract_entities": entities,
                    "logs": [f"[INFO_COLLECT] نوع عقد غير معروف : {choice}"]}

    type_ar  = contract_type.get("type_ar", "عقد")
    type_fr  = contract_type.get("type_fr", "contrat")
    required = REQUIRED_FIELDS.get(type_ar, [])

    if not required:
        return {**state, "contract_type": contract_type,
                "contract_entities": entities,
                "logs": [f"[INFO_COLLECT] لا توجد حقول مُعرَّفة لـ {type_ar}"]}

    sep = "═" * 58
    print(f"\n{sep}")
    print(f"  🗂️  نوع العقد المطلوب : {type_ar} ({type_fr})")
    print(f"  📋  عدد الحقول المطلوبة : {len(required)}")
    print(f"{sep}")

    already_filled = {k: v for k, v in entities.items() if v}
    if already_filled:
        print(f"\n  ✅ المعلومات المُستخرَجة من طلبك :")
        for k, v in already_filled.items():
            print(f"    • {k} : {v}")

    missing_fields = [
        (fk, fl, fo)
        for fk, fl, fo in required
        if fk not in entities or not entities[fk]
    ]

    if missing_fields:
        print(f"\n  📝 يرجى تزويدي بالمعلومات الناقصة التالية :")
        print(f"  (اكتب '-' أو اضغط Enter للتخطي في الحقول الاختيارية)\n")
        for i, (field_key, field_label, is_optional) in enumerate(missing_fields, 1):
            optional_tag = "  [اختياري]" if is_optional else "  [إلزامي]"
            prompt_str   = f"  [{i}/{len(missing_fields)}]{optional_tag} {field_label} : "
            while True:
                val = input(prompt_str).strip()
                if is_optional and (not val or val in ("-","لا","non","no","/")):
                    entities[field_key] = None
                    print(f"    ↳ تم التخطي\n")
                    break
                if not is_optional and not val:
                    print(f"    ⚠️  هذا الحقل إلزامي. يرجى الإجابة.\n")
                    continue
                entities[field_key] = val
                print(f"    ✅ تم التسجيل : {val}\n")
                break
    else:
        print(f"\n  ✅ جميع المعلومات متوفرة — لا حاجة لأسئلة إضافية.")

    filled_count     = sum(1 for v in entities.values() if v)
    optional_skipped = sum(1 for fk, _, opt in required
                           if opt and (entities.get(fk) is None))
    print(f"\n{sep}")
    print(f"  📊 ملخص الجمع : {filled_count} حقل مُعبَّأ"
          f" | {optional_skipped} حقل اختياري متخطَّى")
    print(f"{sep}\n")

    clean_entities = {k: v for k, v in entities.items() if v}
    return {**state, "contract_type": contract_type,
            "contract_entities": clean_entities,
            "logs": [f"[INFO_COLLECT] type={type_ar}"
                     f" | مُستخرَجة={len(already_filled)}"
                     f" | جُمِعت={len(missing_fields)}"
                     f" | إجمالي={len(clean_entities)}"]}

def node_retrieve(state: AgentState) -> AgentState:
    question      = state["question"]
    keywords      = state.get("keywords", [])
    top_k         = state.get("top_k", TOP_K_RETRIEVAL)
    contract_type = state.get("contract_type", {})
    is_contract   = state.get("is_contract_drafting", False)

    direct   = _lookup_direct(question)
    enriched = question + (" " + " ".join(keywords[:4]) if keywords else "")

    if is_contract and contract_type:
        enriched += " " + " ".join(contract_type.get("keywords", [])[:4])
        catalogue_articles = _retrieve_contract_articles_direct(contract_type)
        direct = catalogue_articles + [c for c in direct
                                       if c.get("article") not in
                                       {d.get("article") for d in catalogue_articles}]

    semantic = retriever.retrieve_filtered(enriched, top_k=top_k)
    seen, final, rrf_map = {c.get('article', '') for c in direct}, list(direct), {}

    for r in semantic:
        art_id = r.chunk.get('article', r.chunk_id)
        rrf_map[art_id] = r.rrf_score
        if art_id not in seen:
            final.append(r.chunk)
            seen.add(art_id)

    before = len(final)
    final  = _filter_by_rrf(final, rrf_map, RRF_MIN_SCORE)
    final.sort(key=lambda c: rrf_map.get(c.get('article', ''), 0), reverse=True)

    return {**state, "candidates": final, "rrf_map": rrf_map,
            "logs": [f"[RETRIEVE] {len(final)}/{before} après filtre RRF≥{RRF_MIN_SCORE}"]}

def node_context_filter(state: AgentState) -> AgentState:
    candidates  = state.get("candidates", [])
    question    = state["question"]
    is_contract = state.get("is_contract_drafting", False)

    if not candidates:
        return {**state, "selected": [], "rejected": [],
                "logs": ["[FILTER] Aucun candidat"]}

    cands_with_text = [
        {**ch, "text": ch.get('text', ch.get('contenu', ''))}
        for ch in candidates if ch.get('text', ch.get('contenu', ''))
    ]
    if not cands_with_text:
        return {**state, "selected": [], "rejected": [],
                "logs": ["[FILTER] Candidats sans texte"]}

    ranked    = rerank(question, cands_with_text, reranker, top_k=len(cands_with_text))
    threshold = 0.10 if is_contract else RERANK_MIN_SCORE
    limit     = 12  if is_contract else MAX_SELECTED

    above = [r for r in ranked if r.get("rerank_score", 0) >= threshold]
    below = [r for r in ranked if r.get("rerank_score", 0) <  threshold]

    if is_contract and state.get("contract_type"):
        cat_arts = set(state["contract_type"].get("articles", []) + ARTICLES_GENERAUX_CONTRAT)
        forced   = [r for r in below if r.get("article", "") in cat_arts]
        already  = {r.get("article", "") for r in above}
        above   += [r for r in forced if r.get("article", "") not in already]

    selected = above[:limit]
    rejected = below + above[limit:]

    return {**state, "selected": selected, "rejected": rejected,
            "logs": [f"[FILTER] selected={[r.get('article') for r in selected]}"]}


# ══════════════════════════════════════════════════════════════
# NODE FORMAT — SORTIE JSON STRUCTURÉE, RÉORGANISÉE, TOP-3 ARTICLES
# ══════════════════════════════════════════════════════════════
def _article_score(ch: dict) -> float:
    try:
        return float(ch.get("final_score", ch.get("rerank_score", 0.0)) or 0.0)
    except (TypeError, ValueError):
        return 0.0


def node_format(state: AgentState) -> AgentState:
    selected   = state.get("selected", [])
    rrf_map    = state.get("rrf_map", {})
    ct         = state.get("contract_type", {})
    entities   = state.get("contract_entities", {})
    is_contrat = state.get("is_contract_drafting", False)
    question   = state.get("question", "")

    # ── filtrage des articles non pertinents ─────────────────
    _has_duration = bool(re.search(
        r"(?:لمدة|مدة)\s*(?:سنة|سنتين|\d+\s*سنوات?|شهر|شهرين|\d+\s*أشهر?)",
        question, re.UNICODE))
    _has_penalty  = bool(re.search(
        r"(?:تعويض|شرط جزائي|أضرار)", question, re.UNICODE))
    _banned = set()
    if _has_duration:    _banned.add("688")
    if not _has_penalty: _banned.add("264")

    display_arts = [c for c in selected
                    if true_art_num(c.get("article", "")) not in _banned]

    # [FIX] tri explicite par score (rerank) décroissant — c'est ce
    # tri qui détermine quels articles sont affichés, et pas l'ordre
    # hérité de `selected` (qui peut contenir des articles "forcés"
    # du catalogue ajoutés en fin de liste sans tri).
    display_arts_sorted = sorted(display_arts, key=_article_score, reverse=True)

    # [FIX] on ne garde que les TOP_N_ARTICLES_DISPLAY meilleurs pour
    # l'affichage, même en mode contrat où `selected` peut en
    # contenir jusqu'à 12.
    top_arts = display_arts_sorted[:TOP_N_ARTICLES_DISPLAY]

    # ── construction de la liste des articles affichés (top-3) ──
    articles_list = []
    for rang, ch in enumerate(top_arts, start=1):
        art_id_ocr  = ch.get("article", "?")
        art_id_true = true_art_num(art_id_ocr)
        texte       = ch.get("text", ch.get("contenu", ""))
        final_score = _article_score(ch)                       # score utilisé pour le tri
        bge_raw     = float(ch.get("rerank_score", 0.0) or 0.0) # BGE brut, diagnostic
        rrf_score   = rrf_map.get(art_id_ocr, 0.0)

        articles_list.append({
            "rang":           rang,
            "numero":         art_id_true,
            "numero_ocr":     art_id_ocr,
            "texte":          texte[:800] + (" [...]" if len(texte) > 800 else ""),
            "score_final":    round(final_score, 4),  # score réel ayant déterminé le rang
            "score_rerank":   round(bge_raw, 4),       # BGE brut, à titre diagnostic seulement
            "score_rrf":      round(float(rrf_score), 4),
            "tronque":        len(texte) > 800,
        })

    # ── champs obligatoires manquants ────────────────────────
    type_ar  = ct.get("type_ar", "")
    req_keys = [fk for fk, _, _ in REQUIRED_FIELDS.get(type_ar, [])]
    missing_fields = [
        fk for fk in req_keys
        if fk not in entities or not entities.get(fk)
    ]

    # [FIX] bloc "contrat" unifié : on regroupe entités + complétude
    # SOUS contrat plutôt que comme deux clés racine séparées qui
    # valent toutes deux `null` quand il n'y a pas de contrat — plus
    # simple à consommer côté code appelant (`if contrat: ...`).
    contrat_block = None
    if is_contrat:
        contrat_block = {
            "demande":  True,
            "type_ar":  ct.get("type_ar", None),
            "type_fr":  ct.get("type_fr", None),
            "detecte":  bool(ct),
            "entites": {
                "donnees":          entities,
                "champs_manquants": missing_fields,
                "completude_pct": (
                    round(len([k for k in req_keys if k in entities and entities[k]])
                          / len(req_keys) * 100, 1)
                    if req_keys else 100.0
                ),
            },
        }

    # ── construction du JSON de sortie réorganisé ────────────
    # [FIX] blocs logiques : requete / contrat / articles / metadata
    # au lieu de clés plates mélangées (analyse, entites_extraites,
    # articles_juridiques, contrat séparés au même niveau).
    json_out = {
        "version":   "LOC-v11",
        "timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "requete": {
            "question":    question,
            "type":        state.get("question_type", ""),
            "domaines":    state.get("domains", []),
            "ambiguites":  state.get("ambiguities", []),
            "hors_scope":  state.get("out_of_scope", False),
        },
        "contrat": contrat_block,
        "articles": {
            "total_recuperes":    len(state.get("candidates", [])),
            "total_selectionnes": len(display_arts),
            "top_affiches":       len(articles_list),
            "seuils": {
                "rrf":    RRF_MIN_SCORE,
                "rerank": RERANK_MIN_SCORE,
            },
            "liste":          articles_list,
            "resume_numeros": [a["numero"] for a in articles_list],
        },
        "metadata": {
            "top_k_utilise": state.get("top_k", TOP_K_RETRIEVAL),
            "iteration":     state.get("iteration", 0),
            "logs":          state.get("logs", []),
        },
    }

    final_answer_str = json.dumps(json_out, ensure_ascii=False, indent=2)

    return {**state,
            "final_answer": final_answer_str,
            "json_output":  json_out,
            "logs": [f"[FORMAT-v11-JSON] ok — top-{len(articles_list)} articles affichés"]}


def node_out_of_scope(state: AgentState) -> AgentState:
    json_out = {
        "version":   "LOC-v11",
        "timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "requete": {
            "question":   state.get("question", ""),
            "hors_scope": True,
        },
        "erreur":  "hors_scope",
        "message": "هذا السؤال خارج نطاق ق.ل.ع المغربي.",
        "contrat": None,
        "articles": {
            "total_recuperes":    0,
            "total_selectionnes": 0,
            "top_affiches":       0,
            "seuils":             {"rrf": RRF_MIN_SCORE, "rerank": RERANK_MIN_SCORE},
            "liste":              [],
            "resume_numeros":     [],
        },
        "metadata": {"logs": state.get("logs", [])},
    }
    return {**state,
            "final_answer": json.dumps(json_out, ensure_ascii=False, indent=2),
            "json_output":  json_out,
            "logs": ["[HORS_LOC]"]}


# ══════════════════════════════════════════════════════════════
# ROUTAGE
# ══════════════════════════════════════════════════════════════
def cond_scope(state):
    return "out_of_scope" if state.get("out_of_scope") else "info_collection"


# ══════════════════════════════════════════════════════════════
# GRAPHE v11
# ══════════════════════════════════════════════════════════════
workflow = StateGraph(AgentState)
for name, fn in [
    ("analyse",         node_analyse),
    ("out_of_scope",    node_out_of_scope),
    ("info_collection", node_info_collection),
    ("retrieve",        node_retrieve),
    ("context_filter",  node_context_filter),
    ("format",          node_format),
]:
    workflow.add_node(name, fn)

workflow.set_entry_point("analyse")
workflow.add_conditional_edges(
    "analyse", cond_scope,
    {"out_of_scope": "out_of_scope", "info_collection": "info_collection"})
workflow.add_edge("out_of_scope",    END)
workflow.add_edge("info_collection", "retrieve")
workflow.add_edge("retrieve",        "context_filter")
workflow.add_edge("context_filter",  "format")
workflow.add_edge("format",          END)

langgraph_app = workflow.compile()
print("  ✅ LangGraph v11 compilé (JSON réorganisé, top-3 articles affichés)")


# ══════════════════════════════════════════════════════════════
# POINTS D'ENTRÉE
# ══════════════════════════════════════════════════════════════
def ask_langgraph(question: str, mode: str = "complet", top_k: int = None) -> dict:
    _top_k = top_k or TOP_K_RETRIEVAL
    print(f"\n{'═'*62}")
    print(f"  🤖 LOC v11 — جمع المعلومات وعرض أفضل 3 فصول قانونية")
    print(f"  السؤال : {question}")
    print(f"{'═'*62}")
    initial = AgentState(
        question=question, question_type="", domains=[], ambiguities=[],
        out_of_scope=False, keywords=[], is_direct_lookup=False,
        is_contract_drafting=False, contract_type={}, contract_entities={},
        candidates=[], rrf_map={}, selected=[], rejected=[],
        final_answer="", json_output={}, confidence="moyenne", mode=mode,
        top_k=_top_k, iteration=0, logs=[],
    )
    result = langgraph_app.invoke(initial)

    # Affichage JSON en console
    print(result.get("final_answer", "{}"))

    # Persistance dans le log
    with open(AGENT_LOG, "a", encoding="utf-8") as f:
        f.write(json.dumps({
            "question":     question,
            "mode":         mode,
            "type_contrat": result.get("contract_type", {}).get("type_ar", ""),
            "entities":     result.get("contract_entities", {}),
            "selected":     [c.get("article") for c in result.get("selected", [])],
            "logs":         result.get("logs", []),
            "json_output":  result.get("json_output", {}),
            "timestamp":    time.time(),
        }, ensure_ascii=False) + "\n")

    return result


def ask_loc(question: str, mode: str = "complet",
            top_k: int = TOP_K_RETRIEVAL) -> dict:
    t_start = time.time()
    print(f"\n{'═'*62}")
    print(f"  🤖 LOC AGENTIC AI — LANGGRAPH v11")
    print(f"  Question : {question}")
    print(f"{'═'*62}")
    result  = ask_langgraph(question, mode=mode, top_k=top_k)
    elapsed = round(time.time() - t_start, 2)
    print(f"\n  ⏱️  Temps total : {elapsed}s")
    # Retourne directement le dict JSON pour usage programmatique
    return result.get("json_output", {})


# ══════════════════════════════════════════════════════════════
# INTERFACE INTERACTIVE
# ══════════════════════════════════════════════════════════════
print("\n" + "═"*62)
print("  💬 Entrez votre question juridique (tapez 'exit' pour quitter)")
print("  Exemple : 'أريد عقد إيجار' / 'أحمد ويوسف يؤسسان شركة'")
print("═"*62)

while True:
    user_question = input("\n📝 Votre question : ").strip()
    if user_question.lower() in ("exit", "quit", "q"):
        print("  👋 Au revoir !")
        break
    if not user_question:
        continue
    ask_loc(user_question, mode="complet")

<>:485: SyntaxWarning: invalid escape sequence '\s'
<>:486: SyntaxWarning: invalid escape sequence '\s'
<>:487: SyntaxWarning: invalid escape sequence '\s'
<>:488: SyntaxWarning: invalid escape sequence '\s'
<>:489: SyntaxWarning: invalid escape sequence '\s'
<>:490: SyntaxWarning: invalid escape sequence '\s'
<>:492: SyntaxWarning: invalid escape sequence '\s'
<>:493: SyntaxWarning: invalid escape sequence '\s'
<>:494: SyntaxWarning: invalid escape sequence '\s'
<>:495: SyntaxWarning: invalid escape sequence '\s'
<>:496: SyntaxWarning: invalid escape sequence '\s'
<>:498: SyntaxWarning: invalid escape sequence '\s'
<>:499: SyntaxWarning: invalid escape sequence '\s'
<>:500: SyntaxWarning: invalid escape sequence '\s'
<>:485: SyntaxWarning: invalid escape sequence '\s'
<>:486: SyntaxWarning: invalid escape sequence '\s'
<>:487: SyntaxWarning: invalid escape sequence '\s'
<>:488: SyntaxWarning: invalid escape sequence '\s'
<>:489: SyntaxWarning: invalid escape sequence '\s'
<>:490: Synt

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
CELLULE 11 — LangGraph v11  (JSON output réorganisé, top-3)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
✅ Canonicalisation articles OCR→LOC vérifiée : 618→628 | 619→629 | 617→627
  ✅ LangGraph v11 compilé (JSON réorganisé, top-3 articles affichés)

══════════════════════════════════════════════════════════════
  💬 Entrez votre question juridique (tapez 'exit' pour quitter)
  Exemple : 'أريد عقد إيجار' / 'أحمد ويوسف يؤسسان شركة'
══════════════════════════════════════════════════════════════



📝 Votre question :  exit


  👋 Au revoir !


## METRIQUES EVALUATIONS

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELLULE 12 — Métriques d'Évaluation Professionnelles LOC
# ══════════════════════════════════════════════════════════════


import math
import time
import json
import statistics
from collections import defaultdict, Counter

# ──────────────────────────────────────────────────────────────
# VÉRIFICATION DES DÉPENDANCES
# ──────────────────────────────────────────────────────────────
for _var in ["retriever", "chunks_parsed", "articles_by_num"]:
    assert _var in globals(), f"❌ {_var} non défini — relancez les cellules 1–9"

print("━" * 62)
print("CELLULE 12 — Évaluation Professionnelle RAG-LOC")
print("━" * 62)

# ══════════════════════════════════════════════════════════════
# EVAL SET ÉTENDU (20 requêtes, niveaux de difficulté variés)
# ══════════════════════════════════════════════════════════════
# gold_grades = {article: score_pertinence}
#   3 = central, 2 = fortement lié, 1 = lié, 0 = hors sujet
EVAL_SET_GRADED = [
    # ── Niveau FACILE (concepts isolés, articles bien connus) ──
    {
        "query":      "الأهلية المدنية للقاصر",
        "difficulty": "easy",
        "gold":       {"4": 3, "5": 2, "9": 2, "12": 1, "3": 1},
        "gold_binary":[4, 5, 9, 12],
    },
    {
        "query":      "عقد البيع والشراء",
        "difficulty": "easy",
        "gold":       {"478": 3, "488": 2, "502": 2, "503": 1, "532": 1},
        "gold_binary":[478, 488, 502],
    },
    {
        "query":      "الإفلاس والتسوية القضائية",
        "difficulty": "easy",
        "gold":       {"1247": 3, "1248": 2, "1249": 1},
        "gold_binary":[1247, 1248],
    },
    {
        "query":      "المسؤولية التقصيرية",
        "difficulty": "easy",
        "gold":       {"77": 3, "78": 2, "79": 2, "98": 1},
        "gold_binary":[77, 78, 79],
    },
    {
        "query":      "الأهلية المدنية",
        "difficulty": "easy",
        "gold":       {"4": 3, "5": 2, "12": 2, "9": 1},
        "gold_binary":[4, 5, 12],
    },

    # ── Niveau MOYEN (multi-concepts, chevauchements) ──────────
    {
        "query":      "التعويض عن الضرر",
        "difficulty": "medium",
        "gold":       {"264": 3, "77": 2, "98": 2, "265": 1},
        "gold_binary":[264, 77, 98],
    },
    {
        "query":      "الالتزامات التعاقدية",
        "difficulty": "medium",
        "gold":       {"230": 3, "228": 2, "229": 2, "18": 1},
        "gold_binary":[230, 228, 229],
    },
    {
        "query":      "فسخ العقد",
        "difficulty": "medium",
        "gold":       {"259": 3, "260": 3, "261": 2, "262": 1, "230": 1},
        "gold_binary":[259, 260, 261],
    },
    {
        "query":      "عقد الكفالة والضمان",
        "difficulty": "medium",
        "gold":       {"1117": 3, "1118": 3, "1120": 2, "1119": 2, "1121": 1},
        "gold_binary":[1117, 1118, 1120],
    },
    {
        "query":      "الوكالة والتفويض",
        "difficulty": "medium",
        "gold":       {"879": 3, "880": 2, "893": 2, "881": 1},
        "gold_binary":[879, 880, 893],
    },

    # ── Niveau DIFFICILE (requêtes ambiguës, concepts croisés) ─
    {
        "query":      "انقضاء الالتزامات بالوفاء",
        "difficulty": "hard",
        "gold":       {"320": 3, "321": 2, "322": 2, "323": 1},
        "gold_binary":[320, 321, 322],
    },
    {
        "query":      "شروط صحة العقد والتراضي",
        "difficulty": "hard",
        "gold":       {"2": 3, "19": 3, "39": 2, "4": 1, "57": 1},
        "gold_binary":[2, 19, 39],
    },
    {
        "query":      "عقد الإيجار والتزامات الأطراف",
        "difficulty": "hard",
        "gold":       {"627": 3, "628": 2, "629": 2, "635": 2, "663": 1},
        "gold_binary":[627, 628, 629, 635],
    },
    {
        "query":      "الرهن الحيازي وحقوق الدائن",
        "difficulty": "hard",
        "gold":       {"1170": 3, "1171": 2, "1172": 2, "1180": 1},
        "gold_binary":[1170, 1171, 1172],
    },
    {
        "query":      "بطلان العقد وأسبابه",
        "difficulty": "hard",
        "gold":       {"306": 3, "307": 2, "308": 2, "86": 1},
        "gold_binary":[306, 307, 308],
    },

    # ── Niveau EXPERT (formulations juridiques précises) ──────
    {
        "query":      "مسؤولية الموكل عن تصرفات الوكيل",
        "difficulty": "expert",
        "gold":       {"893": 3, "879": 2, "880": 1},
        "gold_binary":[893, 879],
    },
    {
        "query":      "التزامات المستأجر في عقد الكراء",
        "difficulty": "expert",
        "gold":       {"663": 3, "664": 2, "668": 1, "635": 1},
        "gold_binary":[663, 664, 668],
    },
    {
        "query":      "الإعذار وشروط الفسخ القضائي",
        "difficulty": "expert",
        "gold":       {"255": 3, "256": 2, "259": 2, "260": 1},
        "gold_binary":[255, 256, 259],
    },
    {
        "query":      "الكسب غير المشروع وشبه العقد",
        "difficulty": "expert",
        "gold":       {"66": 3, "67": 2, "72": 1},
        "gold_binary":[66, 67],
    },
    {
        "query":      "القوة الملزمة للعقد وأثره بين الطرفين",
        "difficulty": "expert",
        "gold":       {"230": 3, "228": 2, "229": 2, "231": 1},
        "gold_binary":[230, 228, 229],
    },
]

# ══════════════════════════════════════════════════════════════
# CORE : CALCUL DES MÉTRIQUES
# ══════════════════════════════════════════════════════════════

def _parse_art(art_raw) -> int:
    """Normalise un numéro d'article (OCR ou canonique) en entier."""
    # OCR→vrai (ex: 617→627, 618→628, 619→629)
    _OCR_MAP = {"617": "627", "618": "628", "619": "629"}
    s = str(art_raw).split("-")[0]
    s = _OCR_MAP.get(s, s)
    try:
        return int(s)
    except ValueError:
        return 0


def recall_at_k(retrieved: list, gold_binary: list, k: int) -> float:
    """Recall@K : fraction des articles gold retrouvés dans les K premiers."""
    if not gold_binary:
        return 0.0
    top_k  = [_parse_art(r) for r in retrieved[:k]]
    hits   = sum(1 for g in gold_binary if g in top_k)
    return round(hits / len(gold_binary), 4)


def precision_at_k(retrieved: list, gold_binary: list, k: int) -> float:
    """Precision@K : fraction des K résultats qui sont dans gold."""
    if not retrieved or k == 0:
        return 0.0
    top_k = [_parse_art(r) for r in retrieved[:k]]
    hits  = sum(1 for r in top_k if r in set(gold_binary))
    return round(hits / min(k, len(top_k)), 4)


def f1_at_k(retrieved: list, gold_binary: list, k: int) -> float:
    """F1@K : moyenne harmonique de Precision@K et Recall@K."""
    p = precision_at_k(retrieved, gold_binary, k)
    r = recall_at_k(retrieved, gold_binary, k)
    if p + r == 0:
        return 0.0
    return round(2 * p * r / (p + r), 4)


def reciprocal_rank(retrieved: list, gold_binary: list) -> float:
    """Rang réciproque du premier article gold retrouvé."""
    gold_set = set(gold_binary)
    for rank, r in enumerate(retrieved, 1):
        if _parse_art(r) in gold_set:
            return round(1.0 / rank, 4)
    return 0.0


def average_precision(retrieved: list, gold_binary: list, k: int) -> float:
    """Average Precision@K (AP)."""
    gold_set = set(gold_binary)
    hits, total_ap = 0, 0.0
    for i, r in enumerate(retrieved[:k], 1):
        if _parse_art(r) in gold_set:
            hits += 1
            total_ap += hits / i
    if not gold_set:
        return 0.0
    return round(total_ap / len(gold_set), 4)


def dcg_at_k(retrieved: list, gold_grades: dict, k: int) -> float:
    """Discounted Cumulative Gain@K avec scores de pertinence gradués."""
    score = 0.0
    for i, r in enumerate(retrieved[:k], 1):
        art  = _parse_art(r)
        rel  = gold_grades.get(str(art), 0)
        score += rel / math.log2(i + 1)
    return score


def ndcg_at_k(retrieved: list, gold_grades: dict, k: int) -> float:
    """NDCG@K : DCG normalisé par le DCG idéal."""
    actual_dcg = dcg_at_k(retrieved, gold_grades, k)
    ideal      = sorted(gold_grades.values(), reverse=True)[:k]
    ideal_dcg  = sum(rel / math.log2(i + 2) for i, rel in enumerate(ideal))
    if ideal_dcg == 0:
        return 0.0
    return round(actual_dcg / ideal_dcg, 4)


def hit_at_k(retrieved: list, gold_binary: list, k: int) -> float:
    """Hit@K : 1.0 si au moins 1 article gold dans le top-K."""
    gold_set = set(gold_binary)
    return 1.0 if any(_parse_art(r) in gold_set for r in retrieved[:k]) else 0.0


# ══════════════════════════════════════════════════════════════
# RUNNER : ÉVALUATION COMPLÈTE
# ══════════════════════════════════════════════════════════════

def run_full_evaluation(
    retriever_obj,
    eval_set:     list  = EVAL_SET_GRADED,
    k_values:     list  = [1, 3, 5, 10],
    top_k_ret:    int   = 30,
    verbose:      bool  = True,
) -> dict:
    """
    Lance l'évaluation complète et retourne un dict de résultats.
    """
    print(f"\n{'═'*62}")
    print(f"  ÉVALUATION PROFESSIONNELLE — {len(eval_set)} requêtes | K={k_values}")
    print(f"{'═'*62}\n")

    per_query   = []
    latencies   = []
    n_no_result = 0

    for item in eval_set:
        query        = item["query"]
        gold_binary  = item["gold_binary"]
        gold_grades  = item["gold"]
        difficulty   = item.get("difficulty", "?")

        t0 = time.perf_counter()
        try:
            results  = retriever_obj.retrieve_reranked(
                query, top_k=max(k_values), retrieval_k=top_k_ret
            )
            retrieved = [r["article"] for r in results]
        except Exception as e:
            print(f"  ⚠️  Erreur sur «{query}» : {e}")
            retrieved = []
        latency = time.perf_counter() - t0
        latencies.append(latency)

        if not retrieved:
            n_no_result += 1

        row = {
            "query":      query,
            "difficulty": difficulty,
            "retrieved":  retrieved,
            "latency_s":  round(latency, 3),
        }

        for k in k_values:
            row[f"recall@{k}"]    = recall_at_k(retrieved, gold_binary, k)
            row[f"precision@{k}"] = precision_at_k(retrieved, gold_binary, k)
            row[f"f1@{k}"]        = f1_at_k(retrieved, gold_binary, k)
            row[f"ndcg@{k}"]      = ndcg_at_k(retrieved, gold_grades, k)
            row[f"hit@{k}"]       = hit_at_k(retrieved, gold_binary, k)

        row["rr"]  = reciprocal_rank(retrieved, gold_binary)
        row["ap3"] = average_precision(retrieved, gold_binary, 3)

        per_query.append(row)

        if verbose:
            _sym = "✅" if row["rr"] > 0 else "❌"
            print(f"  {_sym}  [{difficulty:<6}]  {query}")
            print(f"         → {retrieved[:5]}   |  "
                  f"R@3={row['recall@3']:.2f}  "
                  f"NDCG@3={row['ndcg@3']:.2f}  "
                  f"MRR={row['rr']:.3f}  "
                  f"({latency:.2f}s)")

    # ── Agrégation ─────────────────────────────────────────────
    def _mean(key):
        vals = [r[key] for r in per_query if key in r]
        return round(statistics.mean(vals), 4) if vals else 0.0

    aggregated = {
        "n_queries":     len(eval_set),
        "coverage":      round(1 - n_no_result / len(eval_set), 4),
    }
    for k in k_values:
        for metric in ["recall", "precision", "f1", "ndcg", "hit"]:
            aggregated[f"{metric}@{k}"] = _mean(f"{metric}@{k}")
    aggregated["mrr"]  = _mean("rr")
    aggregated["map3"] = _mean("ap3")

    # Latence
    lat_sorted = sorted(latencies)
    n = len(lat_sorted)
    aggregated["latency_p50"] = round(lat_sorted[int(n * 0.50)], 3)
    aggregated["latency_p90"] = round(lat_sorted[min(int(n * 0.90), n-1)], 3)
    aggregated["latency_p95"] = round(lat_sorted[min(int(n * 0.95), n-1)], 3)
    aggregated["latency_mean"]= round(statistics.mean(latencies), 3)

    # Breakdown par difficulté
    difficulties = ["easy", "medium", "hard", "expert"]
    breakdown = {}
    for diff in difficulties:
        sub = [r for r in per_query if r["difficulty"] == diff]
        if sub:
            breakdown[diff] = {
                "n": len(sub),
                "recall@3":     round(statistics.mean(r["recall@3"]    for r in sub), 4),
                "ndcg@3":       round(statistics.mean(r["ndcg@3"]      for r in sub), 4),
                "precision@3":  round(statistics.mean(r["precision@3"] for r in sub), 4),
                "mrr":          round(statistics.mean(r["rr"]          for r in sub), 4),
            }
    aggregated["breakdown_by_difficulty"] = breakdown

    # Articles les plus souvent récupérés (faux positifs fréquents)
    all_retrieved = [_parse_art(a) for r in per_query for a in r["retrieved"]]
    all_gold_flat = set(
        g for item in eval_set for g in item["gold_binary"]
    )
    art_freq     = Counter(all_retrieved)
    top_fps      = [(art, cnt) for art, cnt in art_freq.most_common(10)
                    if art not in all_gold_flat]
    aggregated["top_false_positives"] = top_fps[:5]

    # Articles gold jamais récupérés
    never_retrieved = [
        g for g in all_gold_flat
        if not any(_parse_art(a) == g for r in per_query for a in r["retrieved"])
    ]
    aggregated["never_retrieved_gold"] = sorted(never_retrieved)

    return {"per_query": per_query, "aggregated": aggregated}


# ══════════════════════════════════════════════════════════════
# AFFICHAGE RAPPORT COMPLET
# ══════════════════════════════════════════════════════════════

def print_evaluation_report(results: dict, k_values: list = [1, 3, 5, 10]):
    agg = results["aggregated"]
    pq  = results["per_query"]

    SEP   = "═" * 62
    SEP_T = "─" * 62

    print(f"\n{SEP}")
    print("  RAPPORT D'ÉVALUATION — RAG-LOC  (Loi des Obligations et Contrats)")
    print(SEP)

    # ── 1. Vue d'ensemble ──────────────────────────────────────
    print(f"\n  {'MÉTRIQUES GLOBALES':^58}")
    print(f"  {SEP_T}")
    print(f"  {'Requêtes évaluées':<35} {agg['n_queries']:>6}")
    print(f"  {'Couverture (≥1 résultat)':<35} {agg['coverage']*100:>5.1f}%")
    print()

    # ── 2. Tableau Recall / Precision / F1 ────────────────────
    header = f"  {'Métrique':<20}"
    for k in k_values:
        header += f"  K={k:<4}"
    print(header)
    print(f"  {SEP_T}")

    for metric_name, key_prefix in [
        ("Recall@K",     "recall"),
        ("Precision@K",  "precision"),
        ("F1@K",         "f1"),
        ("NDCG@K",       "ndcg"),
        ("Hit@K",        "hit"),
    ]:
        row = f"  {metric_name:<20}"
        for k in k_values:
            val = agg.get(f"{key_prefix}@{k}", 0)
            row += f"  {val:.3f} "
        print(row)

    print(f"  {SEP_T}")
    print(f"  {'MRR (Mean Reciprocal Rank)':<20}  {agg['mrr']:.4f}")
    print(f"  {'MAP@3':<20}  {agg['map3']:.4f}")

    # ── 3. Latence ─────────────────────────────────────────────
    print(f"\n  {'PERFORMANCE & LATENCE':^58}")
    print(f"  {SEP_T}")
    print(f"  {'Latence moyenne':<35} {agg['latency_mean']:>5.2f}s")
    print(f"  {'Latence P50':<35} {agg['latency_p50']:>5.2f}s")
    print(f"  {'Latence P90':<35} {agg['latency_p90']:>5.2f}s")
    print(f"  {'Latence P95':<35} {agg['latency_p95']:>5.2f}s")

    # ── 4. Breakdown par difficulté ───────────────────────────
    print(f"\n  {'PERFORMANCE PAR NIVEAU DE DIFFICULTÉ':^58}")
    print(f"  {SEP_T}")
    print(f"  {'Niveau':<10}  {'N':>3}  {'R@3':>6}  {'NDCG@3':>7}  {'P@3':>6}  {'MRR':>6}")
    print(f"  {SEP_T}")
    bkd = agg.get("breakdown_by_difficulty", {})
    for lvl in ["easy", "medium", "hard", "expert"]:
        d = bkd.get(lvl)
        if d:
            bar_r = "█" * int(d["recall@3"] * 10)
            print(f"  {lvl:<10}  {d['n']:>3}  "
                  f"{d['recall@3']:>5.3f}  "
                  f"{d['ndcg@3']:>6.3f}  "
                  f"{d['precision@3']:>5.3f}  "
                  f"{d['mrr']:>5.3f}  {bar_r}")

    # ── 5. Top requêtes manquées (MRR=0) ─────────────────────
    failed = [r for r in pq if r["rr"] == 0.0]
    if failed:
        print(f"\n  {'REQUÊTES SANS RÉSULTAT GOLD (MRR=0)':<58}")
        print(f"  {SEP_T}")
        for r in failed[:8]:
            print(f"  ❌  [{r['difficulty']:<6}]  {r['query']}")
            print(f"       Retourné : {r['retrieved'][:4]}")

    # ── 6. Articles gold jamais retrouvés ─────────────────────
    never = agg.get("never_retrieved_gold", [])
    if never:
        print(f"\n  {'ARTICLES GOLD JAMAIS RÉCUPÉRÉS':^58}")
        print(f"  {SEP_T}")
        print(f"  {never}")

    # ── 7. Faux positifs fréquents ────────────────────────────
    fps = agg.get("top_false_positives", [])
    if fps:
        print(f"\n  {'FAUX POSITIFS LES PLUS FRÉQUENTS':^58}")
        print(f"  {SEP_T}")
        for art, cnt in fps:
            print(f"  الفصل {art:<6}  apparu {cnt}× sans être dans gold")

    # ── 8. Qualité par quartile de score ─────────────────────
    scores_r3 = sorted([r["recall@3"] for r in pq], reverse=True)
    n  = len(scores_r3)
    q1 = round(statistics.mean(scores_r3[:n//4]) if n >= 4 else 0, 3)
    q2 = round(statistics.mean(scores_r3[n//4:n//2]) if n >= 4 else 0, 3)
    q3 = round(statistics.mean(scores_r3[n//2:3*n//4]) if n >= 4 else 0, 3)
    q4 = round(statistics.mean(scores_r3[3*n//4:]) if n >= 4 else 0, 3)
    print(f"\n  {'DISTRIBUTION DES SCORES Recall@3':^58}")
    print(f"  {SEP_T}")
    print(f"  Q1 (top 25%)   {q1:.3f}   {'█' * int(q1*20)}")
    print(f"  Q2             {q2:.3f}   {'█' * int(q2*20)}")
    print(f"  Q3             {q3:.3f}   {'█' * int(q3*20)}")
    print(f"  Q4 (bottom 25%){q4:.3f}   {'█' * int(q4*20)}")

    # ── Score synthétique ─────────────────────────────────────
    composite = round(
        0.35 * agg["recall@3"]
        + 0.25 * agg["ndcg@3"]
        + 0.25 * agg["mrr"]
        + 0.15 * agg["precision@3"],
        4
    )
    grade = (
        "Excellent" if composite >= 0.75 else
        "Bon"       if composite >= 0.60 else
        "Moyen"     if composite >= 0.45 else
        "Faible"
    )
    print(f"\n  {SEP}")
    print(f"  Score composite (0.35×R@3 + 0.25×NDCG@3 + 0.25×MRR + 0.15×P@3)")
    print(f"  ► {composite:.4f}  ({grade})")
    print(f"  {SEP}\n")

    return composite


# ══════════════════════════════════════════════════════════════
# COHÉRENCE (CONSISTENCY CHECK)
# ══════════════════════════════════════════════════════════════

def evaluate_consistency(
    retriever_obj,
    n_runs: int = 3,
    sample_queries: list = None,
    k: int = 3,
) -> dict:
    """
    Lance N runs identiques sur un échantillon de requêtes
    et mesure la variance inter-runs (stabilité du retriever).
    """
    if sample_queries is None:
        sample_queries = [
            "الأهلية المدنية للقاصر",
            "فسخ العقد",
            "عقد الكفالة والضمان",
        ]
    print(f"\n{'─'*62}")
    print(f"  TEST DE COHÉRENCE ({n_runs} runs × {len(sample_queries)} requêtes)")
    print("─"*62)

    consistency_scores = {}
    for q in sample_queries:
        runs = []
        for _ in range(n_runs):
            try:
                res = retriever_obj.retrieve_reranked(q, top_k=k, retrieval_k=20)
                runs.append(frozenset(_parse_art(r["article"]) for r in res))
            except Exception:
                runs.append(frozenset())
        # Jaccard moyen entre paires de runs
        jaccard_vals = []
        for i in range(len(runs)):
            for j in range(i+1, len(runs)):
                inter = len(runs[i] & runs[j])
                union = len(runs[i] | runs[j])
                jaccard_vals.append(inter / union if union else 1.0)
        avg_j = round(statistics.mean(jaccard_vals), 4) if jaccard_vals else 0.0
        consistency_scores[q] = avg_j
        icon = "✅" if avg_j >= 0.8 else ("⚠️" if avg_j >= 0.5 else "❌")
        print(f"  {icon}  Jaccard@{k}={avg_j:.3f}  «{q}»")

    overall = round(statistics.mean(consistency_scores.values()), 4)
    print(f"\n  Cohérence globale (Jaccard moyen) : {overall:.4f}")
    return {"per_query": consistency_scores, "overall": overall}


# ══════════════════════════════════════════════════════════════
# SAUVEGARDE JSON DES RÉSULTATS
# ══════════════════════════════════════════════════════════════

def save_eval_results(results: dict, path: str, metadata: dict = None):
    """Sauvegarde les résultats d'évaluation en JSON pour archivage."""
    payload = {
        "metadata":   metadata or {"model": "LOC-RAG", "version": "v3.9"},
        "aggregated": results["aggregated"],
        "per_query":  results["per_query"],
        "timestamp":  time.strftime("%Y-%m-%dT%H:%M:%S"),
    }
    # Les frozensets/Counter ne sont pas JSON-sérialisables → conversion
    def _serialize(obj):
        if isinstance(obj, (set, frozenset)):
            return list(obj)
        if isinstance(obj, Counter):
            return dict(obj)
        raise TypeError(f"Type non sérialisable : {type(obj)}")

    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2, default=_serialize)
    print(f"  ✅ Résultats sauvegardés → {path}")


# ══════════════════════════════════════════════════════════════
# EXÉCUTION PRINCIPALE
# ══════════════════════════════════════════════════════════════
print("\n  ▶ Démarrage de l'évaluation complète…\n")

eval_results = run_full_evaluation(
    retriever_obj = retriever,
    eval_set      = EVAL_SET_GRADED,
    k_values      = [1, 3, 5, 10],
    top_k_ret     = 30,
    verbose       = True,
)

composite_score = print_evaluation_report(eval_results, k_values=[1, 3, 5, 10])

# ── Cohérence (optionnel — commenter pour accélérer) ──────────
# consistency = evaluate_consistency(retriever, n_runs=3)

# ── Export JSON ───────────────────────────────────────────────
if "BASE_DIR" in globals():
    _eval_path = f"{BASE_DIR}/loc_eval_results.json"
    save_eval_results(
        eval_results,
        path=_eval_path,
        metadata={
            "model":   "LOC-RAG",
            "version": "v3.9",
            "embedder": globals().get("db", {}).get("model_name", "?"),
        }
    )

print("\n✅ Cellule 12 terminée")
print("   Variables exportées :")
print("   eval_results         — dict complet (per_query + aggregated)")
print("   composite_score      — score synthétique global")
print("   EVAL_SET_GRADED      — jeu d'évaluation étendu (20 requêtes)")
print("   run_full_evaluation  — fonction d'évaluation principale")
print("   print_evaluation_report — affichage du rapport")
print("   evaluate_consistency — test de stabilité inter-runs")
print("   save_eval_results    — export JSON pour archivage")

## Deploiement Ngrok

In [12]:
# ══════════════════════════════════════════════════════════════
# CELLULE 12 — API FastAPI + ngrok (Kaggle)  v2.1
# Expose le pipeline LangGraph (/ask) — réponse JSON top-3 articles
# + endpoints GET /result/{question_id} et /metrics
# ══════════════════════════════════════════════════════════════

import subprocess, sys

# ──────────────────────────────────────────────────────────────
# 0. INSTALLATION DES DÉPENDANCES
# ──────────────────────────────────────────────────────────────
def _pip_install(pkg: str):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

for _pkg in ["fastapi", "uvicorn", "pyngrok", "nest_asyncio"]:
    try:
        __import__(_pkg.replace("-", "_"))
    except ImportError:
        print(f"  📦 Installation de {_pkg} …")
        _pip_install(_pkg)

print("✅ Dépendances prêtes.")

# ──────────────────────────────────────────────────────────────
# 1. VÉRIFICATION DES DÉPENDANCES AMONT
# ──────────────────────────────────────────────────────────────
_required = [
    "langgraph_app", "AgentState",
    "TOP_K_RETRIEVAL", "AGENT_LOG",
    "node_analyse", "node_retrieve", "node_context_filter", "node_format",
    "node_out_of_scope", "retriever", "reranker", "rerank",
    "articles_by_num", "RRF_MIN_SCORE",
]
_missing = [v for v in _required if v not in globals()]
if _missing:
    raise RuntimeError(
        f"❌ Variables/fonctions manquantes : {_missing}\n"
        "   Exécutez les cellules 1 à 11 avant cette cellule."
    )
print("✅ Dépendances amont vérifiées.")

# ══════════════════════════════════════════════════════════════
# 2. GRAPHE API-SAFE (sans input() bloquant, sans champs obligatoires)
# ══════════════════════════════════════════════════════════════
from langgraph.graph import StateGraph, END
import time as _time, json as _json, uuid as _uuid

def node_skip_collection(state: AgentState) -> AgentState:
    """
    Remplace node_info_collection : ne pose aucune question,
    ne vérifie aucun champ obligatoire.
    Passe directement à la retrieval avec ce qui est disponible.
    """
    return {
        **state,
        "contract_entities": {k: v for k, v in state.get("contract_entities", {}).items() if v},
        "logs": ["[SKIP_COLLECT] mode API — collecte de champs désactivée"],
    }

def cond_scope_api(state):
    return "out_of_scope" if state.get("out_of_scope") else "retrieve"

_wf = StateGraph(AgentState)
for _name, _fn in [
    ("analyse",        node_analyse),
    ("out_of_scope",   node_out_of_scope),
    ("retrieve",       node_retrieve),
    ("context_filter", node_context_filter),
    ("format",         node_format),
]:
    _wf.add_node(_name, _fn)

_wf.set_entry_point("analyse")
_wf.add_conditional_edges(
    "analyse", cond_scope_api,
    {"out_of_scope": "out_of_scope", "retrieve": "retrieve"})
_wf.add_edge("out_of_scope",   END)
_wf.add_edge("retrieve",       "context_filter")
_wf.add_edge("context_filter", "format")
_wf.add_edge("format",         END)

langgraph_app_api = _wf.compile()
print("✅ Graphe API-safe compilé (sans collecte de champs, sans input())")

# ══════════════════════════════════════════════════════════════
# 2bis. STOCKAGE EN MÉMOIRE — résultats + métriques
# ══════════════════════════════════════════════════════════════
# Cache des réponses, indexé par question_id (permet le GET /result/{id})
RESULTS_STORE: dict = {}

# Compteurs globaux pour /metrics
METRICS_STORE = {
    "total_requests":      0,
    "total_errors":        0,
    "total_out_of_scope":  0,
    "total_time_s":        0.0,
    "min_time_s":          None,
    "max_time_s":          None,
    "started_at":          _time.time(),
    "last_request_at":     None,
}

def _update_metrics(elapsed: float, hors_scope: bool, error: bool = False):
    METRICS_STORE["total_requests"]  += 1
    METRICS_STORE["last_request_at"]  = _time.time()
    if error:
        METRICS_STORE["total_errors"] += 1
        return
    if hors_scope:
        METRICS_STORE["total_out_of_scope"] += 1
    METRICS_STORE["total_time_s"] += elapsed
    if METRICS_STORE["min_time_s"] is None or elapsed < METRICS_STORE["min_time_s"]:
        METRICS_STORE["min_time_s"] = elapsed
    if METRICS_STORE["max_time_s"] is None or elapsed > METRICS_STORE["max_time_s"]:
        METRICS_STORE["max_time_s"] = elapsed

print("✅ Stockage en mémoire prêt (RESULTS_STORE, METRICS_STORE).")

# ══════════════════════════════════════════════════════════════
# 3. FONCTION D'INVOCATION — RETOURNE UNIQUEMENT LE TOP-3 ARTICLES
# ══════════════════════════════════════════════════════════════
def ask_api(question: str, top_k: int = None) -> dict:
    """
    Exécute le pipeline et retourne un JSON simplifié :
    {
      "question_id": "...",
      "question": "...",
      "articles": [
        {"rang": 1, "numero": "230", "texte": "...", "score": 0.82},
        ...
      ]
    }
    """
    _top_k = top_k or TOP_K_RETRIEVAL
    t0 = _time.time()
    question_id = str(_uuid.uuid4())

    initial = AgentState(
        question=question, question_type="", domains=[], ambiguities=[],
        out_of_scope=False, keywords=[], is_direct_lookup=False,
        is_contract_drafting=False, contract_type={}, contract_entities={},
        candidates=[], rrf_map={}, selected=[], rejected=[],
        final_answer="", json_output={}, confidence="moyenne", mode="api",
        top_k=_top_k, iteration=0, logs=[],
    )

    try:
        result = langgraph_app_api.invoke(initial)
    except Exception as e:
        elapsed = round(_time.time() - t0, 2)
        _update_metrics(elapsed, hors_scope=False, error=True)
        raise

    json_out = result.get("json_output", {})
    elapsed  = round(_time.time() - t0, 2)
    hors_scope = json_out.get("requete", {}).get("hors_scope", False)

    # ── Extraction et simplification des 3 premiers articles ──
    raw_articles = json_out.get("articles", {}).get("liste", [])

    top3 = [
        {
            "rang":   art.get("rang"),
            "numero": art.get("numero"),
            "texte":  art.get("texte", ""),
            "score":  art.get("score_final", art.get("score_rerank", 0)),
        }
        for art in raw_articles[:3]
    ]

    response = {
        "question_id":       question_id,
        "question":          question,
        "hors_scope":        hors_scope,
        "articles":          top3,
        "temps_ecoule_s":    elapsed,
        "timestamp":         _time.time(),
    }

    # ── Mise en cache pour le GET /result/{question_id} ───────
    RESULTS_STORE[question_id] = response

    # ── Mise à jour des métriques ──────────────────────────────
    _update_metrics(elapsed, hors_scope=hors_scope, error=False)

    # ── Log ───────────────────────────────────────────────────
    try:
        with open(AGENT_LOG, "a", encoding="utf-8") as f:
            f.write(_json.dumps({
                "question_id": question_id,
                "question":  question,
                "mode":      "api-v2",
                "articles":  [a["numero"] for a in top3],
                "timestamp": _time.time(),
            }, ensure_ascii=False) + "\n")
    except Exception as _e:
        print(f"  ⚠️ AGENT_LOG : {_e}")

    return response

print("✅ ask_api() prête.")

# ══════════════════════════════════════════════════════════════
# 4. APPLICATION FASTAPI
# ══════════════════════════════════════════════════════════════
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field
from typing import Optional, List

api = FastAPI(
    title="LOC Agentic AI",
    description="Retourne les 3 premiers articles du ق.ل.ع correspondant à la question",
    version="2.1.0",
)

api.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

# ── Schémas ────────────────────────────────────────────────────
class AskRequest(BaseModel):
    question: str  = Field(..., min_length=1, description="Question juridique")
    top_k:    Optional[int] = Field(None, ge=1, le=100)

class ArticleItem(BaseModel):
    rang:   int
    numero: str
    texte:  str
    score:  float

class AskResponse(BaseModel):
    question_id:    str
    question:       str
    hors_scope:     bool
    articles:       List[ArticleItem]
    temps_ecoule_s: float
    timestamp:      float

class MetricsResponse(BaseModel):
    total_requests:        int
    total_errors:          int
    total_out_of_scope:    int
    avg_time_s:            Optional[float]
    min_time_s:            Optional[float]
    max_time_s:            Optional[float]
    uptime_s:              float
    last_request_at:       Optional[float]

# ── Routes ─────────────────────────────────────────────────────
@api.get("/")
def root():
    return {
        "service":   "LOC Agentic AI v2",
        "status":    "ok",
        "endpoints": [
            "/ask (POST)",
            "/result/{question_id} (GET)",
            "/metrics (GET)",
            "/health (GET)",
            "/docs (GET)",
        ],
    }

@api.get("/health")
def health():
    checks = {
        "retriever":     "retriever"        in globals(),
        "reranker":      "reranker"         in globals(),
        "langgraph_app": "langgraph_app_api" in globals(),
        "articles_db":   "articles_by_num"  in globals(),
    }
    return {"status": "ok" if all(checks.values()) else "degraded", "checks": checks}

@api.post("/ask", response_model=AskResponse)
def ask(req: AskRequest):
    """
    Pose une question juridique et reçoit les 3 premiers articles pertinents du ق.ل.ع.

    Exemple de réponse :
```json
    {
      "question_id": "3f1b2c...-uuid",
      "question": "ما هي شروط بطلان العقد؟",
      "hors_scope": false,
      "articles": [
        {"rang": 1, "numero": "306", "texte": "...", "score": 0.84},
        {"rang": 2, "numero": "307", "texte": "...", "score": 0.71},
        {"rang": 3, "numero": "308", "texte": "...", "score": 0.65}
      ],
      "temps_ecoule_s": 3.21,
      "timestamp": 1719999999.0
    }
```
    Le `question_id` retourné permet de récupérer à nouveau cette même
    réponse plus tard via GET /result/{question_id}.
    """
    question = req.question.strip()
    if not question:
        raise HTTPException(status_code=400, detail="La question est vide.")
    try:
        return ask_api(question=question, top_k=req.top_k)
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Erreur pipeline : {e}")

@api.get("/result/{question_id}", response_model=AskResponse)
def get_result(question_id: str):
    """
    Récupère (en JSON) la réponse du LLM précédemment générée par /ask,
    à partir de son question_id.

    Exemple :
        GET /result/3f1b2c...-uuid
    """
    cached = RESULTS_STORE.get(question_id)
    if cached is None:
        raise HTTPException(
            status_code=404,
            detail=f"Aucun résultat trouvé pour question_id={question_id}."
        )
    return cached

@api.get("/result/latest", response_model=AskResponse)
def get_latest_result():
    """
    Récupère (en JSON) la dernière réponse générée par /ask,
    sans avoir besoin de connaître son question_id.
    """
    if not RESULTS_STORE:
        raise HTTPException(status_code=404, detail="Aucune requête n'a encore été traitée.")
    last = max(RESULTS_STORE.values(), key=lambda r: r["timestamp"])
    return last

@api.get("/metrics", response_model=MetricsResponse)
def get_metrics():
    """
    Récupère les métriques d'utilisation de l'API (en JSON) :
    nombre total de requêtes, erreurs, hors-scope, temps moyen/min/max,
    uptime, et timestamp de la dernière requête.
    """
    n = METRICS_STORE["total_requests"] - METRICS_STORE["total_errors"]
    avg = round(METRICS_STORE["total_time_s"] / n, 3) if n > 0 else None
    return {
        "total_requests":      METRICS_STORE["total_requests"],
        "total_errors":        METRICS_STORE["total_errors"],
        "total_out_of_scope":  METRICS_STORE["total_out_of_scope"],
        "avg_time_s":          avg,
        "min_time_s":          METRICS_STORE["min_time_s"],
        "max_time_s":          METRICS_STORE["max_time_s"],
        "uptime_s":            round(_time.time() - METRICS_STORE["started_at"], 1),
        "last_request_at":     METRICS_STORE["last_request_at"],
    }

print("✅ FastAPI définie : / | /health | /ask | /result/{id} | /result/latest | /metrics | /docs")

# ══════════════════════════════════════════════════════════════
# 5. LANCEMENT UVICORN + TUNNEL NGROK
# ══════════════════════════════════════════════════════════════
import os, nest_asyncio, uvicorn, threading
from pyngrok import ngrok

nest_asyncio.apply()
API_PORT = 8000

# ── 5.1 Token ngrok (via Kaggle Secrets) ──────────────────────
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
_ngrok_token = user_secrets.get_secret("Ngok_key")
ngrok.set_auth_token(_ngrok_token)
print("✅ Authtoken ngrok configuré (depuis Kaggle Secrets).")

# ── 5.2 Fermeture des tunnels précédents ──────────────────────
try:
    for _t in ngrok.get_tunnels():
        ngrok.disconnect(_t.public_url)
except Exception:
    pass

# ── 5.3 Serveur uvicorn (thread daemon) ───────────────────────
def _run():
    uvicorn.run(api, host="0.0.0.0", port=API_PORT, log_level="warning")

threading.Thread(target=_run, daemon=True).start()
_time.sleep(2)
print(f"✅ Uvicorn démarré sur le port {API_PORT}.")

# ── 5.4 Tunnel public ─────────────────────────────────────────
PUBLIC_URL = ngrok.connect(API_PORT, "http").public_url

print("\n" + "═" * 62)
print("  🌍 API PUBLIQUE :")
print(f"     {PUBLIC_URL}")
print(f"     {PUBLIC_URL}/docs    → Swagger UI")
print(f"     {PUBLIC_URL}/health  → état du pipeline")
print(f"     {PUBLIC_URL}/metrics → métriques d'usage")
print("═" * 62)
print("\n  Exemple curl (POST puis GET) :")
print(f"""  curl -X POST "{PUBLIC_URL}/ask" \\
       -H "Content-Type: application/json" \\
       -d '{{"question": "ما هي شروط بطلان العقد؟"}}'

  # Avec le question_id reçu dans la réponse ci-dessus :
  curl "{PUBLIC_URL}/result/<question_id>"

  # Ou directement la dernière réponse :
  curl "{PUBLIC_URL}/result/latest"

  # Métriques :
  curl "{PUBLIC_URL}/metrics" """)
print("═" * 62)

print("\n✅ Cellule 12 v2.1 terminée")
print("   Variables exportées : api, ask_api, langgraph_app_api, PUBLIC_URL, RESULTS_STORE, METRICS_STORE")

✅ Dépendances prêtes.
✅ Dépendances amont vérifiées.
✅ Graphe API-safe compilé (sans collecte de champs, sans input())
✅ Stockage en mémoire prêt (RESULTS_STORE, METRICS_STORE).
✅ ask_api() prête.
✅ FastAPI définie : / | /health | /ask | /result/{id} | /result/latest | /metrics | /docs
✅ Authtoken ngrok configuré (depuis Kaggle Secrets).
✅ Uvicorn démarré sur le port 8000.

══════════════════════════════════════════════════════════════
  🌍 API PUBLIQUE :
     https://bagginess-defame-virus.ngrok-free.dev
     https://bagginess-defame-virus.ngrok-free.dev/docs    → Swagger UI
     https://bagginess-defame-virus.ngrok-free.dev/health  → état du pipeline
     https://bagginess-defame-virus.ngrok-free.dev/metrics → métriques d'usage
══════════════════════════════════════════════════════════════

  Exemple curl (POST puis GET) :
  curl -X POST "https://bagginess-defame-virus.ngrok-free.dev/ask" \
       -H "Content-Type: application/json" \
       -d '{"question": "ما هي شروط بطلان العقد؟"}